<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/10_SPP_GAN_Differential_Privacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

print("\n" + "=" * 100)
print("1. HEADER & SCOPE")
print("=" * 100)

from pathlib import Path
from google.colab import drive

# --------------------------------------------------------------------------------------------------
# Notebook identity
# --------------------------------------------------------------------------------------------------

NOTEBOOK_ID = "10"
NOTEBOOK_NAME = "SPP-GAN Differential Privacy"
FRAMEWORK_NAME = "SPP-GAN"

# --------------------------------------------------------------------------------------------------
# Canonical Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive"
)

MYDRIVE = (
    DRIVE_ROOT /
    "MyDrive"
)

if not MYDRIVE.exists():

    print(
        "Google Drive not mounted. Mounting..."
    )

    drive.mount(
        "/content/drive"
    )

if not MYDRIVE.exists():

    raise RuntimeError(
        "Google Drive mount failed.\n"
        f"Expected MyDrive at:\n{MYDRIVE}"
    )

# --------------------------------------------------------------------------------------------------
# Canonical project root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE /
    "SPP_GAN_Research"
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "Canonical SPP-GAN project root not found:\n"
        f"{PROJECT_ROOT}\n\n"
        "Verify that the project exists in Google Drive."
    )

# --------------------------------------------------------------------------------------------------
# Notebook scope
# --------------------------------------------------------------------------------------------------

NOTEBOOK_SCOPE = {

    "purpose": (
        "Define and validate the differential privacy mechanism "
        "for the SPP-GAN discriminator."
    ),

    "protected_component":
        "SPP-GAN discriminator",

    "privacy_mechanism":
        "DP-SGD",

    "gradient_mechanism":
        "per-example discriminator gradients",

    "clipping_mechanism":
        "flat L2 clipping",

    "noise_mechanism":
        "Gaussian",

    "sampling_mechanism":
        "Poisson",

    "accountant":
        "RDP",

    "training_performed":
        False,

    "accounting_performed":
        False,

    "synthetic_generation_performed":
        False,

    "end_to_end_privacy_claim":
        False,
}

# --------------------------------------------------------------------------------------------------
# Privacy boundary
# --------------------------------------------------------------------------------------------------

PRIVACY_BOUNDARY = {

    "discriminator_update_private":
        True,

    "generator_update_private_in_notebook_10":
        False,

    "statistical_guidance_private_in_notebook_10":
        False,

    "preprocessing_private_in_notebook_10":
        False,

    "conditioning_private_in_notebook_10":
        False,

    "formal_accounting_notebook":
        "11",

    "training_notebook":
        "12",

    "generation_notebook":
        "13",
}

# --------------------------------------------------------------------------------------------------
# Final scope display
# --------------------------------------------------------------------------------------------------

print(
    f"Notebook ID                    : {NOTEBOOK_ID}"
)

print(
    f"Notebook                       : {NOTEBOOK_NAME}"
)

print(
    f"Framework                      : {FRAMEWORK_NAME}"
)

print(
    f"MyDrive                        : {MYDRIVE}"
)

print(
    f"Project root                   : {PROJECT_ROOT}"
)

print(
    f"Privacy mechanism              : "
    f"{NOTEBOOK_SCOPE['privacy_mechanism']}"
)

print(
    f"Protected component            : "
    f"{NOTEBOOK_SCOPE['protected_component']}"
)

print(
    f"Gradient mechanism             : "
    f"{NOTEBOOK_SCOPE['gradient_mechanism']}"
)

print(
    f"Clipping mechanism             : "
    f"{NOTEBOOK_SCOPE['clipping_mechanism']}"
)

print(
    f"Noise mechanism                : "
    f"{NOTEBOOK_SCOPE['noise_mechanism']}"
)

print(
    f"Sampling mechanism             : "
    f"{NOTEBOOK_SCOPE['sampling_mechanism']}"
)

print(
    f"Privacy accountant             : "
    f"{NOTEBOOK_SCOPE['accountant']}"
)

print()
print(
    "Privacy boundary:"
)

print(
    "  ✓ Discriminator update       : PRIVATE"
)

print(
    "  ✓ Statistical guidance       : "
    "NOT PRIVATIZED IN NB10"
)

print(
    "  ✓ Preprocessing              : "
    "NOT PRIVATIZED IN NB10"
)

print(
    "  ✓ Generator update           : "
    "NOT PRIVATIZED IN NB10"
)

print(
    "  ✓ Formal accounting          : "
    "DEFERRED TO NB11"
)

print(
    "  ✓ SPP-GAN training           : "
    "DEFERRED TO NB12"
)

print(
    "  ✓ Synthetic generation       : "
    "DEFERRED TO NB13"
)

print()
print(
    "✓ Google Drive verified."
)

print(
    "✓ Canonical project root verified."
)

print(
    "✓ Notebook 10 scope established."
)

print(
    "SECTION 1 STATUS: PASS"
)


1. HEADER & SCOPE
Notebook ID                    : 10
Notebook                       : SPP-GAN Differential Privacy
Framework                      : SPP-GAN
MyDrive                        : /content/drive/MyDrive
Project root                   : /content/drive/MyDrive/SPP_GAN_Research
Privacy mechanism              : DP-SGD
Protected component            : SPP-GAN discriminator
Gradient mechanism             : per-example discriminator gradients
Clipping mechanism             : flat L2 clipping
Noise mechanism                : Gaussian
Sampling mechanism             : Poisson
Privacy accountant             : RDP

Privacy boundary:
  ✓ Discriminator update       : PRIVATE
  ✓ Statistical guidance       : NOT PRIVATIZED IN NB10
  ✓ Preprocessing              : NOT PRIVATIZED IN NB10
  ✓ Generator update           : NOT PRIVATIZED IN NB10
  ✓ Formal accounting          : DEFERRED TO NB11
  ✓ SPP-GAN training           : DEFERRED TO NB12
  ✓ Synthetic generation       : DEFERRED TO NB13



In [55]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("2. LOAD CONFIGURATION")
print("=" * 100)

import json
import hashlib
import math
import sys
import subprocess
import importlib.util
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --------------------------------------------------------------------------------------------------
# Google Drive
# --------------------------------------------------------------------------------------------------

from google.colab import drive

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

if not MYDRIVE.exists():

    drive.mount(
        "/content/drive"
    )

if not MYDRIVE.exists():

    raise RuntimeError(
        "Google Drive is not available."
    )

print(
    f"✓ MyDrive        : {MYDRIVE}"
)

# --------------------------------------------------------------------------------------------------
# Canonical roots
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE /
    "SPP_GAN_Research"
)

NB02_ROOT = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "notebook_02"
)

NB02_MANIFEST_ROOT = (
    PROJECT_ROOT /
    "results" /
    "raw_validation" /
    "notebook_02_manifests"
)

NB08_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_08"
)

NB09_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_09"
)

NB10_ROOT = (
    PROJECT_ROOT /
    "results" /
    "notebooks" /
    "notebook_10"
)

for name, path in {
    "Project root":
        PROJECT_ROOT,

    "Notebook 02 processed root":
        NB02_ROOT,

    "Notebook 02 manifest root":
        NB02_MANIFEST_ROOT,

    "Notebook 08":
        NB08_ROOT,

    "Notebook 09":
        NB09_ROOT,
}.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n"
            f"{path}"
        )

# --------------------------------------------------------------------------------------------------
# Notebook 10 directories
# --------------------------------------------------------------------------------------------------

DIRS = {

    "root":
        NB10_ROOT,

    "models":
        NB10_ROOT /
        "models",

    "configuration":
        NB10_ROOT /
        "configuration",

    "metadata":
        NB10_ROOT /
        "metadata",

    "validation":
        NB10_ROOT /
        "validation",

    "manifests":
        NB10_ROOT /
        "manifests",
}

for directory in DIRS.values():

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# --------------------------------------------------------------------------------------------------
# Dataset registry
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [

    "adult_income",

    "bank_marketing",

    "diabetes_130us",
]

if len(DATASET_IDS) != 3:

    raise RuntimeError(
        "Expected exactly three datasets."
    )

if len(
    set(DATASET_IDS)
) != len(DATASET_IDS):

    raise RuntimeError(
        "Dataset registry contains duplicate dataset IDs."
    )

# --------------------------------------------------------------------------------------------------
# Canonical Notebook 02 combined split manifest
# --------------------------------------------------------------------------------------------------

NB02_COMBINED_SPLIT_MANIFEST_PATH = (
    NB02_MANIFEST_ROOT /
    "combined_split_manifest.csv"
)

if not NB02_COMBINED_SPLIT_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Canonical Notebook 02 combined split manifest not found:\n"
        f"{NB02_COMBINED_SPLIT_MANIFEST_PATH}\n\n"
        "Notebook 10 requires the persisted Notebook 02 "
        "split manifest for privacy-parameter derivation."
    )

print()
print(
    "Notebook 02 split provenance:"
)

print(
    f"✓ Combined split manifest:\n"
    f"  {NB02_COMBINED_SPLIT_MANIFEST_PATH}"
)

# --------------------------------------------------------------------------------------------------
# Load canonical combined split manifest
# --------------------------------------------------------------------------------------------------

NB02_COMBINED_SPLIT_DF = pd.read_csv(
    NB02_COMBINED_SPLIT_MANIFEST_PATH
)

if NB02_COMBINED_SPLIT_DF.empty:

    raise RuntimeError(
        "Notebook 02 combined split manifest is empty."
    )

print(
    f"✓ Combined manifest rows : "
    f"{len(NB02_COMBINED_SPLIT_DF):,}"
)

print(
    f"✓ Combined manifest cols : "
    f"{NB02_COMBINED_SPLIT_DF.shape[1]}"
)

# --------------------------------------------------------------------------------------------------
# Validate required manifest columns
# --------------------------------------------------------------------------------------------------

MANIFEST_COLUMNS = set(
    NB02_COMBINED_SPLIT_DF.columns
)

REQUIRED_MANIFEST_COLUMNS = {

    "dataset_id",

    "split",
}

missing_manifest_columns = (
    REQUIRED_MANIFEST_COLUMNS
    -
    MANIFEST_COLUMNS
)

if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 combined split manifest is missing "
        "required columns:\n"
        f"{sorted(missing_manifest_columns)}\n\n"
        f"Available columns:\n"
        f"{list(NB02_COMBINED_SPLIT_DF.columns)}"
    )

# --------------------------------------------------------------------------------------------------
# Validate dataset coverage
# --------------------------------------------------------------------------------------------------

manifest_datasets = set(
    NB02_COMBINED_SPLIT_DF[
        "dataset_id"
    ]
    .astype(str)
    .unique()
)

expected_datasets = set(
    DATASET_IDS
)

if manifest_datasets != expected_datasets:

    raise RuntimeError(
        "Notebook 02 manifest dataset coverage mismatch.\n"
        f"Expected : {sorted(expected_datasets)}\n"
        f"Found    : {sorted(manifest_datasets)}"
    )

# --------------------------------------------------------------------------------------------------
# Validate split labels
# --------------------------------------------------------------------------------------------------

manifest_splits = set(
    NB02_COMBINED_SPLIT_DF[
        "split"
    ]
    .astype(str)
    .str.lower()
    .unique()
)

EXPECTED_SPLITS = {
    "train",
    "validation",
    "test",
}

if manifest_splits != EXPECTED_SPLITS:

    raise RuntimeError(
        "Notebook 02 manifest split coverage mismatch.\n"
        f"Expected : {sorted(EXPECTED_SPLITS)}\n"
        f"Found    : {sorted(manifest_splits)}"
    )

# --------------------------------------------------------------------------------------------------
# Canonical training-row counts
# --------------------------------------------------------------------------------------------------

TRAIN_ROWS = {}

for dataset_id in DATASET_IDS:

    dataset_mask = (
        NB02_COMBINED_SPLIT_DF[
            "dataset_id"
        ]
        .astype(str)
        .eq(dataset_id)
    )

    train_mask = (
        NB02_COMBINED_SPLIT_DF[
            "split"
        ]
        .astype(str)
        .str.lower()
        .eq("train")
    )

    dataset_train = (
        NB02_COMBINED_SPLIT_DF[
            dataset_mask &
            train_mask
        ]
    )

    if dataset_train.empty:

        raise RuntimeError(
            f"No TRAIN rows found for dataset: "
            f"{dataset_id}"
        )

    TRAIN_ROWS[
        dataset_id
    ] = int(
        len(dataset_train)
    )

# --------------------------------------------------------------------------------------------------
# Validate canonical split counts
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_mask = (
        NB02_COMBINED_SPLIT_DF[
            "dataset_id"
        ]
        .astype(str)
        .eq(dataset_id)
    )

    dataset_rows = (
        NB02_COMBINED_SPLIT_DF[
            dataset_mask
        ]
    )

    split_counts = (
        dataset_rows[
            "split"
        ]
        .astype(str)
        .str.lower()
        .value_counts()
        .to_dict()
    )

    for split_name in EXPECTED_SPLITS:

        if split_name not in split_counts:

            raise RuntimeError(
                f"Dataset '{dataset_id}' is missing "
                f"split '{split_name}'."
            )

        if split_counts[
            split_name
        ] <= 0:

            raise RuntimeError(
                f"Dataset '{dataset_id}' has zero rows "
                f"for split '{split_name}'."
            )

# --------------------------------------------------------------------------------------------------
# Validate canonical TRAIN row counts against established Notebook 02 split totals
# --------------------------------------------------------------------------------------------------
#
# These are validation references only.
# TRAIN_ROWS themselves are derived exclusively from the persisted manifest above.
#

EXPECTED_NOTEBOOK_02_TRAIN_ROWS = {

    "adult_income":
        34189,

    "bank_marketing":
        31647,

    "diabetes_130us":
        71236,
}

for dataset_id in DATASET_IDS:

    observed_rows = (
        TRAIN_ROWS[
            dataset_id
        ]
    )

    expected_rows = (
        EXPECTED_NOTEBOOK_02_TRAIN_ROWS[
            dataset_id
        ]
    )

    if observed_rows != expected_rows:

        raise RuntimeError(
            "Notebook 02 TRAIN row-count integrity check failed.\n"
            f"Dataset  : {dataset_id}\n"
            f"Observed : {observed_rows:,}\n"
            f"Expected : {expected_rows:,}"
        )

print()
print(
    "✓ Canonical Notebook 02 TRAIN row counts:"
)

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id:<20} : "
        f"{TRAIN_ROWS[dataset_id]:,}"
    )

print()
print(
    "✓ Notebook 02 split provenance validated."
)

# --------------------------------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025

np.random.seed(
    MASTER_SEED
)

torch.manual_seed(
    MASTER_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        MASTER_SEED
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print()
print(
    f"✓ Project root   : {PROJECT_ROOT}"
)

print(
    f"✓ Notebook 02    : {NB02_ROOT}"
)

print(
    f"✓ Notebook 08    : {NB08_ROOT}"
)

print(
    f"✓ Notebook 09    : {NB09_ROOT}"
)

print(
    f"✓ Notebook 10    : {NB10_ROOT}"
)

print(
    f"✓ Master seed    : {MASTER_SEED}"
)

print(
    f"✓ Device         : {DEVICE}"
)

# --------------------------------------------------------------------------------------------------
# Notebook 09 configuration
# --------------------------------------------------------------------------------------------------

NB09_CONFIGURATION_PATH = (
    NB09_ROOT /
    "configuration" /
    "sppgan_statistical_guidance_configuration.json"
)

if not NB09_CONFIGURATION_PATH.exists():

    raise FileNotFoundError(
        "Notebook 09 configuration not found:\n"
        f"{NB09_CONFIGURATION_PATH}"
    )

with open(
    NB09_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    NB09_GUIDANCE_CONFIGURATION = json.load(
        f
    )

if not isinstance(
    NB09_GUIDANCE_CONFIGURATION,
    dict,
):

    raise TypeError(
        "Notebook 09 configuration must contain a JSON object."
    )

# --------------------------------------------------------------------------------------------------
# Required Notebook 09 configuration keys
# --------------------------------------------------------------------------------------------------

REQUIRED_NB09_KEYS = {

    "configuration_version",

    "objective",

    "weights",

    "distribution_guidance",

    "moment_guidance",

    "categorical_guidance",

    "dependency_guidance",

    "privacy",
}

missing_nb09_keys = (
    REQUIRED_NB09_KEYS
    -
    set(
        NB09_GUIDANCE_CONFIGURATION.keys()
    )
)

if missing_nb09_keys:

    raise RuntimeError(
        "Notebook 09 configuration missing keys:\n"
        f"{sorted(missing_nb09_keys)}"
    )

# --------------------------------------------------------------------------------------------------
# Validate Notebook 09 objective
# --------------------------------------------------------------------------------------------------

if not isinstance(
    NB09_GUIDANCE_CONFIGURATION[
        "objective"
    ],
    dict,
):

    raise TypeError(
        "Notebook 09 'objective' configuration must be a dictionary."
    )

if (
    "lambda_stat"
    not in NB09_GUIDANCE_CONFIGURATION[
        "objective"
    ]
):

    raise KeyError(
        "Notebook 09 objective is missing 'lambda_stat'."
    )

LAMBDA_STAT = float(
    NB09_GUIDANCE_CONFIGURATION[
        "objective"
    ][
        "lambda_stat"
    ]
)

if not np.isfinite(
    LAMBDA_STAT
):

    raise ValueError(
        "lambda_stat must be finite."
    )

if LAMBDA_STAT < 0:

    raise ValueError(
        "lambda_stat must be non-negative."
    )

# --------------------------------------------------------------------------------------------------
# Final configuration display
# --------------------------------------------------------------------------------------------------

print()

print(
    f"✓ Notebook 09 configuration loaded:\n"
    f"  {NB09_CONFIGURATION_PATH}"
)

print(
    f"✓ λ_stat = {LAMBDA_STAT:.6f}"
)

print()
print(
    "SECTION 2 STATUS: PASS"
)


2. LOAD CONFIGURATION
✓ MyDrive        : /content/drive/MyDrive

Notebook 02 split provenance:
✓ Combined split manifest:
  /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/combined_split_manifest.csv
✓ Combined manifest rows : 195,819
✓ Combined manifest cols : 3

✓ Canonical Notebook 02 TRAIN row counts:
  adult_income         : 34,189
  bank_marketing       : 31,647
  diabetes_130us       : 71,236

✓ Notebook 02 split provenance validated.

✓ Project root   : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 02    : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Notebook 08    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08
✓ Notebook 09    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09
✓ Notebook 10    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ Master seed    : 2025
✓ Device         : cuda

✓ Notebook 09 configuration loaded:
  /content/drive/M

In [57]:
# ==================================================================================================
# 3. LOAD SPP-GAN ARCHITECTURE
# ==================================================================================================

print("\n" + "=" * 100)
print("3. LOAD SPP-GAN ARCHITECTURE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Canonical Notebook 08 architecture artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_PATH = (
    NB08_ROOT /
    "architecture" /
    "sppgan_architecture_summary.csv"
)

PARAMETER_COUNT_PATH = (
    NB08_ROOT /
    "metadata" /
    "sppgan_parameter_count.csv"
)

ARCHITECTURE_CONFIG_PATH = (
    NB08_ROOT /
    "config" /
    "sppgan_model_configuration.json"
)

ARCHITECTURE_REGISTRY_PATH = (
    NB08_ROOT /
    "metadata" /
    "sppgan_architecture_artifacts.csv"
)

REQUIRED_ARCHITECTURE_FILES = {
    "architecture_summary":
        ARCHITECTURE_SUMMARY_PATH,
    "parameter_count":
        PARAMETER_COUNT_PATH,
    "model_configuration":
        ARCHITECTURE_CONFIG_PATH,
    "architecture_registry":
        ARCHITECTURE_REGISTRY_PATH,
}

# --------------------------------------------------------------------------------------------------
# Verify canonical Notebook 08 artifacts
# --------------------------------------------------------------------------------------------------

for artifact_name, artifact_path in (
    REQUIRED_ARCHITECTURE_FILES.items()
):

    if not artifact_path.exists():

        raise FileNotFoundError(
            "Required Notebook 08 architecture artifact not found.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )

    if not artifact_path.is_file():

        raise RuntimeError(
            "Notebook 08 architecture artifact is not a file.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )

    if artifact_path.stat().st_size == 0:

        raise RuntimeError(
            "Notebook 08 architecture artifact is empty.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )

print()
print("✓ Canonical Notebook 08 architecture artifacts verified:")

for artifact_name, artifact_path in (
    REQUIRED_ARCHITECTURE_FILES.items()
):

    print(
        f"  {artifact_name:<24}: {artifact_path}"
    )

# --------------------------------------------------------------------------------------------------
# Load architecture summary
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_DF = pd.read_csv(
    ARCHITECTURE_SUMMARY_PATH
)

# --------------------------------------------------------------------------------------------------
# Required architecture columns
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCH_COLUMNS = {
    "dataset",
    "target",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
    "numerical_activation",
    "categorical_training_activation",
    "categorical_probability_mapping",
    "hard_decoding",
    "critic_output",
    "statistical_guidance",
    "differential_privacy",
    "privacy_accounting",
    "training",
}

missing_columns = (
    REQUIRED_ARCH_COLUMNS
    -
    set(ARCHITECTURE_SUMMARY_DF.columns)
)

if missing_columns:

    raise RuntimeError(
        "Notebook 08 architecture summary is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )

# --------------------------------------------------------------------------------------------------
# Normalize dataset identifiers
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_DF["dataset"] = (
    ARCHITECTURE_SUMMARY_DF["dataset"]
    .astype(str)
    .str.strip()
)

ARCHITECTURE_SUMMARY_DF = (
    ARCHITECTURE_SUMMARY_DF[
        ARCHITECTURE_SUMMARY_DF["dataset"].isin(
            DATASET_IDS
        )
    ]
    .copy()
)

if len(ARCHITECTURE_SUMMARY_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Notebook 08 architecture summary does not contain "
        "exactly one row for every registered dataset."
    )

if ARCHITECTURE_SUMMARY_DF["dataset"].duplicated().any():

    raise RuntimeError(
        "Duplicate dataset entries found in Notebook 08 architecture summary."
    )

ARCHITECTURE_SUMMARY_DF = (
    ARCHITECTURE_SUMMARY_DF
    .set_index("dataset")
    .loc[DATASET_IDS]
    .reset_index()
)

# --------------------------------------------------------------------------------------------------
# Validate integer architecture fields
# --------------------------------------------------------------------------------------------------

INTEGER_ARCH_COLUMNS = [
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
]

for column in INTEGER_ARCH_COLUMNS:

    ARCHITECTURE_SUMMARY_DF[column] = pd.to_numeric(
        ARCHITECTURE_SUMMARY_DF[column],
        errors="coerce",
    )

    if ARCHITECTURE_SUMMARY_DF[column].isna().any():

        raise RuntimeError(
            f"Invalid or missing architecture values in column: {column}"
        )

    ARCHITECTURE_SUMMARY_DF[column] = (
        ARCHITECTURE_SUMMARY_DF[column]
        .astype(int)
    )

# --------------------------------------------------------------------------------------------------
# Frozen architecture contract
# --------------------------------------------------------------------------------------------------

EXPECTED_LATENT_DIMENSION = 128

EXPECTED_GENERATOR_HIDDEN_1 = 256

EXPECTED_GENERATOR_HIDDEN_2 = 256

EXPECTED_CRITIC_HIDDEN_1 = 256

EXPECTED_CRITIC_HIDDEN_2 = 256

EXPECTED_CRITIC_OUTPUT = "scalar"

EXPECTED_NUMERICAL_ACTIVATION = "identity"

EXPECTED_CATEGORICAL_TRAINING_ACTIVATION = "gumbel_softmax"

EXPECTED_CATEGORICAL_PROBABILITY_MAPPING = "softmax"

# --------------------------------------------------------------------------------------------------
# Validate architecture dimensions
# --------------------------------------------------------------------------------------------------

for row in ARCHITECTURE_SUMMARY_DF.itertuples():

    expected_generative_dimension = (
        row.numerical_features
        +
        row.categorical_features
        +
        1
    )

    if (
        row.generative_dimension
        !=
        expected_generative_dimension
    ):

        raise RuntimeError(
            f"{row.dataset}: generative dimension mismatch.\n"
            f"Observed : {row.generative_dimension}\n"
            f"Expected : {expected_generative_dimension}"
        )

    if row.transformed_dimension <= 0:

        raise RuntimeError(
            f"{row.dataset}: transformed dimension must be positive."
        )

    if row.latent_dim != EXPECTED_LATENT_DIMENSION:

        raise RuntimeError(
            f"{row.dataset}: latent dimension mismatch.\n"
            f"Observed : {row.latent_dim}\n"
            f"Expected : {EXPECTED_LATENT_DIMENSION}"
        )

    if row.generator_hidden_1 != EXPECTED_GENERATOR_HIDDEN_1:

        raise RuntimeError(
            f"{row.dataset}: generator hidden layer 1 mismatch."
        )

    if row.generator_hidden_2 != EXPECTED_GENERATOR_HIDDEN_2:

        raise RuntimeError(
            f"{row.dataset}: generator hidden layer 2 mismatch."
        )

    if row.critic_hidden_1 != EXPECTED_CRITIC_HIDDEN_1:

        raise RuntimeError(
            f"{row.dataset}: critic hidden layer 1 mismatch."
        )

    if row.critic_hidden_2 != EXPECTED_CRITIC_HIDDEN_2:

        raise RuntimeError(
            f"{row.dataset}: critic hidden layer 2 mismatch."
        )

    if row.total_trainable_parameters <= 0:

        raise RuntimeError(
            f"{row.dataset}: total trainable parameter count "
            "must be positive."
        )

# --------------------------------------------------------------------------------------------------
# Validate output and activation contracts
# --------------------------------------------------------------------------------------------------

NUMERICAL_ACTIVATION_VALUES = (
    ARCHITECTURE_SUMMARY_DF["numerical_activation"]
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

if NUMERICAL_ACTIVATION_VALUES != [
    EXPECTED_NUMERICAL_ACTIVATION
]:

    raise RuntimeError(
        "Unexpected Notebook 08 numerical activation.\n"
        f"Expected : {EXPECTED_NUMERICAL_ACTIVATION}\n"
        f"Observed : {NUMERICAL_ACTIVATION_VALUES}"
    )

CATEGORICAL_TRAINING_ACTIVATION_VALUES = (
    ARCHITECTURE_SUMMARY_DF[
        "categorical_training_activation"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

if CATEGORICAL_TRAINING_ACTIVATION_VALUES != [
    EXPECTED_CATEGORICAL_TRAINING_ACTIVATION
]:

    raise RuntimeError(
        "Unexpected Notebook 08 categorical training activation.\n"
        f"Expected : {EXPECTED_CATEGORICAL_TRAINING_ACTIVATION}\n"
        f"Observed : {CATEGORICAL_TRAINING_ACTIVATION_VALUES}"
    )

CATEGORICAL_MAPPING_VALUES = (
    ARCHITECTURE_SUMMARY_DF[
        "categorical_probability_mapping"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

if CATEGORICAL_MAPPING_VALUES != [
    EXPECTED_CATEGORICAL_PROBABILITY_MAPPING
]:

    raise RuntimeError(
        "Unexpected Notebook 08 categorical probability mapping.\n"
        f"Expected : {EXPECTED_CATEGORICAL_PROBABILITY_MAPPING}\n"
        f"Observed : {CATEGORICAL_MAPPING_VALUES}"
    )

CRITIC_OUTPUT_VALUES = (
    ARCHITECTURE_SUMMARY_DF[
        "critic_output"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

if CRITIC_OUTPUT_VALUES != [
    EXPECTED_CRITIC_OUTPUT
]:

    raise RuntimeError(
        "Unexpected Notebook 08 critic output contract.\n"
        f"Expected : {EXPECTED_CRITIC_OUTPUT}\n"
        f"Observed : {CRITIC_OUTPUT_VALUES}"
    )

# --------------------------------------------------------------------------------------------------
# Resolve frozen dimensions directly from canonical architecture summary
# --------------------------------------------------------------------------------------------------

LATENT_DIMENSION_VALUES = (
    ARCHITECTURE_SUMMARY_DF[
        "latent_dim"
    ]
    .unique()
    .tolist()
)

if LATENT_DIMENSION_VALUES != [
    EXPECTED_LATENT_DIMENSION
]:

    raise RuntimeError(
        "Notebook 08 latent dimension is not globally consistent."
    )

LATENT_DIMENSION = EXPECTED_LATENT_DIMENSION

CRITIC_HIDDEN_1 = EXPECTED_CRITIC_HIDDEN_1

CRITIC_HIDDEN_2 = EXPECTED_CRITIC_HIDDEN_2

# --------------------------------------------------------------------------------------------------
# Load model configuration
# --------------------------------------------------------------------------------------------------

with open(
    ARCHITECTURE_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    NB08_MODEL_CONFIGURATION = json.load(f)

if not isinstance(
    NB08_MODEL_CONFIGURATION,
    dict,
):

    raise RuntimeError(
        "Notebook 08 model configuration must be a JSON object."
    )

# --------------------------------------------------------------------------------------------------
# Load parameter-count artifact
# --------------------------------------------------------------------------------------------------

PARAMETER_COUNT_DF = pd.read_csv(
    PARAMETER_COUNT_PATH
)

if PARAMETER_COUNT_DF.empty:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact is empty."
    )

if "dataset" not in PARAMETER_COUNT_DF.columns:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact must contain "
        "the 'dataset' column."
    )

if "total_trainable_parameters" not in PARAMETER_COUNT_DF.columns:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact must contain "
        "the 'total_trainable_parameters' column."
    )

PARAMETER_COUNT_DF["dataset"] = (
    PARAMETER_COUNT_DF["dataset"]
    .astype(str)
    .str.strip()
)

parameter_datasets = set(
    PARAMETER_COUNT_DF["dataset"]
)

if parameter_datasets != set(DATASET_IDS):

    raise RuntimeError(
        "Notebook 08 parameter-count dataset coverage mismatch.\n"
        f"Expected : {sorted(DATASET_IDS)}\n"
        f"Found    : {sorted(parameter_datasets)}"
    )

if PARAMETER_COUNT_DF["dataset"].duplicated().any():

    raise RuntimeError(
        "Duplicate dataset entries found in Notebook 08 "
        "parameter-count artifact."
    )

PARAMETER_COUNT_DF[
    "total_trainable_parameters"
] = pd.to_numeric(
    PARAMETER_COUNT_DF[
        "total_trainable_parameters"
    ],
    errors="coerce",
)

if PARAMETER_COUNT_DF[
    "total_trainable_parameters"
].isna().any():

    raise RuntimeError(
        "Invalid total_trainable_parameters values in "
        "Notebook 08 parameter-count artifact."
    )

# --------------------------------------------------------------------------------------------------
# Cross-validate architecture parameter counts
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    architecture_row = (
        ARCHITECTURE_SUMMARY_DF[
            ARCHITECTURE_SUMMARY_DF["dataset"]
            ==
            dataset_id
        ]
        .iloc[0]
    )

    parameter_row = (
        PARAMETER_COUNT_DF[
            PARAMETER_COUNT_DF["dataset"]
            ==
            dataset_id
        ]
        .iloc[0]
    )

    architecture_total = int(
        architecture_row[
            "total_trainable_parameters"
        ]
    )

    parameter_total = int(
        parameter_row[
            "total_trainable_parameters"
        ]
    )

    if architecture_total != parameter_total:

        raise RuntimeError(
            f"{dataset_id}: total trainable parameter mismatch.\n"
            f"Architecture summary : {architecture_total}\n"
            f"Parameter artifact   : {parameter_total}"
        )

# --------------------------------------------------------------------------------------------------
# Load architecture registry
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_REGISTRY_DF = pd.read_csv(
    ARCHITECTURE_REGISTRY_PATH
)

if ARCHITECTURE_REGISTRY_DF.empty:

    raise RuntimeError(
        "Notebook 08 architecture registry is empty."
    )

if "dataset" in ARCHITECTURE_REGISTRY_DF.columns:

    registry_datasets = set(
        ARCHITECTURE_REGISTRY_DF[
            "dataset"
        ]
        .astype(str)
        .str.strip()
    )

    if not registry_datasets.issuperset(
        set(DATASET_IDS)
    ):

        raise RuntimeError(
            "Notebook 08 architecture registry does not cover "
            "all canonical datasets."
        )

# --------------------------------------------------------------------------------------------------
# Build critic input-dimension registry
# --------------------------------------------------------------------------------------------------

CRITIC_INPUT_DIMENSIONS = {

    row.dataset:
        int(
            row.transformed_dimension
        )

    for row
    in ARCHITECTURE_SUMMARY_DF.itertuples()
}

if set(
    CRITIC_INPUT_DIMENSIONS
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Critic input-dimension registry does not cover "
        "all registered datasets."
    )

# --------------------------------------------------------------------------------------------------
# Final architecture display
# --------------------------------------------------------------------------------------------------

print()
print("✓ Architecture summary:")
print(
    f"  {ARCHITECTURE_SUMMARY_PATH}"
)

display(
    ARCHITECTURE_SUMMARY_DF[
        [
            "dataset",
            "generative_dimension",
            "transformed_dimension",
            "numerical_features",
            "categorical_features",
            "latent_dim",
            "generator_hidden_1",
            "generator_hidden_2",
            "critic_hidden_1",
            "critic_hidden_2",
            "total_trainable_parameters",
        ]
    ]
)

print(
    f"✓ Latent dimension     : {LATENT_DIMENSION}"
)

print(
    f"✓ Generator hidden     : "
    f"{EXPECTED_GENERATOR_HIDDEN_1} / "
    f"{EXPECTED_GENERATOR_HIDDEN_2}"
)

print(
    f"✓ Critic hidden        : "
    f"{CRITIC_HIDDEN_1} / "
    f"{CRITIC_HIDDEN_2}"
)

print(
    f"✓ Critic output        : "
    f"{EXPECTED_CRITIC_OUTPUT}"
)

print(
    f"✓ Numerical activation : "
    f"{EXPECTED_NUMERICAL_ACTIVATION}"
)

print(
    f"✓ Categorical training : "
    f"{EXPECTED_CATEGORICAL_TRAINING_ACTIVATION}"
)

print(
    f"✓ Categorical mapping  : "
    f"{EXPECTED_CATEGORICAL_PROBABILITY_MAPPING}"
)

print()
print("✓ Critic input dimensions:")

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id:18s}: "
        f"{CRITIC_INPUT_DIMENSIONS[dataset_id]}"
    )

print()
print("✓ Parameter-count consistency validated.")
print("✓ Architecture registry validated.")
print("✓ Canonical Notebook 08 architecture artifacts loaded.")
print("✓ Frozen architecture contract validated.")
print("SECTION 3 STATUS: PASS")


3. LOAD SPP-GAN ARCHITECTURE

✓ Canonical Notebook 08 architecture artifacts verified:
  architecture_summary    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture/sppgan_architecture_summary.csv
  parameter_count         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_parameter_count.csv
  model_configuration     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json
  architecture_registry   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_architecture_artifacts.csv

✓ Architecture summary:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture/sppgan_architecture_summary.csv


,dataset,generative_dimension,transformed_dimension,numerical_features,categorical_features,latent_dim,generator_hidden_1,generator_hidden_2,critic_hidden_1,critic_hidden_2,total_trainable_parameters
0,adult_income,15,105,6,8,128,256,256,256,256,218986
1,bank_marketing,17,51,7,9,128,256,256,256,256,191284
2,diabetes_130us,48,2329,11,36,128,256,256,256,256,1359898


✓ Latent dimension     : 128
✓ Generator hidden     : 256 / 256
✓ Critic hidden        : 256 / 256
✓ Critic output        : scalar
✓ Numerical activation : identity
✓ Categorical training : gumbel_softmax
✓ Categorical mapping  : softmax

✓ Critic input dimensions:
  adult_income      : 105
  bank_marketing    : 51
  diabetes_130us    : 2329

✓ Parameter-count consistency validated.
✓ Architecture registry validated.
✓ Canonical Notebook 08 architecture artifacts loaded.
✓ Frozen architecture contract validated.
SECTION 3 STATUS: PASS


In [58]:
# ==================================================================================================
# SECTION 4 — LOAD PRIVACY CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — LOAD PRIVACY CONFIGURATION")
print("=" * 100)

from pathlib import Path
import json
import math
import platform
import sys

import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------------------
# 4.1 — Canonical Project Paths
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

NB10_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_10"

CONFIG_DIR = NB10_ROOT / "configuration"
VALIDATION_DIR = NB10_ROOT / "validation"
METADATA_DIR = NB10_ROOT / "metadata"
MODELS_DIR = NB10_ROOT / "models"

for directory in [
    CONFIG_DIR,
    VALIDATION_DIR,
    METADATA_DIR,
    MODELS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ NB10 root    : {NB10_ROOT}")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Canonical project root does not exist: {PROJECT_ROOT}"
    )

# --------------------------------------------------------------------------------------------------
# 4.2 — Dataset Registry
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

print(f"✓ Datasets     : {DATASET_IDS}")

# --------------------------------------------------------------------------------------------------
# 4.3 — Privacy Boundary
# --------------------------------------------------------------------------------------------------

PRIVACY_SCOPE = {
    "protected_component": "SPP-GAN discriminator training",
    "mechanism": "DP-SGD",
    "gradient_privacy": True,
    "generator_privacy": False,
    "statistical_guidance_privacy": False,
    "preprocessing_privacy": False,
    "conditioning_privacy": False,
    "synthetic_generation_privacy": False,
    "end_to_end_record_level_dp_claim": False,
}

# --------------------------------------------------------------------------------------------------
# 4.4 — Core DP Configuration
# --------------------------------------------------------------------------------------------------

DP_CONFIG = {
    "target_epsilon": 5.0,
    "delta_rule": "min(1e-5, 1/N_train)",
    "max_grad_norm": 1.0,
    "batch_size": 128,
    "epochs": 300,
    "sampling": "poisson",
    "accountant": "rdp",
    "clipping": "flat",
    "noise_mechanism": "gaussian",
    "optimizer": "DP-SGD",
    "protected_component": "discriminator",
    "secure_mode": False,
    "rng_mode": "STANDARD_PYTORCH_RNG",
    "cryptographically_secure_rng": False,
    "manual_production_clipping": False,
    "manual_production_noise": False,
}

# --------------------------------------------------------------------------------------------------
# 4.5 — Runtime / Implementation Metadata
# --------------------------------------------------------------------------------------------------

RUNTIME_METADATA = {
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "pytorch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

# --------------------------------------------------------------------------------------------------
# 4.6 — Opacus Availability
# --------------------------------------------------------------------------------------------------

try:
    import opacus

    OPACUS_VERSION = opacus.__version__

except Exception as exc:
    raise ImportError(
        "Opacus is required for Notebook 10 DP-SGD implementation."
    ) from exc

RUNTIME_METADATA["opacus_version"] = OPACUS_VERSION

# --------------------------------------------------------------------------------------------------
# 4.7 — RNG Policy
# --------------------------------------------------------------------------------------------------

RNG_POLICY = {
    "mode": "STANDARD_PYTORCH_RNG",
    "secure_mode": False,
    "torchcsprng_required": False,
    "cryptographic_randomness_claim": False,
    "purpose": (
        "Standard PyTorch pseudorandomness is used for stochastic DP-SGD "
        "operations. No cryptographically secure randomness claim is made."
    ),
}

# --------------------------------------------------------------------------------------------------
# 4.8 — Persist Configuration
# --------------------------------------------------------------------------------------------------

PRIVACY_CONFIGURATION = {
    "notebook": "Notebook 10",
    "section": "Section 4 — Load Privacy Configuration",
    "datasets": DATASET_IDS,
    "privacy_scope": PRIVACY_SCOPE,
    "dp_config": DP_CONFIG,
    "rng_policy": RNG_POLICY,
    "runtime": RUNTIME_METADATA,
}

PRIVACY_CONFIG_PATH = CONFIG_DIR / "notebook_10_dp_configuration.json"

with open(PRIVACY_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        PRIVACY_CONFIGURATION,
        f,
        indent=2,
        default=str,
    )

print("-" * 100)
print("PRIVACY CONFIGURATION")
print("-" * 100)

for key, value in DP_CONFIG.items():
    print(f"{key:35s}: {value}")

print("-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

for key, value in PRIVACY_SCOPE.items():
    print(f"{key:35s}: {value}")

print("-" * 100)
print("RNG POLICY")
print("-" * 100)

for key, value in RNG_POLICY.items():
    print(f"{key:35s}: {value}")

print("-" * 100)
print(f"✓ Opacus version : {OPACUS_VERSION}")
print(f"✓ Configuration  : {PRIVACY_CONFIG_PATH}")
print("✓ Section 4 PASS")

SECTION 4 — LOAD PRIVACY CONFIGURATION
✓ Project root : /content/drive/MyDrive/SPP_GAN_Research
✓ NB10 root    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ Datasets     : ['adult_income', 'bank_marketing', 'diabetes_130us']
----------------------------------------------------------------------------------------------------
PRIVACY CONFIGURATION
----------------------------------------------------------------------------------------------------
target_epsilon                     : 5.0
delta_rule                         : min(1e-5, 1/N_train)
max_grad_norm                      : 1.0
batch_size                         : 128
epochs                             : 300
sampling                           : poisson
accountant                         : rdp
clipping                           : flat
noise_mechanism                    : gaussian
optimizer                          : DP-SGD
protected_component                : discriminator
secure_mode                     

In [60]:
# ==================================================================================================
# SECTION 5 — VALIDATE PRIVACY PARAMETERS
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — VALIDATE PRIVACY PARAMETERS")
print("=" * 100)

import math
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------------------
# 5.1 — Required Configuration
# --------------------------------------------------------------------------------------------------

if "DP_CONFIG" not in globals():
    raise RuntimeError(
        "DP_CONFIG is not available. "
        "Run Section 4 before Section 5."
    )

if "TRAIN_ROWS" not in globals():
    raise RuntimeError(
        "TRAIN_ROWS is not available. "
        "Section 2 must derive TRAIN_ROWS from the canonical "
        "Notebook 02 combined split manifest before Section 5."
    )

if set(TRAIN_ROWS) != set(DATASET_IDS):
    raise RuntimeError(
        "TRAIN_ROWS dataset coverage mismatch.\n"
        f"Expected : {sorted(DATASET_IDS)}\n"
        f"Found    : {sorted(TRAIN_ROWS)}"
    )

TARGET_EPSILON = float(
    DP_CONFIG["target_epsilon"]
)

MAX_GRAD_NORM = float(
    DP_CONFIG["max_grad_norm"]
)

BATCH_SIZE = int(
    DP_CONFIG["batch_size"]
)

EPOCHS = int(
    DP_CONFIG["epochs"]
)

ACCOUNTANT_NAME = str(
    DP_CONFIG["accountant"]
).strip().lower()

SAMPLING_NAME = str(
    DP_CONFIG["sampling"]
).strip().lower()

CLIPPING_NAME = str(
    DP_CONFIG["clipping"]
).strip().lower()

NOISE_NAME = str(
    DP_CONFIG["noise_mechanism"]
).strip().lower()

# --------------------------------------------------------------------------------------------------
# 5.2 — Validate Notebook 02 Training-Row Provenance
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    n_train = int(
        TRAIN_ROWS[dataset_id]
    )

    if n_train <= 0:

        raise RuntimeError(
            f"{dataset_id}: TRAIN_ROWS must be positive."
        )

print()
print(
    "✓ Training-row counts inherited from Section 2 "
    "Notebook 02 split provenance."
)

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id:18s}: "
        f"{TRAIN_ROWS[dataset_id]:,}"
    )

# --------------------------------------------------------------------------------------------------
# 5.3 — Delta Calculation
# --------------------------------------------------------------------------------------------------

DELTA_BY_DATASET = {

    dataset_id:
        min(
            1e-5,
            1.0 /
            float(
                TRAIN_ROWS[dataset_id]
            ),
        )

    for dataset_id
    in DATASET_IDS
}

# --------------------------------------------------------------------------------------------------
# 5.4 — Validate Core Privacy Configuration
# --------------------------------------------------------------------------------------------------

if TARGET_EPSILON <= 0:

    raise ValueError(
        "target_epsilon must be > 0."
    )

if MAX_GRAD_NORM <= 0:

    raise ValueError(
        "max_grad_norm must be > 0."
    )

if BATCH_SIZE <= 0:

    raise ValueError(
        "batch_size must be > 0."
    )

if EPOCHS <= 0:

    raise ValueError(
        "epochs must be > 0."
    )

if ACCOUNTANT_NAME != "rdp":

    raise ValueError(
        "Notebook 10 requires the RDP accountant."
    )

if SAMPLING_NAME != "poisson":

    raise ValueError(
        "Notebook 10 requires Poisson sampling."
    )

if CLIPPING_NAME != "flat":

    raise ValueError(
        "Notebook 10 requires flat clipping."
    )

if NOISE_NAME != "gaussian":

    raise ValueError(
        "Notebook 10 requires Gaussian noise."
    )

# --------------------------------------------------------------------------------------------------
# 5.5 — Build Dataset-Specific Privacy Parameters
# --------------------------------------------------------------------------------------------------

privacy_parameter_rows = []

for dataset_id in DATASET_IDS:

    n_train = int(
        TRAIN_ROWS[dataset_id]
    )

    delta = float(
        DELTA_BY_DATASET[dataset_id]
    )

    sampling_rate = (
        BATCH_SIZE /
        float(n_train)
    )

    if sampling_rate > 1.0:

        raise RuntimeError(
            f"{dataset_id}: nominal batch size exceeds "
            "training population."
        )

    planned_steps_per_epoch = int(
        math.ceil(
            n_train /
            BATCH_SIZE
        )
    )

    planned_total_steps = int(
        planned_steps_per_epoch *
        EPOCHS
    )

    privacy_parameter_rows.append({

        "dataset":
            dataset_id,

        "train_rows":
            n_train,

        "batch_size":
            BATCH_SIZE,

        "sampling_rate_q":
            sampling_rate,

        "epochs":
            EPOCHS,

        "planned_steps_per_epoch":
            planned_steps_per_epoch,

        "planned_total_steps":
            planned_total_steps,

        "target_epsilon":
            TARGET_EPSILON,

        "delta":
            delta,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "accountant":
            ACCOUNTANT_NAME,

        "sampling":
            SAMPLING_NAME,

        "clipping":
            CLIPPING_NAME,

        "noise_mechanism":
            NOISE_NAME,

        "secure_mode":
            bool(
                DP_CONFIG["secure_mode"]
            ),

        "rng_mode":
            DP_CONFIG["rng_mode"],
    })

PRIVACY_PARAMETER_DF = pd.DataFrame(
    privacy_parameter_rows
)

# --------------------------------------------------------------------------------------------------
# 5.6 — Validate Numerical Properties
# --------------------------------------------------------------------------------------------------

if PRIVACY_PARAMETER_DF[
    "train_rows"
].isna().any():

    raise RuntimeError(
        "Training-row metadata contains missing values."
    )

if not np.isfinite(
    PRIVACY_PARAMETER_DF[
        "sampling_rate_q"
    ]
).all():

    raise RuntimeError(
        "Sampling-rate values contain NaN or Inf."
    )

if not (
    PRIVACY_PARAMETER_DF[
        "sampling_rate_q"
    ]
    > 0
).all():

    raise RuntimeError(
        "All Poisson sampling rates must be positive."
    )

if not (
    PRIVACY_PARAMETER_DF[
        "sampling_rate_q"
    ]
    <= 1.0
).all():

    raise RuntimeError(
        "Poisson sampling rates must not exceed 1."
    )

if not (
    PRIVACY_PARAMETER_DF[
        "delta"
    ]
    > 0
).all():

    raise RuntimeError(
        "All delta values must be positive."
    )

if not (
    PRIVACY_PARAMETER_DF[
        "delta"
    ]
    <= 1e-5
).all():

    raise RuntimeError(
        "Delta values exceed the configured upper bound."
    )

# --------------------------------------------------------------------------------------------------
# 5.7 — Privacy Parameter Validation Table
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("PRIVACY PARAMETER VALIDATION")
print("-" * 100)

print(
    PRIVACY_PARAMETER_DF[
        [
            "dataset",
            "train_rows",
            "batch_size",
            "sampling_rate_q",
            "epochs",
            "planned_steps_per_epoch",
            "planned_total_steps",
            "target_epsilon",
            "delta",
            "max_grad_norm",
            "accountant",
            "sampling",
            "secure_mode",
            "rng_mode",
        ]
    ].to_string(
        index=False
    )
)

# --------------------------------------------------------------------------------------------------
# 5.8 — Important Accounting Boundary
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("ACCOUNTING BOUNDARY")
print("-" * 100)

print(
    "✓ Training-row counts originate from the canonical "
    "Notebook 02 split manifest."
)

print(
    "✓ Planned step counts are configuration/calibration "
    "metadata only."
)

print(
    "✓ Final achieved epsilon must be computed from the actual "
    "DP-SGD training/accounting trajectory in Notebook 11."
)

print(
    "✓ Notebook 10 does not claim that planned steps equal "
    "actual Poisson-sampling optimizer steps."
)

print(
    "✓ Target epsilon is not reported as achieved epsilon."
)

# --------------------------------------------------------------------------------------------------
# 5.9 — Persist Validation
# --------------------------------------------------------------------------------------------------

PRIVACY_PARAMETER_PATH = (
    CONFIG_DIR /
    "privacy_parameter_validation.csv"
)

PRIVACY_PARAMETER_DF.to_csv(
    PRIVACY_PARAMETER_PATH,
    index=False,
)

print("-" * 100)

print(
    f"✓ Parameter validation saved: "
    f"{PRIVACY_PARAMETER_PATH}"
)

print("✓ Section 5 PASS")

SECTION 5 — VALIDATE PRIVACY PARAMETERS

✓ Training-row counts inherited from Section 2 Notebook 02 split provenance.
  adult_income      : 34,189
  bank_marketing    : 31,647
  diabetes_130us    : 71,236
----------------------------------------------------------------------------------------------------
PRIVACY PARAMETER VALIDATION
----------------------------------------------------------------------------------------------------
       dataset  train_rows  batch_size  sampling_rate_q  epochs  planned_steps_per_epoch  planned_total_steps  target_epsilon   delta  max_grad_norm accountant sampling  secure_mode             rng_mode
  adult_income       34189         128         0.003744     300                      268                80400             5.0 0.00001            1.0        rdp  poisson        False STANDARD_PYTORCH_RNG
bank_marketing       31647         128         0.004045     300                      248                74400             5.0 0.00001            1.0        rd

In [61]:
# ==================================================================================================
# SECTION 6 — DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS
# ==================================================================================================

print("=" * 100)
print("SECTION 6 — DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS")
print("=" * 100)

import inspect
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator


# --------------------------------------------------------------------------------------------------
# 6.1 — Validate Required Notebook 10 Dependencies
# --------------------------------------------------------------------------------------------------

if "SPPGANCritic" not in globals():
    raise RuntimeError(
        "SPPGANCritic is not available. "
        "Run Notebook 10 Section 3 before Section 6."
    )

if "DATASET_IDS" not in globals():
    raise RuntimeError(
        "DATASET_IDS is not available. "
        "Run Notebook 10 Section 2 or Section 4 before Section 6."
    )

if "BATCH_SIZE" not in globals():
    raise RuntimeError(
        "BATCH_SIZE is not available. "
        "Run Notebook 10 Section 4/5 before Section 6."
    )

if "MAX_GRAD_NORM" not in globals():
    raise RuntimeError(
        "MAX_GRAD_NORM is not available. "
        "Run Notebook 10 Section 4/5 before Section 6."
    )


# --------------------------------------------------------------------------------------------------
# 6.2 — Canonical Validation Directories
# --------------------------------------------------------------------------------------------------

if "VALIDATION_DIR" not in globals():
    raise RuntimeError(
        "VALIDATION_DIR is not available. "
        "Run Notebook 10 Section 4 before Section 6."
    )

if "METADATA_DIR" not in globals():
    raise RuntimeError(
        "METADATA_DIR is not available. "
        "Run Notebook 10 Section 4 before Section 6."
    )

VALIDATION_DIR = Path(VALIDATION_DIR)
METADATA_DIR = Path(METADATA_DIR)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 6.3 — Inspect the Frozen Notebook 08 Critic Interface
# --------------------------------------------------------------------------------------------------

CRITIC_CLASS = SPPGANCritic

CRITIC_SIGNATURE = inspect.signature(
    CRITIC_CLASS.__init__
)

CRITIC_PARAMETERS = {
    name: parameter
    for name, parameter in CRITIC_SIGNATURE.parameters.items()
    if name != "self"
}

print("-" * 100)
print("FROZEN SPP-GAN CRITIC INTERFACE")
print("-" * 100)

print(
    f"Critic class : {CRITIC_CLASS.__name__}"
)

print(
    f"Constructor  : {CRITIC_SIGNATURE}"
)

print("-" * 100)
print("Constructor parameters")
print("-" * 100)

for name, parameter in CRITIC_PARAMETERS.items():

    default = (
        "<required>"
        if parameter.default is inspect.Parameter.empty
        else parameter.default
    )

    print(
        f"{name:30s}: "
        f"kind={parameter.kind}, "
        f"default={default}"
    )


# --------------------------------------------------------------------------------------------------
# 6.4 — Resolve Architecture Dimensions from Notebook 08
# --------------------------------------------------------------------------------------------------
#
# Section 3 should already have loaded the canonical architecture information.
#
# We intentionally DO NOT introduce a new hidden_dims argument here.
#
# The constructor is resolved using:
#   1. Existing Notebook 08 architecture configuration, when available.
#   2. The actual constructor's accepted parameter names.
#
# This prevents Notebook 10 from silently creating a different critic architecture.
# --------------------------------------------------------------------------------------------------

CRITIC_INPUT_DIMS = {
    "adult_income": 105,
    "bank_marketing": 51,
    "diabetes_130us": 2329,
}


def _get_architecture_config(dataset_id):
    """
    Locate the architecture metadata already loaded by Notebook 10 Section 3.

    This function does not create or modify architecture metadata.
    """

    candidate_names = [
        "ARCHITECTURE_CONFIG",
        "SPPGAN_ARCHITECTURE_CONFIG",
        "ARCHITECTURE_DF",
        "ARCHITECTURE_SUMMARY_DF",
        "SPPGAN_ARCHITECTURE_DF",
    ]

    for name in candidate_names:

        if name not in globals():
            continue

        candidate = globals()[name]

        # DataFrame
        if isinstance(candidate, pd.DataFrame):

            if "dataset" in candidate.columns:

                matches = candidate[
                    candidate["dataset"].astype(str)
                    == str(dataset_id)
                ]

                if len(matches) == 1:
                    return matches.iloc[0].to_dict()

        # Dictionary
        if isinstance(candidate, dict):

            if dataset_id in candidate:

                value = candidate[dataset_id]

                if isinstance(value, dict):
                    return value

    return None


def _resolve_constructor_kwargs(
    dataset_id,
    input_dim,
):
    """
    Build constructor kwargs from the actual frozen SPPGANCritic signature.

    No unsupported keyword is injected.
    """

    architecture_config = _get_architecture_config(
        dataset_id
    )

    kwargs = {}

    # ----------------------------------------------------------------------------------------------
    # Input dimension
    # ----------------------------------------------------------------------------------------------

    input_candidates = [
        "input_dim",
        "in_dim",
        "input_size",
        "feature_dim",
        "n_features",
    ]

    input_parameter = next(
        (
            name
            for name in input_candidates
            if name in CRITIC_PARAMETERS
        ),
        None,
    )

    if input_parameter is None:

        required_without_defaults = [
            name
            for name, parameter
            in CRITIC_PARAMETERS.items()
            if (
                parameter.default
                is inspect.Parameter.empty
                and parameter.kind
                in (
                    inspect.Parameter.POSITIONAL_ONLY,
                    inspect.Parameter.POSITIONAL_OR_KEYWORD,
                )
            )
        ]

        if len(required_without_defaults) == 1:

            input_parameter = (
                required_without_defaults[0]
            )

        else:

            raise TypeError(
                "Unable to resolve the input-dimension "
                "argument of the frozen SPPGANCritic constructor. "
                f"Constructor signature: {CRITIC_SIGNATURE}"
            )

    kwargs[input_parameter] = int(
        input_dim
    )

    # ----------------------------------------------------------------------------------------------
    # Optional architecture parameters
    # ----------------------------------------------------------------------------------------------
    #
    # Only parameters that:
    #   - actually exist in the frozen constructor, and
    #   - are available in the architecture metadata
    #
    # are passed.
    #
    # No hidden_dims argument is invented.
    # ----------------------------------------------------------------------------------------------

    if architecture_config is not None:

        parameter_aliases = {
            "hidden_dim": [
                "hidden_dim",
                "hidden_size",
                "width",
                "critic_hidden_dim",
            ],
            "num_layers": [
                "num_layers",
                "n_layers",
                "layers",
                "critic_layers",
            ],
            "dropout": [
                "dropout",
                "dropout_rate",
            ],
            "negative_slope": [
                "negative_slope",
                "leaky_relu_slope",
            ],
        }

        for constructor_parameter, aliases in (
            parameter_aliases.items()
        ):

            if constructor_parameter not in CRITIC_PARAMETERS:
                continue

            if constructor_parameter in kwargs:
                continue

            found_value = None
            found = False

            for alias in aliases:

                if alias in architecture_config:

                    found_value = (
                        architecture_config[alias]
                    )

                    found = True
                    break

            if found:

                kwargs[constructor_parameter] = (
                    found_value
                )

    # ----------------------------------------------------------------------------------------------
    # Verify that no unsupported kwargs are being created
    # ----------------------------------------------------------------------------------------------

    unsupported = [
        name
        for name in kwargs
        if name not in CRITIC_PARAMETERS
    ]

    if unsupported:

        raise RuntimeError(
            "Unsupported constructor arguments resolved: "
            f"{unsupported}"
        )

    return kwargs


# --------------------------------------------------------------------------------------------------
# 6.5 — Critic Factory
# --------------------------------------------------------------------------------------------------

def build_frozen_sppgan_critic(
    dataset_id,
):
    """
    Instantiate the exact SPPGANCritic class loaded from Notebook 08.

    The function intentionally uses the actual constructor signature
    instead of assuming a hidden_dims argument.
    """

    if dataset_id not in CRITIC_INPUT_DIMS:
        raise KeyError(
            f"Missing canonical critic input dimension for "
            f"{dataset_id}."
        )

    input_dim = int(
        CRITIC_INPUT_DIMS[dataset_id]
    )

    kwargs = _resolve_constructor_kwargs(
        dataset_id=dataset_id,
        input_dim=input_dim,
    )

    print(
        f"Constructor kwargs for {dataset_id}: "
        f"{kwargs}"
    )

    critic = CRITIC_CLASS(
        **kwargs
    )

    return critic, kwargs


# --------------------------------------------------------------------------------------------------
# 6.6 — Gradient Validation Containers
# --------------------------------------------------------------------------------------------------

GRADIENT_VALIDATION_ROWS = []
GRADIENT_TENSOR_ROWS = []
GRADIENT_VARIATION_ROWS = []


# --------------------------------------------------------------------------------------------------
# 6.7 — Reproducible Validation Seed
# --------------------------------------------------------------------------------------------------

SECTION_06_SEED = 2025

torch.manual_seed(
    SECTION_06_SEED
)

np.random.seed(
    SECTION_06_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SECTION_06_SEED
    )


# --------------------------------------------------------------------------------------------------
# 6.8 — Validate Per-Example Gradients Dataset by Dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Build the frozen Notebook 08 critic
    # ----------------------------------------------------------------------------------------------

    critic, constructor_kwargs = (
        build_frozen_sppgan_critic(
            dataset_id
        )
    )

    critic = ModuleValidator.fix(
        critic,
        strict=False,
    )

    critic.train()

    # ----------------------------------------------------------------------------------------------
    # Verify actual critic input dimension
    # ----------------------------------------------------------------------------------------------

    expected_input_dim = int(
        CRITIC_INPUT_DIMS[dataset_id]
    )

    # ----------------------------------------------------------------------------------------------
    # Validation dataset
    # ----------------------------------------------------------------------------------------------

    test_batch_size = min(
        int(BATCH_SIZE),
        16,
    )

    validation_rows = max(
        test_batch_size * 2,
        32,
    )

    validation_dataset = (
        torch.utils.data.TensorDataset(
            torch.randn(
                validation_rows,
                expected_input_dim,
                dtype=torch.float32,
            )
        )
    )

    validation_loader = (
        torch.utils.data.DataLoader(
            validation_dataset,
            batch_size=test_batch_size,
            shuffle=False,
            drop_last=False,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Optimizer
    # ----------------------------------------------------------------------------------------------

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
    )

    # ----------------------------------------------------------------------------------------------
    # Opacus Privacy Engine
    # ----------------------------------------------------------------------------------------------
    #
    # secure_mode=False is intentional and matches Section 4.
    # Production clipping/noise are handled by Opacus.
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant="rdp",
        secure_mode=False,
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=validation_loader,
        noise_multiplier=1.0,
        max_grad_norm=float(
            MAX_GRAD_NORM
        ),
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve one Poisson-sampled batch
    # ----------------------------------------------------------------------------------------------

    batch = next(
        iter(private_loader)
    )

    x_real = batch[0]

    if x_real.ndim != 2:
        raise RuntimeError(
            f"Unexpected discriminator input shape for "
            f"{dataset_id}: {tuple(x_real.shape)}"
        )

    if x_real.shape[1] != expected_input_dim:
        raise RuntimeError(
            f"Critic input dimension mismatch for "
            f"{dataset_id}: "
            f"observed={x_real.shape[1]}, "
            f"expected={expected_input_dim}"
        )

    # ----------------------------------------------------------------------------------------------
    # Forward pass
    # ----------------------------------------------------------------------------------------------

    scores = private_critic(
        x_real
    )

    if scores.ndim == 0:
        scores = scores.reshape(1)

    elif scores.ndim > 1:
        scores = scores.reshape(
            scores.shape[0],
            -1,
        )

        if scores.shape[1] != 1:
            raise RuntimeError(
                f"Critic output is not scalar per record for "
                f"{dataset_id}: "
                f"{tuple(scores.shape)}"
            )

        scores = scores[:, 0]

    if scores.shape[0] != x_real.shape[0]:
        raise RuntimeError(
            f"Critic output batch mismatch for "
            f"{dataset_id}: "
            f"output={scores.shape[0]}, "
            f"input={x_real.shape[0]}"
        )

    # ----------------------------------------------------------------------------------------------
    # Per-example loss
    # ----------------------------------------------------------------------------------------------
    #
    # For gradient-sample validation we use a mean critic objective.
    # Opacus decomposes the resulting computation into per-example
    # gradient samples.
    # ----------------------------------------------------------------------------------------------

    loss = scores.mean()

    private_optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    # ----------------------------------------------------------------------------------------------
    # Inspect grad_sample tensors
    # ----------------------------------------------------------------------------------------------

    parameter_count = 0
    finite_count = 0
    variation_count = 0
    shape_valid_count = 0

    observed_batch_sizes = set()

    for name, parameter in (
        private_critic.named_parameters()
    ):

        if not parameter.requires_grad:
            continue

        parameter_count += 1

        grad_sample = getattr(
            parameter,
            "grad_sample",
            None,
        )

        if grad_sample is None:

            raise RuntimeError(
                f"Missing grad_sample for parameter "
                f"'{name}' in dataset "
                f"'{dataset_id}'."
            )

        if isinstance(
            grad_sample,
            list,
        ):

            grad_sample = torch.cat(
                grad_sample,
                dim=0,
            )

        # ------------------------------------------------------------------------------------------
        # Expected shape:
        #
        # [batch_size, *parameter.shape]
        # ------------------------------------------------------------------------------------------

        expected_shape = (
            int(x_real.shape[0]),
            *tuple(parameter.shape),
        )

        shape_valid = (
            tuple(grad_sample.shape)
            == expected_shape
        )

        if shape_valid:
            shape_valid_count += 1

        if not shape_valid:

            raise RuntimeError(
                f"Invalid grad_sample shape for "
                f"'{name}' in '{dataset_id}'. "
                f"Observed={tuple(grad_sample.shape)}, "
                f"Expected={expected_shape}"
            )

        observed_batch_sizes.add(
            int(grad_sample.shape[0])
        )

        finite = bool(
            torch.isfinite(
                grad_sample
            ).all().item()
        )

        if finite:
            finite_count += 1

        flattened = grad_sample.reshape(
            grad_sample.shape[0],
            -1,
        )

        if flattened.shape[0] > 1:

            sample_variation = float(
                flattened.std(
                    dim=0,
                    unbiased=False,
                ).mean().item()
            )

        else:

            sample_variation = 0.0

        if sample_variation > 0:
            variation_count += 1

        GRADIENT_TENSOR_ROWS.append({
            "dataset": dataset_id,
            "parameter": name,
            "parameter_shape": list(
                parameter.shape
            ),
            "grad_sample_shape": list(
                grad_sample.shape
            ),
            "expected_grad_sample_shape": list(
                expected_shape
            ),
            "shape_valid": shape_valid,
            "finite": finite,
            "sample_variation": sample_variation,
        })

        GRADIENT_VARIATION_ROWS.append({
            "dataset": dataset_id,
            "parameter": name,
            "sample_variation": sample_variation,
            "variation_detected": (
                sample_variation > 0
            ),
        })

    # ----------------------------------------------------------------------------------------------
    # Dataset-level validation
    # ----------------------------------------------------------------------------------------------

    expected_parameter_count = parameter_count

    gradient_pass = all([
        parameter_count > 0,
        shape_valid_count == expected_parameter_count,
        finite_count == expected_parameter_count,
        variation_count > 0,
        observed_batch_sizes == {
            int(x_real.shape[0])
        },
    ])

    GRADIENT_VALIDATION_ROWS.append({
        "dataset": dataset_id,
        "critic_class": CRITIC_CLASS.__name__,
        "critic_input_dim": expected_input_dim,
        "observed_batch_size": int(
            x_real.shape[0]
        ),
        "parameter_count": parameter_count,
        "shape_valid_parameter_count": shape_valid_count,
        "finite_parameter_count": finite_count,
        "variation_parameter_count": variation_count,
        "grad_samples_present": True,
        "shape_validation": (
            shape_valid_count
            == expected_parameter_count
        ),
        "finite_gradients": (
            finite_count
            == expected_parameter_count
        ),
        "sample_variation_detected": (
            variation_count > 0
        ),
        "secure_mode": False,
        "rng_mode": "STANDARD_PYTORCH_RNG",
        "status": (
            "PASS"
            if gradient_pass
            else "FAIL"
        ),
    })

    print(
        f"Critic class              : "
        f"{CRITIC_CLASS.__name__}"
    )

    print(
        f"Observed input dimension  : "
        f"{x_real.shape[1]}"
    )

    print(
        f"Expected input dimension  : "
        f"{expected_input_dim}"
    )

    print(
        f"Observed batch size       : "
        f"{x_real.shape[0]}"
    )

    print(
        f"Parameters validated      : "
        f"{parameter_count}"
    )

    print(
        f"Valid grad_sample shapes  : "
        f"{shape_valid_count}"
    )

    print(
        f"Finite grad_sample params : "
        f"{finite_count}"
    )

    print(
        f"Variation detected        : "
        f"{variation_count}"
    )

    print(
        f"Status                    : "
        f"{'PASS' if gradient_pass else 'FAIL'}"
    )

    if not gradient_pass:

        raise RuntimeError(
            f"Per-example discriminator gradient "
            f"validation failed for {dataset_id}."
        )


# --------------------------------------------------------------------------------------------------
# 6.9 — Create Validation DataFrames
# --------------------------------------------------------------------------------------------------

GRADIENT_VALIDATION_DF = pd.DataFrame(
    GRADIENT_VALIDATION_ROWS
)

GRADIENT_TENSOR_VALIDATION_DF = pd.DataFrame(
    GRADIENT_TENSOR_ROWS
)

GRADIENT_VARIATION_DF = pd.DataFrame(
    GRADIENT_VARIATION_ROWS
)


# --------------------------------------------------------------------------------------------------
# 6.10 — Global Validation
# --------------------------------------------------------------------------------------------------

if len(
    GRADIENT_VALIDATION_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Section 6 did not produce exactly one "
        "validation record per dataset."
    )

if not (
    GRADIENT_VALIDATION_DF["status"]
    == "PASS"
).all():

    raise RuntimeError(
        "One or more datasets failed "
        "per-example gradient validation."
    )

if not (
    GRADIENT_TENSOR_VALIDATION_DF[
        "shape_valid"
    ].all()
):

    raise RuntimeError(
        "One or more grad_sample tensors "
        "failed shape validation."
    )

if not (
    GRADIENT_TENSOR_VALIDATION_DF[
        "finite"
    ].all()
):

    raise RuntimeError(
        "One or more grad_sample tensors "
        "contain non-finite values."
    )


# --------------------------------------------------------------------------------------------------
# 6.11 — Persist Validation Artifacts
# --------------------------------------------------------------------------------------------------

gradient_validation_path = (
    VALIDATION_DIR
    / "per_example_gradient_validation.csv"
)

gradient_tensor_path = (
    VALIDATION_DIR
    / "per_example_gradient_tensor_validation.csv"
)

gradient_variation_path = (
    VALIDATION_DIR
    / "per_example_gradient_variation.csv"
)

GRADIENT_VALIDATION_DF.to_csv(
    gradient_validation_path,
    index=False,
)

GRADIENT_TENSOR_VALIDATION_DF.to_csv(
    gradient_tensor_path,
    index=False,
)

GRADIENT_VARIATION_DF.to_csv(
    gradient_variation_path,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 6.12 — Section 6 Manifest
# --------------------------------------------------------------------------------------------------

GRADIENT_MANIFEST = {
    "notebook": "Notebook 10",
    "section": 6,
    "title": (
        "Define Per-Example Discriminator Gradients"
    ),
    "critic_class": CRITIC_CLASS.__name__,
    "critic_constructor": str(
        CRITIC_SIGNATURE
    ),
    "datasets": list(DATASET_IDS),
    "grad_sample_mode": "hooks",
    "opacus_version": (
        opacus.__version__
        if "opacus" in globals()
        else None
    ),
    "secure_mode": False,
    "rng_mode": "STANDARD_PYTORCH_RNG",
    "manual_gradient_calculation": False,
    "production_gradient_provider": "Opacus",
    "validation_checks": [
        "grad_sample_presence",
        "grad_sample_shape",
        "finite_grad_sample_values",
        "sample_level_gradient_variation",
        "critic_input_dimension",
        "critic_output_shape",
    ],
    "status": "PASS",
}

gradient_manifest_path = (
    METADATA_DIR
    / "section_06_per_example_gradient_manifest.json"
)

with open(
    gradient_manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        GRADIENT_MANIFEST,
        f,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 6.13 — Final Summary
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("PER-EXAMPLE GRADIENT VALIDATION SUMMARY")
print("-" * 100)

print(
    GRADIENT_VALIDATION_DF.to_string(
        index=False
    )
)

print("-" * 100)
print("PERSISTED ARTIFACTS")
print("-" * 100)

print(
    f"✓ {gradient_validation_path}"
)

print(
    f"✓ {gradient_tensor_path}"
)

print(
    f"✓ {gradient_variation_path}"
)

print(
    f"✓ {gradient_manifest_path}"
)

print("-" * 100)
print(
    "✓ Notebook 08 frozen SPPGANCritic interface preserved."
)

print(
    "✓ No hidden_dims constructor argument injected."
)

print(
    "✓ Per-example discriminator gradients validated through Opacus."
)

print(
    "✓ Section 6 PASS"
)

SECTION 6 — DEFINE PER-EXAMPLE DISCRIMINATOR GRADIENTS
----------------------------------------------------------------------------------------------------
FROZEN SPP-GAN CRITIC INTERFACE
----------------------------------------------------------------------------------------------------
Critic class : SPPGANCritic
Constructor  : (self, input_dim)
----------------------------------------------------------------------------------------------------
Constructor parameters
----------------------------------------------------------------------------------------------------
input_dim                     : kind=POSITIONAL_OR_KEYWORD, default=<required>
----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Constructor kwargs for adult_income: {'input_dim': 105}
Critic class              : SPPGANCritic
Observed input dimension 

/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/tmp/ipykernel_474/2862002326.py:609: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()
/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/tmp/ipykernel_474/2862002326.py:609: 

Critic class              : SPPGANCritic
Observed input dimension  : 2329
Expected input dimension  : 2329
Observed batch size       : 18
Parameters validated      : 6
Valid grad_sample shapes  : 6
Finite grad_sample params : 6
Variation detected        : 5
Status                    : PASS
----------------------------------------------------------------------------------------------------
PER-EXAMPLE GRADIENT VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------
       dataset critic_class  critic_input_dim  observed_batch_size  parameter_count  shape_valid_parameter_count  finite_parameter_count  variation_parameter_count  grad_samples_present  shape_validation  finite_gradients  sample_variation_detected  secure_mode             rng_mode status
  adult_income SPPGANCritic               105                   18                6                            6                       6                          5              

In [62]:
# ==================================================================================================
# SECTION 7 — DEFINE GRADIENT CLIPPING
# ==================================================================================================

print("=" * 100)
print("SECTION 7 — DEFINE GRADIENT CLIPPING")
print("=" * 100)

import json
import math
import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------------------
# 7.1 — Validation-Only Flat Clipping Function
# --------------------------------------------------------------------------------------------------

def clip_per_example_gradients(
    gradients,
    max_grad_norm,
):
    """
    Validation-only mathematical implementation of flat
    per-example L2 gradient clipping.

    Production DP-SGD clipping is performed by Opacus.
    """

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be > 0."
        )

    clipped = []

    for gradient in gradients:

        gradient = gradient.clone()

        norm = torch.linalg.vector_norm(
            gradient
        )

        if norm > max_grad_norm:
            gradient = (
                gradient
                * (
                    max_grad_norm
                    / norm
                )
            )

        clipped.append(gradient)

    return clipped

# --------------------------------------------------------------------------------------------------
# 7.2 — Mathematical Validation Cases
# --------------------------------------------------------------------------------------------------

CLIPPING_ROWS = []

TEST_CASES = {
    "zero": torch.zeros(32),
    "below_threshold": torch.ones(32) * 0.05,
    "exact_threshold": torch.ones(32) / math.sqrt(32),
    "above_threshold": torch.ones(32),
}

for case_name, gradient in TEST_CASES.items():

    original_norm = float(
        torch.linalg.vector_norm(
            gradient
        ).item()
    )

    clipped = clip_per_example_gradients(
        [gradient],
        MAX_GRAD_NORM,
    )[0]

    clipped_norm = float(
        torch.linalg.vector_norm(
            clipped
        ).item()
    )

    CLIPPING_ROWS.append({
        "case": case_name,
        "original_norm": original_norm,
        "clipped_norm": clipped_norm,
        "threshold": MAX_GRAD_NORM,
        "within_threshold": (
            clipped_norm <=
            MAX_GRAD_NORM + 1e-6
        ),
        "finite": bool(
            torch.isfinite(
                clipped
            ).all().item()
        ),
    })

CLIPPING_VALIDATION_DF = pd.DataFrame(
    CLIPPING_ROWS
)

# --------------------------------------------------------------------------------------------------
# 7.3 — Dataset-Shaped Validation
# --------------------------------------------------------------------------------------------------

DATASET_CLIPPING_ROWS = []

rng = np.random.default_rng(2025)

for dataset_id in DATASET_IDS:

    raw_gradients = []

    for _ in range(32):
        raw_gradients.append(
            torch.tensor(
                rng.normal(
                    0,
                    1,
                    size=128,
                ),
                dtype=torch.float32,
            )
        )

    raw_norms = np.array([
        float(
            torch.linalg.vector_norm(
                gradient
            ).item()
        )
        for gradient in raw_gradients
    ])

    clipped_gradients = clip_per_example_gradients(
        raw_gradients,
        MAX_GRAD_NORM,
    )

    clipped_norms = np.array([
        float(
            torch.linalg.vector_norm(
                gradient
            ).item()
        )
        for gradient in clipped_gradients
    ])

    DATASET_CLIPPING_ROWS.append({
        "dataset": dataset_id,
        "samples": len(raw_gradients),
        "raw_max_norm": float(
            raw_norms.max()
        ),
        "clipped_max_norm": float(
            clipped_norms.max()
        ),
        "threshold": MAX_GRAD_NORM,
        "all_within_threshold": bool(
            np.all(
                clipped_norms <=
                MAX_GRAD_NORM + 1e-6
            )
        ),
        "finite": bool(
            np.isfinite(
                clipped_norms
            ).all()
        ),
    })

DATASET_CLIPPING_DF = pd.DataFrame(
    DATASET_CLIPPING_ROWS
)

# --------------------------------------------------------------------------------------------------
# 7.4 — Production Boundary
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("PRODUCTION BOUNDARY")
print("-" * 100)

print(
    "✓ Manual clipping is validation-only."
)

print(
    "✓ Production DP-SGD clipping is delegated to Opacus."
)

print(
    f"✓ Maximum gradient norm C = {MAX_GRAD_NORM}"
)

# --------------------------------------------------------------------------------------------------
# 7.5 — Validation
# --------------------------------------------------------------------------------------------------

if not CLIPPING_VALIDATION_DF[
    "within_threshold"
].all():
    raise RuntimeError(
        "Mathematical clipping validation failed."
    )

if not DATASET_CLIPPING_DF[
    "all_within_threshold"
].all():
    raise RuntimeError(
        "Dataset-shaped clipping validation failed."
    )

if not CLIPPING_VALIDATION_DF[
    "finite"
].all():
    raise RuntimeError(
        "Non-finite clipped gradient detected."
    )

if not DATASET_CLIPPING_DF[
    "finite"
].all():
    raise RuntimeError(
        "Non-finite dataset-shaped clipped gradient detected."
    )

# --------------------------------------------------------------------------------------------------
# 7.6 — Persist Results
# --------------------------------------------------------------------------------------------------

CLIPPING_VALIDATION_DF.to_csv(
    VALIDATION_DIR / "gradient_clipping_validation.csv",
    index=False,
)

DATASET_CLIPPING_DF.to_csv(
    VALIDATION_DIR / "gradient_clipping_dataset_validation.csv",
    index=False,
)

CLIPPING_MANIFEST = {
    "section": 7,
    "clipping": "flat per-example L2 clipping",
    "max_grad_norm": MAX_GRAD_NORM,
    "manual_clipping_production": False,
    "production_provider": "Opacus",
    "datasets": DATASET_IDS,
    "status": "PASS",
}

with open(
    METADATA_DIR / "section_07_gradient_clipping_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        CLIPPING_MANIFEST,
        f,
        indent=2,
    )

print("-" * 100)
print(
    CLIPPING_VALIDATION_DF.to_string(
        index=False
    )
)

print("-" * 100)
print("✓ Gradient clipping validated.")
print("✓ Section 7 PASS")

SECTION 7 — DEFINE GRADIENT CLIPPING
----------------------------------------------------------------------------------------------------
PRODUCTION BOUNDARY
----------------------------------------------------------------------------------------------------
✓ Manual clipping is validation-only.
✓ Production DP-SGD clipping is delegated to Opacus.
✓ Maximum gradient norm C = 1.0
----------------------------------------------------------------------------------------------------
           case  original_norm  clipped_norm  threshold  within_threshold  finite
           zero       0.000000      0.000000        1.0              True    True
below_threshold       0.282843      0.282843        1.0              True    True
exact_threshold       1.000000      1.000000        1.0              True    True
above_threshold       5.656854      1.000000        1.0              True    True
----------------------------------------------------------------------------------------------------
✓ Grad

In [63]:
# ==================================================================================================
# SECTION 8 — DEFINE GAUSSIAN NOISE MECHANISM
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — DEFINE GAUSSIAN NOISE MECHANISM")
print("=" * 100)

import json
import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------------------
# 8.1 — Validation-Only Gaussian Mechanism
# --------------------------------------------------------------------------------------------------

def gaussian_noise(
    shape,
    noise_multiplier,
    max_grad_norm,
    device=None,
    dtype=torch.float32,
):
    """
    Validation-only Gaussian perturbation.

    Standard PyTorch RNG is intentionally used because the final
    implementation does not claim cryptographically secure RNG.

    Production DP noise is generated by Opacus.
    """

    if noise_multiplier <= 0:
        raise ValueError(
            "noise_multiplier must be > 0."
        )

    if max_grad_norm <= 0:
        raise ValueError(
            "max_grad_norm must be > 0."
        )

    return (
        torch.randn(
            shape,
            device=device,
            dtype=dtype,
        )
        * float(noise_multiplier)
        * float(max_grad_norm)
    )

# --------------------------------------------------------------------------------------------------
# 8.2 — Reference Noise Multiplier
# --------------------------------------------------------------------------------------------------
# Section 8 validates the Gaussian mechanism mathematically.
# Dataset-specific calibrated values are supplied by the privacy
# accounting/calibration pipeline rather than being hard-coded as
# final achieved epsilon values.

REFERENCE_SIGMA = 1.0

# --------------------------------------------------------------------------------------------------
# 8.3 — Distributional Validation
# --------------------------------------------------------------------------------------------------

torch.manual_seed(2025)

N_NOISE = 20000

noise = gaussian_noise(
    shape=(N_NOISE,),
    noise_multiplier=REFERENCE_SIGMA,
    max_grad_norm=MAX_GRAD_NORM,
)

noise_mean = float(
    noise.mean().item()
)

noise_std = float(
    noise.std(
        unbiased=True
    ).item()
)

expected_std = (
    REFERENCE_SIGMA
    * MAX_GRAD_NORM
)

mean_tolerance = 0.05
std_relative_tolerance = 0.05

mean_pass = abs(
    noise_mean
) <= mean_tolerance

std_pass = abs(
    noise_std - expected_std
) <= (
    std_relative_tolerance
    * expected_std
)

if not mean_pass:
    raise RuntimeError(
        f"Gaussian noise mean validation failed: "
        f"{noise_mean}"
    )

if not std_pass:
    raise RuntimeError(
        f"Gaussian noise standard deviation validation failed: "
        f"observed={noise_std}, expected={expected_std}"
    )

# --------------------------------------------------------------------------------------------------
# 8.4 — Independence Validation
# --------------------------------------------------------------------------------------------------

noise_a = gaussian_noise(
    shape=(1024,),
    noise_multiplier=REFERENCE_SIGMA,
    max_grad_norm=MAX_GRAD_NORM,
)

noise_b = gaussian_noise(
    shape=(1024,),
    noise_multiplier=REFERENCE_SIGMA,
    max_grad_norm=MAX_GRAD_NORM,
)

if torch.equal(
    noise_a,
    noise_b,
):
    raise RuntimeError(
        "Independent Gaussian draws unexpectedly identical."
    )

# --------------------------------------------------------------------------------------------------
# 8.5 — Additive Consistency
# --------------------------------------------------------------------------------------------------

gradient = torch.ones(
    1024,
    dtype=torch.float32,
)

noise_c = gaussian_noise(
    shape=gradient.shape,
    noise_multiplier=REFERENCE_SIGMA,
    max_grad_norm=MAX_GRAD_NORM,
)

perturbed_gradient = (
    gradient + noise_c
)

if not torch.allclose(
    perturbed_gradient - gradient,
    noise_c,
    atol=1e-7,
):
    raise RuntimeError(
        "Gaussian additive perturbation validation failed."
    )

# --------------------------------------------------------------------------------------------------
# 8.6 — Zero Gradient Validation
# --------------------------------------------------------------------------------------------------

zero_gradient = torch.zeros(
    1024,
    dtype=torch.float32,
)

noise_d = gaussian_noise(
    shape=zero_gradient.shape,
    noise_multiplier=REFERENCE_SIGMA,
    max_grad_norm=MAX_GRAD_NORM,
)

zero_perturbed = (
    zero_gradient + noise_d
)

if not torch.isfinite(
    zero_perturbed
).all():
    raise RuntimeError(
        "Gaussian perturbation generated non-finite values."
    )

if torch.allclose(
    zero_perturbed,
    zero_gradient,
):
    raise RuntimeError(
        "Gaussian perturbation appears inactive."
    )

# --------------------------------------------------------------------------------------------------
# 8.7 — Validation Summary
# --------------------------------------------------------------------------------------------------

NOISE_VALIDATION_ROWS = [
    {
        "test": "distribution_mean",
        "observed": noise_mean,
        "expected": 0.0,
        "passed": mean_pass,
    },
    {
        "test": "distribution_std",
        "observed": noise_std,
        "expected": expected_std,
        "passed": std_pass,
    },
    {
        "test": "independent_draws",
        "observed": True,
        "expected": True,
        "passed": True,
    },
    {
        "test": "additive_consistency",
        "observed": True,
        "expected": True,
        "passed": True,
    },
    {
        "test": "zero_gradient_finite",
        "observed": True,
        "expected": True,
        "passed": True,
    },
]

NOISE_VALIDATION_DF = pd.DataFrame(
    NOISE_VALIDATION_ROWS
)

# --------------------------------------------------------------------------------------------------
# 8.8 — Production Boundary
# --------------------------------------------------------------------------------------------------

print("-" * 100)
print("GAUSSIAN MECHANISM")
print("-" * 100)

print(
    f"Noise multiplier used for validation : {REFERENCE_SIGMA}"
)

print(
    f"Clipping norm C                      : {MAX_GRAD_NORM}"
)

print(
    f"Expected standard deviation          : {expected_std:.6f}"
)

print(
    f"Observed standard deviation          : {noise_std:.6f}"
)

print(
    f"Observed mean                        : {noise_mean:.6f}"
)

print("-" * 100)
print("PRODUCTION BOUNDARY")
print("-" * 100)

print(
    "✓ Manual Gaussian noise is validation-only."
)

print(
    "✓ Production DP noise is generated by Opacus."
)

print(
    "✓ Standard PyTorch RNG is explicitly documented."
)

print(
    "✓ No cryptographic RNG claim is made."
)

# --------------------------------------------------------------------------------------------------
# 8.9 — Persist Results
# --------------------------------------------------------------------------------------------------

NOISE_VALIDATION_DF.to_csv(
    VALIDATION_DIR / "gaussian_noise_validation.csv",
    index=False,
)

NOISE_MANIFEST = {
    "section": 8,
    "mechanism": "Gaussian",
    "validation_noise_multiplier": REFERENCE_SIGMA,
    "max_grad_norm": MAX_GRAD_NORM,
    "manual_noise_production": False,
    "production_provider": "Opacus",
    "rng_mode": "STANDARD_PYTORCH_RNG",
    "cryptographic_rng_claim": False,
    "status": "PASS",
}

with open(
    METADATA_DIR / "section_08_gaussian_noise_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        NOISE_MANIFEST,
        f,
        indent=2,
    )

print("-" * 100)
print(
    NOISE_VALIDATION_DF.to_string(
        index=False
    )
)

print("-" * 100)
print("✓ Gaussian noise mechanism validated.")
print("✓ Section 8 PASS")

SECTION 8 — DEFINE GAUSSIAN NOISE MECHANISM
----------------------------------------------------------------------------------------------------
GAUSSIAN MECHANISM
----------------------------------------------------------------------------------------------------
Noise multiplier used for validation : 1.0
Clipping norm C                      : 1.0
Expected standard deviation          : 1.000000
Observed standard deviation          : 0.992424
Observed mean                        : -0.000445
----------------------------------------------------------------------------------------------------
PRODUCTION BOUNDARY
----------------------------------------------------------------------------------------------------
✓ Manual Gaussian noise is validation-only.
✓ Production DP noise is generated by Opacus.
✓ Standard PyTorch RNG is explicitly documented.
✓ No cryptographic RNG claim is made.
----------------------------------------------------------------------------------------------------
    

In [64]:
# ==================================================================================================
# SECTION 9 — DEFINE DP-SGD DISCRIMINATOR UPDATE
# ==================================================================================================

print("=" * 100)
print("SECTION 9 — DEFINE DP-SGD DISCRIMINATOR UPDATE")
print("=" * 100)

import inspect
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator


# --------------------------------------------------------------------------------------------------
# 1. VALIDATE REQUIRED OBJECTS FROM PREVIOUS SECTIONS
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_9_OBJECTS = [
    "SPPGANCritic",
    "CRITIC_INPUT_DIMS",
    "DATASET_IDS",
    "MAX_GRAD_NORM",
    "REFERENCE_SIGMA",
    "BATCH_SIZE",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_9_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 9 is missing required objects from previous sections: "
        + ", ".join(missing_objects)
    )

print("✓ Required Notebook 10 objects detected")


# --------------------------------------------------------------------------------------------------
# 2. RESOLVE MASTER SEED
# --------------------------------------------------------------------------------------------------

SECTION_09_MASTER_SEED = 2025

print(
    f"✓ Section 9 validation seed : "
    f"{SECTION_09_MASTER_SEED}"
)


# --------------------------------------------------------------------------------------------------
# 3. FROZEN NOTEBOOK 08 CRITIC FACTORY
# --------------------------------------------------------------------------------------------------

def build_frozen_critic(dataset_id):
    """
    Instantiate the exact frozen Notebook 08 SPP-GAN critic.

    Frozen constructor:
        SPPGANCritic(input_dim)

    No hidden_dims argument is permitted.
    No alternative architecture is reconstructed here.
    """

    if dataset_id not in CRITIC_INPUT_DIMS:
        raise KeyError(
            f"Missing critic input dimension for dataset: {dataset_id}"
        )

    input_dim = int(
        CRITIC_INPUT_DIMS[dataset_id]
    )

    constructor_signature = inspect.signature(
        SPPGANCritic.__init__
    )

    constructor_parameters = [
        parameter
        for parameter in constructor_signature.parameters.values()
        if parameter.name != "self"
    ]

    constructor_parameter_names = [
        parameter.name
        for parameter in constructor_parameters
    ]

    if "input_dim" not in constructor_parameter_names:
        raise RuntimeError(
            "Frozen SPPGANCritic constructor does not expose "
            f"'input_dim'. Observed constructor: "
            f"{constructor_signature}"
        )

    if "hidden_dims" in constructor_parameter_names:
        raise RuntimeError(
            "Unexpected 'hidden_dims' argument detected in frozen "
            f"SPPGANCritic constructor: {constructor_signature}"
        )

    critic = SPPGANCritic(
        input_dim=input_dim
    )

    observed_input_dim = int(
        getattr(
            critic,
            "input_dim",
            input_dim,
        )
    )

    if observed_input_dim != input_dim:
        raise RuntimeError(
            f"{dataset_id}: critic input dimension mismatch. "
            f"Expected={input_dim}, "
            f"Observed={observed_input_dim}"
        )

    return critic, constructor_signature


# --------------------------------------------------------------------------------------------------
# 4. DP-SGD VALIDATION CONFIGURATION
# --------------------------------------------------------------------------------------------------

DP_SGD_VALIDATION_CONFIG = {
    "optimizer": "DP-SGD",
    "accountant": "rdp",
    "sampling": "poisson",
    "clipping": "flat",
    "grad_sample_mode": "hooks",
    "max_grad_norm": float(MAX_GRAD_NORM),
    "noise_multiplier": float(REFERENCE_SIGMA),
    "batch_size": int(BATCH_SIZE),
    "secure_mode": False,
    "protected_component": "discriminator",
    "rng_mode": "STANDARD_PYTORCH_RNG",
    "cryptographically_secure_rng": False,
    "manual_production_clipping": False,
    "manual_production_noise": False,
}

print("\nDP-SGD VALIDATION CONFIGURATION")
print("-" * 100)

for key, value in DP_SGD_VALIDATION_CONFIG.items():
    print(
        f"{key:<36}: {value}"
    )

print("-" * 100)

assert DP_SGD_VALIDATION_CONFIG["optimizer"] == "DP-SGD"
assert DP_SGD_VALIDATION_CONFIG["accountant"] == "rdp"
assert DP_SGD_VALIDATION_CONFIG["sampling"] == "poisson"
assert DP_SGD_VALIDATION_CONFIG["clipping"] == "flat"
assert DP_SGD_VALIDATION_CONFIG["grad_sample_mode"] == "hooks"
assert DP_SGD_VALIDATION_CONFIG["secure_mode"] is False
assert DP_SGD_VALIDATION_CONFIG["manual_production_clipping"] is False
assert DP_SGD_VALIDATION_CONFIG["manual_production_noise"] is False


# --------------------------------------------------------------------------------------------------
# 5. CREATE SYNTHETIC MECHANISM-VALIDATION DATASET
# --------------------------------------------------------------------------------------------------

def create_validation_dataset(
    input_dim,
    dataset_size,
    seed,
):
    """
    Create paired synthetic validation records.

    Each item contains:
        (real_record, fake_record)

    These tensors are mechanism-validation data only.
    """

    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        int(seed)
    )

    real_data = torch.randn(
        dataset_size,
        input_dim,
        generator=generator,
        dtype=torch.float32,
    )

    fake_data = torch.randn(
        dataset_size,
        input_dim,
        generator=generator,
        dtype=torch.float32,
    )

    return TensorDataset(
        real_data,
        fake_data,
    )


# --------------------------------------------------------------------------------------------------
# 6. DEFINE WGAN-STYLE CRITIC LOSS
# --------------------------------------------------------------------------------------------------

def wgan_critic_loss(
    real_scores,
    fake_scores,
):
    """
    WGAN-style critic objective:

        L_D = E[D(fake)] - E[D(real)]

    Used only for validating the DP-SGD discriminator update.
    """

    return (
        fake_scores.mean()
        - real_scores.mean()
    )


# --------------------------------------------------------------------------------------------------
# 7. DEFINE ROBUST POISSON-SAMPLING VERIFICATION
# --------------------------------------------------------------------------------------------------

def verify_opacus_poisson_sampling(
    private_data_loader,
):
    """
    Verify that Opacus returned its DPDataLoader and that the loader
    exposes the sampling behavior expected from poisson_sampling=True.

    Important:
    Opacus does not require the returned class name to contain the word
    'Poisson'. The returned loader is commonly named DPDataLoader.

    Therefore class-name matching is NOT used as the primary test.
    """

    loader_class_name = (
        private_data_loader.__class__.__name__
    )

    loader_module = (
        private_data_loader.__class__.__module__
    )

    # Opacus converts the original DataLoader into DPDataLoader when
    # poisson_sampling=True.
    is_opacus_dp_loader = (
        loader_class_name == "DPDataLoader"
        or (
            "opacus" in loader_module.lower()
            and "dataloader" in loader_class_name.lower()
        )
    )

    # Inspect the batch sampler because this is the actual sampling
    # component attached to the returned loader.
    batch_sampler = getattr(
        private_data_loader,
        "batch_sampler",
        None,
    )

    batch_sampler_name = (
        batch_sampler.__class__.__name__
        if batch_sampler is not None
        else ""
    )

    batch_sampler_module = (
        batch_sampler.__class__.__module__
        if batch_sampler is not None
        else ""
    )

    batch_sampler_repr = (
        repr(batch_sampler)
        if batch_sampler is not None
        else ""
    )

    # Opacus DPDataLoader uses a uniform-with-replacement / Poisson-style
    # sampling process internally. The returned DPDataLoader itself is
    # the important public mechanism-level indicator.
    #
    # We therefore accept the canonical Opacus DPDataLoader representation
    # rather than requiring the class name to contain "Poisson".
    verified = bool(
        is_opacus_dp_loader
    )

    return {
        "verified": verified,
        "loader_class": loader_class_name,
        "loader_module": loader_module,
        "batch_sampler_class": batch_sampler_name,
        "batch_sampler_module": batch_sampler_module,
        "batch_sampler_repr": batch_sampler_repr,
    }


# --------------------------------------------------------------------------------------------------
# 8. INITIALIZE VALIDATION CONTAINERS
# --------------------------------------------------------------------------------------------------

DP_SGD_VALIDATION_RESULTS = []
DP_SGD_GRADIENT_VALIDATION = []
DP_SGD_PARAMETER_VALIDATION = []


# --------------------------------------------------------------------------------------------------
# 9. RUN OPACUS DP-SGD UPDATE FOR EACH DATASET
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(DATASET_IDS):

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    try:

        # ------------------------------------------------------------------------------------------
        # 9.1 BUILD EXACT FROZEN CRITIC
        # ------------------------------------------------------------------------------------------

        input_dim = int(
            CRITIC_INPUT_DIMS[dataset_id]
        )

        critic, constructor_signature = build_frozen_critic(
            dataset_id
        )

        print("FROZEN SPP-GAN CRITIC INTERFACE")
        print(
            f"Critic class        : "
            f"{critic.__class__.__name__}"
        )
        print(
            f"Constructor         : "
            f"{constructor_signature}"
        )
        print(
            f"Input dimension     : "
            f"{input_dim}"
        )

        # ------------------------------------------------------------------------------------------
        # 9.2 OPACUS COMPATIBILITY VALIDATION
        # ------------------------------------------------------------------------------------------

        pre_validation_errors = ModuleValidator.validate(
            critic,
            strict=False,
        )

        print(
            f"Opacus pre-validation issues : "
            f"{len(pre_validation_errors)}"
        )

        critic = ModuleValidator.fix(
            critic
        )

        post_validation_errors = ModuleValidator.validate(
            critic,
            strict=False,
        )

        if post_validation_errors:
            raise RuntimeError(
                f"{dataset_id}: Opacus compatibility validation "
                "failed after ModuleValidator.fix(). "
                f"Remaining issues: {post_validation_errors}"
            )

        print(
            "✓ Opacus compatibility validation passed"
        )

        # ------------------------------------------------------------------------------------------
        # 9.3 DETERMINE VALIDATION DATASET SIZE
        # ------------------------------------------------------------------------------------------

        validation_dataset_size = min(
            32,
            max(
                16,
                input_dim // 8,
            ),
        )

        if dataset_id == "diabetes_130us":
            validation_dataset_size = 16

        print(
            f"Validation dataset size        : "
            f"{validation_dataset_size}"
        )

        # ------------------------------------------------------------------------------------------
        # 9.4 CREATE VALIDATION DATASET
        # ------------------------------------------------------------------------------------------

        validation_dataset = create_validation_dataset(
            input_dim=input_dim,
            dataset_size=validation_dataset_size,
            seed=(
                SECTION_09_MASTER_SEED
                + dataset_index
                + 9000
            ),
        )

        # ------------------------------------------------------------------------------------------
        # 9.5 CREATE STANDARD PYTORCH DATALOADER
        # ------------------------------------------------------------------------------------------

        validation_data_loader = DataLoader(
            validation_dataset,
            batch_size=validation_dataset_size,
            shuffle=False,
            drop_last=False,
        )

        if not hasattr(
            validation_data_loader,
            "dataset",
        ):
            raise RuntimeError(
                f"{dataset_id}: validation DataLoader "
                "does not expose .dataset"
            )

        print(
            "✓ PyTorch DataLoader created"
        )

        print(
            f"DataLoader dataset size       : "
            f"{len(validation_data_loader.dataset)}"
        )

        # ------------------------------------------------------------------------------------------
        # 9.6 CREATE BASE OPTIMIZER
        # ------------------------------------------------------------------------------------------

        optimizer = torch.optim.Adam(
            critic.parameters(),
            lr=2e-4,
        )

        # ------------------------------------------------------------------------------------------
        # 9.7 CREATE OPACUS PRIVACY ENGINE
        # ------------------------------------------------------------------------------------------

        privacy_engine = PrivacyEngine(
            accountant="rdp",
            secure_mode=False,
        )

        # ------------------------------------------------------------------------------------------
        # 9.8 ATTACH OPACUS DP-SGD
        # ------------------------------------------------------------------------------------------

        critic, private_optimizer, private_data_loader = (
            privacy_engine.make_private(
                module=critic,
                optimizer=optimizer,
                data_loader=validation_data_loader,
                noise_multiplier=float(
                    REFERENCE_SIGMA
                ),
                max_grad_norm=float(
                    MAX_GRAD_NORM
                ),
                poisson_sampling=True,
                clipping="flat",
                grad_sample_mode="hooks",
            )
        )

        # ------------------------------------------------------------------------------------------
        # 9.9 VERIFY OPACUS DP-SGD CONFIGURATION
        # ------------------------------------------------------------------------------------------

        observed_noise_multiplier = float(
            getattr(
                private_optimizer,
                "noise_multiplier",
                np.nan,
            )
        )

        observed_max_grad_norm = float(
            getattr(
                private_optimizer,
                "max_grad_norm",
                np.nan,
            )
        )

        observed_secure_rng = bool(
            getattr(
                private_optimizer,
                "secure_rng",
                False,
            )
        )

        poisson_verification = (
            verify_opacus_poisson_sampling(
                private_data_loader
            )
        )

        sampling_is_poisson = bool(
            poisson_verification["verified"]
        )

        print("\nOPACUS DP-SGD CONFIGURATION")
        print(
            f"Noise multiplier     : "
            f"{observed_noise_multiplier}"
        )
        print(
            f"Maximum grad norm    : "
            f"{observed_max_grad_norm}"
        )
        print(
            f"Secure RNG           : "
            f"{observed_secure_rng}"
        )
        print(
            f"Sampler              : "
            f"{poisson_verification['loader_class']}"
        )
        print(
            f"Sampler module       : "
            f"{poisson_verification['loader_module']}"
        )
        print(
            f"Batch sampler        : "
            f"{poisson_verification['batch_sampler_class']}"
        )
        print(
            f"Poisson sampling     : "
            f"{sampling_is_poisson}"
        )

        if not np.isfinite(
            observed_noise_multiplier
        ):
            raise RuntimeError(
                f"{dataset_id}: Opacus noise multiplier "
                "is not finite."
            )

        if not np.isclose(
            observed_noise_multiplier,
            float(REFERENCE_SIGMA),
            rtol=1e-6,
            atol=1e-8,
        ):
            raise RuntimeError(
                f"{dataset_id}: noise multiplier mismatch. "
                f"Expected={REFERENCE_SIGMA}, "
                f"Observed={observed_noise_multiplier}"
            )

        if not np.isclose(
            observed_max_grad_norm,
            float(MAX_GRAD_NORM),
            rtol=1e-6,
            atol=1e-8,
        ):
            raise RuntimeError(
                f"{dataset_id}: max_grad_norm mismatch. "
                f"Expected={MAX_GRAD_NORM}, "
                f"Observed={observed_max_grad_norm}"
            )

        if observed_secure_rng:
            raise RuntimeError(
                f"{dataset_id}: secure RNG unexpectedly enabled. "
                "Notebook 10 requires STANDARD_PYTORCH_RNG."
            )

        if not sampling_is_poisson:
            raise RuntimeError(
                f"{dataset_id}: Opacus DPDataLoader "
                "could not be verified."
            )

        print(
            "✓ Noise multiplier verified"
        )
        print(
            "✓ Flat clipping norm verified"
        )
        print(
            "✓ Standard PyTorch RNG boundary verified"
        )
        print(
            "✓ Opacus DPDataLoader / Poisson sampling verified"
        )

        # ------------------------------------------------------------------------------------------
        # 9.10 SNAPSHOT PARAMETERS BEFORE DP-SGD UPDATE
        # ------------------------------------------------------------------------------------------

        parameters_before = {
            name: parameter.detach().clone()
            for name, parameter in critic.named_parameters()
            if parameter.requires_grad
        }

        trainable_parameter_count = len(
            parameters_before
        )

        # ------------------------------------------------------------------------------------------
        # 9.11 OBTAIN ONE POISSON-SAMPLED BATCH
        # ------------------------------------------------------------------------------------------

        private_iterator = iter(
            private_data_loader
        )

        try:
            sampled_real, sampled_fake = next(
                private_iterator
            )
        except StopIteration:
            raise RuntimeError(
                f"{dataset_id}: Opacus DPDataLoader "
                "produced no validation batch."
            )

        actual_batch_size = int(
            sampled_real.shape[0]
        )

        if actual_batch_size <= 0:
            raise RuntimeError(
                f"{dataset_id}: sampled batch is empty."
            )

        if sampled_real.shape != sampled_fake.shape:
            raise RuntimeError(
                f"{dataset_id}: real/fake batch shape mismatch. "
                f"Real={tuple(sampled_real.shape)}, "
                f"Fake={tuple(sampled_fake.shape)}"
            )

        print(
            f"Actual sampled batch size : "
            f"{actual_batch_size}"
        )

        # ------------------------------------------------------------------------------------------
        # 9.12 FORWARD PASS
        # ------------------------------------------------------------------------------------------

        real_scores = critic(
            sampled_real
        )

        fake_scores = critic(
            sampled_fake
        )

        real_scores = real_scores.reshape(-1)
        fake_scores = fake_scores.reshape(-1)

        if real_scores.numel() != actual_batch_size:
            raise RuntimeError(
                f"{dataset_id}: real critic output "
                "size mismatch."
            )

        if fake_scores.numel() != actual_batch_size:
            raise RuntimeError(
                f"{dataset_id}: fake critic output "
                "size mismatch."
            )

        if not torch.isfinite(
            real_scores
        ).all():
            raise RuntimeError(
                f"{dataset_id}: non-finite real critic scores."
            )

        if not torch.isfinite(
            fake_scores
        ).all():
            raise RuntimeError(
                f"{dataset_id}: non-finite fake critic scores."
            )

        # ------------------------------------------------------------------------------------------
        # 9.13 WGAN-STYLE CRITIC LOSS
        # ------------------------------------------------------------------------------------------

        loss = wgan_critic_loss(
            real_scores=real_scores,
            fake_scores=fake_scores,
        )

        if not torch.isfinite(
            loss
        ):
            raise RuntimeError(
                f"{dataset_id}: critic loss is non-finite."
            )

        print(
            f"Critic loss          : "
            f"{float(loss.detach().cpu().item()):.8f}"
        )

        # ------------------------------------------------------------------------------------------
        # 9.14 BACKWARD PASS
        # ------------------------------------------------------------------------------------------

        private_optimizer.zero_grad()

        loss.backward()

        # ------------------------------------------------------------------------------------------
        # 9.15 VALIDATE PER-EXAMPLE GRADIENTS
        # ------------------------------------------------------------------------------------------

        grad_sample_parameter_count = 0
        finite_grad_sample_parameter_count = 0

        invalid_grad_sample_parameters = []

        for name, parameter in critic.named_parameters():

            if not parameter.requires_grad:
                continue

            grad_sample = getattr(
                parameter,
                "grad_sample",
                None,
            )

            if grad_sample is None:
                invalid_grad_sample_parameters.append(
                    (
                        name,
                        "missing",
                    )
                )
                continue

            grad_sample_parameter_count += 1

            if not torch.isfinite(
                grad_sample
            ).all():
                invalid_grad_sample_parameters.append(
                    (
                        name,
                        "non_finite",
                    )
                )
                continue

            finite_grad_sample_parameter_count += 1

        print("\nPER-EXAMPLE GRADIENT STATUS")
        print(
            f"Trainable parameters            : "
            f"{trainable_parameter_count}"
        )
        print(
            f"Parameters with grad_sample      : "
            f"{grad_sample_parameter_count}"
        )
        print(
            f"Finite grad_sample parameters    : "
            f"{finite_grad_sample_parameter_count}"
        )

        if invalid_grad_sample_parameters:
            raise RuntimeError(
                f"{dataset_id}: invalid grad_sample parameters: "
                f"{invalid_grad_sample_parameters}"
            )

        if (
            grad_sample_parameter_count
            != trainable_parameter_count
        ):
            raise RuntimeError(
                f"{dataset_id}: not all trainable parameters "
                "have Opacus grad_sample tensors."
            )

        if (
            finite_grad_sample_parameter_count
            != trainable_parameter_count
        ):
            raise RuntimeError(
                f"{dataset_id}: non-finite Opacus "
                "grad_sample detected."
            )

        print(
            "✓ Per-example discriminator gradients verified"
        )

        # ------------------------------------------------------------------------------------------
        # 9.16 EXECUTE ACTUAL OPACUS DP-SGD UPDATE
        # ------------------------------------------------------------------------------------------

        private_optimizer.step()

        # ------------------------------------------------------------------------------------------
        # 9.17 VERIFY PARAMETERS CHANGED
        # ------------------------------------------------------------------------------------------

        changed_parameter_count = 0
        parameter_change_norms = []

        for name, parameter in critic.named_parameters():

            if not parameter.requires_grad:
                continue

            before = parameters_before[name]
            after = parameter.detach()

            difference = after - before

            difference_norm = float(
                torch.linalg.vector_norm(
                    difference.reshape(-1)
                ).item()
            )

            parameter_change_norms.append(
                difference_norm
            )

            if difference_norm > 0.0:
                changed_parameter_count += 1

        parameters_changed = (
            changed_parameter_count > 0
        )

        if not parameters_changed:
            raise RuntimeError(
                f"{dataset_id}: DP-SGD optimizer step "
                "did not change any trainable parameter."
            )

        max_parameter_change_norm = max(
            parameter_change_norms
        )

        print("\nDP-SGD PARAMETER UPDATE")
        print(
            f"Parameters changed               : "
            f"{changed_parameter_count}/"
            f"{trainable_parameter_count}"
        )
        print(
            f"Maximum parameter change norm    : "
            f"{max_parameter_change_norm:.8e}"
        )

        print(
            "✓ DP-SGD optimizer step changed "
            "critic parameters"
        )

        # ------------------------------------------------------------------------------------------
        # 9.18 RECORD RESULTS
        # ------------------------------------------------------------------------------------------

        DP_SGD_VALIDATION_RESULTS.append({
            "dataset_id": dataset_id,
            "critic_input_dim": input_dim,
            "constructor_signature": str(
                constructor_signature
            ),
            "trainable_parameter_count": (
                trainable_parameter_count
            ),
            "validation_dataset_size": (
                validation_dataset_size
            ),
            "actual_poisson_batch_size": (
                actual_batch_size
            ),
            "noise_multiplier": (
                observed_noise_multiplier
            ),
            "max_grad_norm": (
                observed_max_grad_norm
            ),
            "secure_rng": (
                observed_secure_rng
            ),
            "rng_mode": (
                "STANDARD_PYTORCH_RNG"
            ),
            "cryptographically_secure_rng": False,
            "poisson_sampling": (
                sampling_is_poisson
            ),
            "opacus_loader_class": (
                poisson_verification["loader_class"]
            ),
            "opacus_loader_module": (
                poisson_verification["loader_module"]
            ),
            "batch_sampler_class": (
                poisson_verification["batch_sampler_class"]
            ),
            "clipping": "flat",
            "grad_sample_mode": "hooks",
            "grad_sample_parameter_count": (
                grad_sample_parameter_count
            ),
            "finite_grad_sample_parameter_count": (
                finite_grad_sample_parameter_count
            ),
            "changed_parameter_count": (
                changed_parameter_count
            ),
            "parameters_changed": (
                parameters_changed
            ),
            "critic_loss": (
                float(
                    loss.detach().cpu().item()
                )
            ),
            "status": "PASS",
        })

        DP_SGD_GRADIENT_VALIDATION.append({
            "dataset_id": dataset_id,
            "trainable_parameter_count": (
                trainable_parameter_count
            ),
            "grad_sample_parameter_count": (
                grad_sample_parameter_count
            ),
            "finite_grad_sample_parameter_count": (
                finite_grad_sample_parameter_count
            ),
            "actual_poisson_batch_size": (
                actual_batch_size
            ),
            "status": "PASS",
        })

        DP_SGD_PARAMETER_VALIDATION.append({
            "dataset_id": dataset_id,
            "changed_parameter_count": (
                changed_parameter_count
            ),
            "trainable_parameter_count": (
                trainable_parameter_count
            ),
            "parameters_changed": (
                parameters_changed
            ),
            "max_parameter_change_norm": (
                max_parameter_change_norm
            ),
            "status": "PASS",
        })

        print(
            f"\n✓ {dataset_id} DP-SGD update validation PASS"
        )

    except Exception:

        print(
            f"\n✗ {dataset_id} DP-SGD validation FAILED"
        )

        traceback.print_exc()

        raise


# --------------------------------------------------------------------------------------------------
# 10. CREATE VALIDATION DATAFRAMES
# --------------------------------------------------------------------------------------------------

DP_SGD_VALIDATION_DF = pd.DataFrame(
    DP_SGD_VALIDATION_RESULTS
)

DP_SGD_GRADIENT_VALIDATION_DF = pd.DataFrame(
    DP_SGD_GRADIENT_VALIDATION
)

DP_SGD_PARAMETER_VALIDATION_DF = pd.DataFrame(
    DP_SGD_PARAMETER_VALIDATION
)


# --------------------------------------------------------------------------------------------------
# 11. GLOBAL VALIDATION ASSERTIONS
# --------------------------------------------------------------------------------------------------

if DP_SGD_VALIDATION_DF.empty:
    raise RuntimeError(
        "Section 9 produced no DP-SGD validation results."
    )

if len(DP_SGD_VALIDATION_DF) != len(
    DATASET_IDS
):
    raise RuntimeError(
        "Section 9 did not produce exactly one "
        "validation result for every dataset."
    )

assert (
    DP_SGD_VALIDATION_DF["status"] == "PASS"
).all()

assert (
    DP_SGD_VALIDATION_DF["parameters_changed"]
).all()

assert (
    DP_SGD_VALIDATION_DF["poisson_sampling"]
).all()

assert (
    ~DP_SGD_VALIDATION_DF["secure_rng"]
).all()

assert (
    DP_SGD_VALIDATION_DF["clipping"] == "flat"
).all()

assert (
    DP_SGD_VALIDATION_DF["grad_sample_mode"] == "hooks"
).all()

assert (
    np.isclose(
        DP_SGD_VALIDATION_DF["noise_multiplier"],
        float(REFERENCE_SIGMA),
    )
).all()

assert (
    np.isclose(
        DP_SGD_VALIDATION_DF["max_grad_norm"],
        float(MAX_GRAD_NORM),
    )
).all()


# --------------------------------------------------------------------------------------------------
# 12. DISPLAY FINAL VALIDATION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("DP-SGD DISCRIMINATOR UPDATE VALIDATION SUMMARY")
print("=" * 100)

summary_columns = [
    "dataset_id",
    "critic_input_dim",
    "trainable_parameter_count",
    "actual_poisson_batch_size",
    "noise_multiplier",
    "max_grad_norm",
    "secure_rng",
    "poisson_sampling",
    "opacus_loader_class",
    "batch_sampler_class",
    "grad_sample_parameter_count",
    "finite_grad_sample_parameter_count",
    "changed_parameter_count",
    "parameters_changed",
    "status",
]

print(
    DP_SGD_VALIDATION_DF[
        summary_columns
    ].to_string(index=False)
)


# --------------------------------------------------------------------------------------------------
# 13. PERSIST VALIDATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

NB10_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/"
    "results/notebooks/notebook_10"
)

VALIDATION_DIR = (
    NB10_ROOT / "validation"
)

METADATA_DIR = (
    NB10_ROOT / "metadata"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


dp_sgd_validation_path = (
    VALIDATION_DIR /
    "dp_sgd_discriminator_validation.csv"
)

dp_sgd_gradient_path = (
    VALIDATION_DIR /
    "dp_sgd_discriminator_gradient_validation.csv"
)

dp_sgd_parameter_path = (
    VALIDATION_DIR /
    "dp_sgd_discriminator_parameter_validation.csv"
)


DP_SGD_VALIDATION_DF.to_csv(
    dp_sgd_validation_path,
    index=False,
)

DP_SGD_GRADIENT_VALIDATION_DF.to_csv(
    dp_sgd_gradient_path,
    index=False,
)

DP_SGD_PARAMETER_VALIDATION_DF.to_csv(
    dp_sgd_parameter_path,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 14. SAVE SECTION 9 MANIFEST
# --------------------------------------------------------------------------------------------------

section_09_manifest = {
    "notebook": "10",
    "section": "9",
    "section_title": (
        "DEFINE DP-SGD DISCRIMINATOR UPDATE"
    ),
    "project_root": (
        "/content/drive/MyDrive/SPP_GAN_Research"
    ),
    "datasets": list(DATASET_IDS),
    "master_seed": SECTION_09_MASTER_SEED,
    "critic_interface": (
        "SPPGANCritic(input_dim)"
    ),
    "architecture_source": (
        "Notebook 08 frozen architecture"
    ),
    "protected_component": "discriminator",
    "optimizer": "DP-SGD",
    "accountant": "rdp",
    "sampling": "poisson",
    "clipping": "flat",
    "max_grad_norm": float(
        MAX_GRAD_NORM
    ),
    "noise_multiplier": float(
        REFERENCE_SIGMA
    ),
    "secure_mode": False,
    "rng_mode": "STANDARD_PYTORCH_RNG",
    "cryptographically_secure_rng": False,
    "manual_production_clipping": False,
    "manual_production_noise": False,
    "grad_sample_mode": "hooks",
    "validation_only_noise_multiplier": True,
    "final_epsilon_claim": False,
    "final_accounting_deferred_to_notebook_11": True,
    "training_deferred_to_notebook_12": True,
    "synthetic_generation_deferred": True,
    "status": "PASS",
}


manifest_path = (
    METADATA_DIR /
    "section_09_dp_sgd_discriminator_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        section_09_manifest,
        f,
        indent=2,
    )


# --------------------------------------------------------------------------------------------------
# 15. FINAL SECTION 9 VERIFICATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 9 FINAL VERIFICATION")
print("=" * 100)

print(
    f"✓ Datasets validated              : "
    f"{len(DP_SGD_VALIDATION_DF)}"
)

print(
    "✓ Frozen critic interface         : "
    "SPPGANCritic(input_dim)"
)

print(
    "✓ Opacus DP-SGD update            : "
    "validated"
)

print(
    "✓ Per-example gradients           : "
    "validated"
)

print(
    "✓ Flat clipping                   : "
    f"C={MAX_GRAD_NORM}"
)

print(
    "✓ Gaussian noise                  : "
    f"sigma={REFERENCE_SIGMA}"
)

print(
    "✓ Opacus DPDataLoader             : "
    "validated"
)

print(
    "✓ Poisson sampling                : "
    "validated"
)

print(
    "✓ Standard PyTorch RNG            : "
    "explicitly documented"
)

print(
    "✓ Cryptographic RNG claim         : "
    "NOT made"
)

print(
    "✓ Manual production clipping      : "
    "NOT used"
)

print(
    "✓ Manual production noise         : "
    "NOT used"
)

print(
    "✓ Actual DP-SGD parameter update  : "
    "validated"
)

print(
    "✓ Final epsilon                   : "
    "deferred to Notebook 11"
)

print(
    f"✓ Validation artifact             : "
    f"{dp_sgd_validation_path}"
)

print(
    f"✓ Gradient artifact               : "
    f"{dp_sgd_gradient_path}"
)

print(
    f"✓ Parameter artifact              : "
    f"{dp_sgd_parameter_path}"
)

print(
    f"✓ Manifest                        : "
    f"{manifest_path}"
)

print("\n✓ SECTION 9 PASS")
print("=" * 100)

SECTION 9 — DEFINE DP-SGD DISCRIMINATOR UPDATE
✓ Required Notebook 10 objects detected
✓ Section 9 validation seed : 2025

DP-SGD VALIDATION CONFIGURATION
----------------------------------------------------------------------------------------------------
optimizer                           : DP-SGD
accountant                          : rdp
sampling                            : poisson
clipping                            : flat
grad_sample_mode                    : hooks
max_grad_norm                       : 1.0
noise_multiplier                    : 1.0
batch_size                          : 128
secure_mode                         : False
protected_component                 : discriminator
rng_mode                            : STANDARD_PYTORCH_RNG
cryptographically_secure_rng        : False
manual_production_clipping          : False
manual_production_noise             : False
----------------------------------------------------------------------------------------------------

---------

/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/tmp/ipykernel_474/2726176884.py:757: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()



DP-SGD PARAMETER UPDATE
Parameters changed               : 6/6
Maximum parameter change norm    : 1.54486686e-01
✓ DP-SGD optimizer step changed critic parameters

✓ diabetes_130us DP-SGD update validation PASS

DP-SGD DISCRIMINATOR UPDATE VALIDATION SUMMARY
    dataset_id  critic_input_dim  trainable_parameter_count  actual_poisson_batch_size  noise_multiplier  max_grad_norm  secure_rng  poisson_sampling opacus_loader_class           batch_sampler_class  grad_sample_parameter_count  finite_grad_sample_parameter_count  changed_parameter_count  parameters_changed status
  adult_income               105                          6                         16               1.0            1.0       False              True        DPDataLoader UniformWithReplacementSampler                            6                                   6                        6                True   PASS
bank_marketing                51                          6                         16               1.0  

In [65]:
# ==================================================================================================
# 10. DEFINE SAMPLING MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("10. DEFINE SAMPLING MECHANISM")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_10_OBJECTS = [
    "DATASET_IDS",
    "MASTER_SEED",
    "TRAIN_ROWS",
    "DP_BATCH_SIZE",
    "SAMPLING_MECHANISM",
    "PrivacyEngine",
    "SPPGANCritic",
    "CRITIC_INPUT_DIMENSIONS",
    "REFERENCE_SIGMA",
    "MAX_GRAD_NORM",
    "DEVICE",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_10_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 10 is missing required objects: "
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate sampling configuration
# --------------------------------------------------------------------------------------------------

if str(SAMPLING_MECHANISM).lower() != "poisson":
    raise RuntimeError(
        "Notebook 10 requires Poisson sampling. "
        f"Received: {SAMPLING_MECHANISM}"
    )

if int(DP_BATCH_SIZE) <= 0:
    raise RuntimeError(
        "DP_BATCH_SIZE must be positive."
    )

if not isinstance(TRAIN_ROWS, dict):
    raise TypeError(
        "TRAIN_ROWS must be a dictionary."
    )

EXPECTED_DATASETS = {
    str(dataset_id)
    for dataset_id in DATASET_IDS
}

AVAILABLE_TRAIN_ROWS = {
    str(dataset_id)
    for dataset_id in TRAIN_ROWS.keys()
}

if AVAILABLE_TRAIN_ROWS != EXPECTED_DATASETS:
    raise RuntimeError(
        "TRAIN_ROWS dataset coverage does not match DATASET_IDS. "
        f"Expected={sorted(EXPECTED_DATASETS)}, "
        f"Received={sorted(AVAILABLE_TRAIN_ROWS)}"
    )

if not isinstance(CRITIC_INPUT_DIMENSIONS, dict):
    raise TypeError(
        "CRITIC_INPUT_DIMENSIONS must be a dictionary."
    )

AVAILABLE_CRITIC_DIMENSIONS = {
    str(dataset_id)
    for dataset_id in CRITIC_INPUT_DIMENSIONS.keys()
}

if AVAILABLE_CRITIC_DIMENSIONS != EXPECTED_DATASETS:
    raise RuntimeError(
        "CRITIC_INPUT_DIMENSIONS dataset coverage does not match "
        "DATASET_IDS. "
        f"Expected={sorted(EXPECTED_DATASETS)}, "
        f"Received={sorted(AVAILABLE_CRITIC_DIMENSIONS)}"
    )

REFERENCE_SIGMA = float(REFERENCE_SIGMA)

if not np.isfinite(REFERENCE_SIGMA) or REFERENCE_SIGMA <= 0.0:
    raise ValueError(
        "REFERENCE_SIGMA must be finite and strictly positive."
    )

MAX_GRAD_NORM = float(MAX_GRAD_NORM)

if not np.isfinite(MAX_GRAD_NORM) or MAX_GRAD_NORM <= 0.0:
    raise ValueError(
        "MAX_GRAD_NORM must be finite and strictly positive."
    )

print(
    "✓ Sampling configuration validated."
)

print(
    "✓ Configured sampling mechanism : Poisson"
)

print(
    f"✓ Nominal DP batch size        : {int(DP_BATCH_SIZE)}"
)

print(
    f"✓ Reference noise multiplier   : {REFERENCE_SIGMA:.6f}"
)

print(
    f"✓ Maximum gradient norm        : {MAX_GRAD_NORM:.6f}"
)

print(
    "✓ Critic input dimensions      : validated"
)


# --------------------------------------------------------------------------------------------------
# 3. Define diagnostic Bernoulli-inclusion / Poisson-subsampling sampler
# --------------------------------------------------------------------------------------------------

def poisson_sample_indices(
    n_samples,
    sample_rate,
    generator=None,
):
    """
    Diagnostic independent Bernoulli inclusion sampler.

    For each record i:

        I_i ~ Bernoulli(q)

    where q is the sampling probability.

    The realized batch size therefore follows:

        B ~ Binomial(N, q)

    with:

        E[B]   = Nq
        Var[B] = Nq(1-q)

    This function is used ONLY for mathematical validation.

    Production DP training uses the Opacus DPDataLoader.
    """

    n_samples = int(n_samples)
    sample_rate = float(sample_rate)

    if n_samples <= 0:
        raise ValueError(
            "n_samples must be positive."
        )

    if not np.isfinite(sample_rate):
        raise ValueError(
            "sample_rate must be finite."
        )

    if not (
        0.0 < sample_rate <= 1.0
    ):
        raise ValueError(
            "sample_rate must be in (0, 1]."
        )

    included = (
        torch.rand(
            n_samples,
            generator=generator,
            device="cpu",
        )
        < sample_rate
    )

    return torch.nonzero(
        included,
        as_tuple=False,
    ).flatten()


print(
    "✓ Diagnostic Bernoulli-inclusion sampler defined."
)

print(
    "✓ Diagnostic sampler is mathematical validation only."
)


# --------------------------------------------------------------------------------------------------
# 4. Define RAM-safe diagnostic validator
# --------------------------------------------------------------------------------------------------

def validate_poisson_sampling(
    n_samples,
    sample_rate,
    repetitions=1000,
    seed=None,
):
    """
    RAM-safe empirical validation of independent Bernoulli
    inclusion / Poisson subsampling.

    For:

        B ~ Binomial(N, q)

    the expected batch size is:

        E[B] = Nq

    and the variance is:

        Var(B) = Nq(1-q)
    """

    n_samples = int(n_samples)
    sample_rate = float(sample_rate)
    repetitions = int(repetitions)

    if n_samples <= 0:
        raise ValueError(
            "n_samples must be positive."
        )

    if not (
        0.0 < sample_rate <= 1.0
    ):
        raise ValueError(
            "sample_rate must be in (0, 1]."
        )

    if repetitions <= 0:
        raise ValueError(
            "repetitions must be positive."
        )

    if seed is None:
        seed = int(MASTER_SEED)

    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        int(seed)
    )

    batch_sizes = []

    for _ in range(repetitions):

        indices = poisson_sample_indices(
            n_samples=n_samples,
            sample_rate=sample_rate,
            generator=generator,
        )

        batch_sizes.append(
            int(
                indices.numel()
            )
        )

    batch_sizes = np.asarray(
        batch_sizes,
        dtype=np.int64,
    )

    expected_batch_size = (
        n_samples
        *
        sample_rate
    )

    expected_variance = (
        n_samples
        *
        sample_rate
        *
        (
            1.0
            -
            sample_rate
        )
    )

    expected_std = float(
        np.sqrt(
            expected_variance
        )
    )

    observed_mean = float(
        batch_sizes.mean()
    )

    observed_std = float(
        batch_sizes.std(
            ddof=1
        )
    )

    return {
        "n_samples": int(
            n_samples
        ),
        "sample_rate": float(
            sample_rate
        ),
        "repetitions": int(
            repetitions
        ),
        "mean_batch_size": float(
            observed_mean
        ),
        "min_batch_size": int(
            batch_sizes.min()
        ),
        "max_batch_size": int(
            batch_sizes.max()
        ),
        "observed_std_batch_size": float(
            observed_std
        ),
        "expected_batch_size": float(
            expected_batch_size
        ),
        "expected_variance": float(
            expected_variance
        ),
        "expected_std_batch_size": float(
            expected_std
        ),
        "seed": int(
            seed
        ),
    }


print(
    "✓ Diagnostic Poisson validator defined."
)


# --------------------------------------------------------------------------------------------------
# 5. Mathematical validation of diagnostic sampler
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("POISSON SAMPLING MATHEMATICAL VALIDATION")
print("-" * 100)


DIAGNOSTIC_N = 10000
DIAGNOSTIC_Q = 0.10
DIAGNOSTIC_REPETITIONS = 1000

DIAGNOSTIC_RESULT = validate_poisson_sampling(
    n_samples=DIAGNOSTIC_N,
    sample_rate=DIAGNOSTIC_Q,
    repetitions=DIAGNOSTIC_REPETITIONS,
    seed=int(MASTER_SEED),
)


# --------------------------------------------------------------------------------------------------
# 5.1 Expected batch-size validation
# --------------------------------------------------------------------------------------------------

EXPECTED_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "expected_batch_size"
    ]
)

OBSERVED_MEAN_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "mean_batch_size"
    ]
)

EXPECTED_STD_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "expected_std_batch_size"
    ]
)

OBSERVED_STD_BATCH_SIZE = float(
    DIAGNOSTIC_RESULT[
        "observed_std_batch_size"
    ]
)

MEAN_TOLERANCE = float(
    max(
        1.0,
        5.0
        *
        EXPECTED_STD_BATCH_SIZE
        /
        np.sqrt(
            DIAGNOSTIC_REPETITIONS
        ),
    )
)

MEAN_VALID = bool(
    abs(
        OBSERVED_MEAN_BATCH_SIZE
        -
        EXPECTED_BATCH_SIZE
    )
    <=
    MEAN_TOLERANCE
)

if not MEAN_VALID:
    raise RuntimeError(
        "Poisson empirical mean batch size is outside "
        "the validation tolerance. "
        f"Expected={EXPECTED_BATCH_SIZE:.6f}, "
        f"Observed={OBSERVED_MEAN_BATCH_SIZE:.6f}, "
        f"Tolerance={MEAN_TOLERANCE:.6f}"
    )

print(
    f"✓ Expected batch size E[B]=Nq : PASS "
    f"(expected={EXPECTED_BATCH_SIZE:.3f}, "
    f"observed={OBSERVED_MEAN_BATCH_SIZE:.3f})"
)


# --------------------------------------------------------------------------------------------------
# 5.2 Batch-size variability validation
# --------------------------------------------------------------------------------------------------

if EXPECTED_STD_BATCH_SIZE > 0:

    STD_RATIO = (
        OBSERVED_STD_BATCH_SIZE
        /
        EXPECTED_STD_BATCH_SIZE
    )

else:

    STD_RATIO = 1.0


STD_VALID = bool(
    0.90
    <=
    STD_RATIO
    <=
    1.10
)

if not STD_VALID:
    raise RuntimeError(
        "Poisson batch-size standard deviation is outside "
        "the validation interval. "
        f"Expected={EXPECTED_STD_BATCH_SIZE:.6f}, "
        f"Observed={OBSERVED_STD_BATCH_SIZE:.6f}"
    )

print(
    f"✓ Poisson batch-size variability : PASS "
    f"(expected SD={EXPECTED_STD_BATCH_SIZE:.3f}, "
    f"observed SD={OBSERVED_STD_BATCH_SIZE:.3f})"
)


# --------------------------------------------------------------------------------------------------
# 5.3 Batch-size range validation
# --------------------------------------------------------------------------------------------------

MIN_BATCH_SIZE = int(
    DIAGNOSTIC_RESULT[
        "min_batch_size"
    ]
)

MAX_BATCH_SIZE = int(
    DIAGNOSTIC_RESULT[
        "max_batch_size"
    ]
)

BATCH_RANGE_VALID = bool(
    0
    <=
    MIN_BATCH_SIZE
    <=
    MAX_BATCH_SIZE
    <=
    DIAGNOSTIC_N
)

if not BATCH_RANGE_VALID:
    raise RuntimeError(
        "Observed Poisson batch-size range is invalid."
    )

print(
    f"✓ Batch-size bounds [0,N] : PASS "
    f"(min={MIN_BATCH_SIZE}, max={MAX_BATCH_SIZE})"
)


# --------------------------------------------------------------------------------------------------
# 5.4 Independent-sampling validation
# --------------------------------------------------------------------------------------------------

GENERATOR_A = torch.Generator(
    device="cpu"
)

GENERATOR_B = torch.Generator(
    device="cpu"
)

GENERATOR_A.manual_seed(
    int(MASTER_SEED) + 101
)

GENERATOR_B.manual_seed(
    int(MASTER_SEED) + 202
)

SAMPLE_A = poisson_sample_indices(
    n_samples=DIAGNOSTIC_N,
    sample_rate=DIAGNOSTIC_Q,
    generator=GENERATOR_A,
)

SAMPLE_B = poisson_sample_indices(
    n_samples=DIAGNOSTIC_N,
    sample_rate=DIAGNOSTIC_Q,
    generator=GENERATOR_B,
)

INDEPENDENT_SAMPLING_VALID = bool(
    not torch.equal(
        SAMPLE_A,
        SAMPLE_B,
    )
)

if not INDEPENDENT_SAMPLING_VALID:
    raise RuntimeError(
        "Independent Poisson sampling draws were identical."
    )

print(
    "✓ Independent sampling draws : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5.5 Zero-inclusion possibility validation
# --------------------------------------------------------------------------------------------------

SMALL_N = 10
SMALL_Q = 0.05
ZERO_TEST_REPETITIONS = 5000

ZERO_TEST_GENERATOR = torch.Generator(
    device="cpu"
)

ZERO_TEST_GENERATOR.manual_seed(
    int(MASTER_SEED) + 303
)

zero_batch_observed = False

for _ in range(
    ZERO_TEST_REPETITIONS
):

    zero_indices = poisson_sample_indices(
        n_samples=SMALL_N,
        sample_rate=SMALL_Q,
        generator=ZERO_TEST_GENERATOR,
    )

    if zero_indices.numel() == 0:
        zero_batch_observed = True
        break

ZERO_INCLUSION_BEHAVIOR_VALID = bool(
    zero_batch_observed
)

if not ZERO_INCLUSION_BEHAVIOR_VALID:
    raise RuntimeError(
        "Diagnostic Poisson sampler did not demonstrate "
        "zero-inclusion behavior where it is probabilistically possible."
    )

print(
    "✓ Zero-inclusion behavior possible : PASS"
)


# --------------------------------------------------------------------------------------------------
# 5.6 Input validation
# --------------------------------------------------------------------------------------------------

INPUT_VALIDATION_PASS = True


try:

    poisson_sample_indices(
        n_samples=0,
        sample_rate=0.1,
    )

    INPUT_VALIDATION_PASS = False

except ValueError:
    pass


try:

    poisson_sample_indices(
        n_samples=100,
        sample_rate=0.0,
    )

    INPUT_VALIDATION_PASS = False

except ValueError:
    pass


try:

    poisson_sample_indices(
        n_samples=100,
        sample_rate=1.1,
    )

    INPUT_VALIDATION_PASS = False

except ValueError:
    pass


INPUT_VALIDATION_PASS = bool(
    INPUT_VALIDATION_PASS
)

if not INPUT_VALIDATION_PASS:
    raise RuntimeError(
        "Diagnostic Poisson sampler input validation failed."
    )

print(
    "✓ Sampler input validation : PASS"
)


# --------------------------------------------------------------------------------------------------
# 6. Dataset-specific sampling validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("DATASET-SPECIFIC SAMPLING VALIDATION")
print("-" * 100)


SECTION_10_RESULTS = []

DATASET_REPETITIONS = 500

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_id = str(
        dataset_id
    )

    n_train = int(
        TRAIN_ROWS[
            dataset_id
        ]
    )

    if n_train <= 0:
        raise RuntimeError(
            f"{dataset_id}: invalid training-row count."
        )

    sample_rate = float(
        min(
            1.0,
            int(DP_BATCH_SIZE)
            /
            n_train,
        )
    )

    dataset_seed = (
        int(MASTER_SEED)
        +
        10000
        +
        dataset_index
    )

    result = validate_poisson_sampling(
        n_samples=n_train,
        sample_rate=sample_rate,
        repetitions=DATASET_REPETITIONS,
        seed=dataset_seed,
    )

    expected_batch_size = float(
        result[
            "expected_batch_size"
        ]
    )

    observed_mean = float(
        result[
            "mean_batch_size"
        ]
    )

    expected_std = float(
        result[
            "expected_std_batch_size"
        ]
    )

    observed_std = float(
        result[
            "observed_std_batch_size"
        ]
    )

    if expected_std > 0:

        std_ratio = (
            observed_std
            /
            expected_std
        )

    else:

        std_ratio = 1.0

    dataset_mean_tolerance = float(
        max(
            1.0,
            5.0
            *
            expected_std
            /
            np.sqrt(
                DATASET_REPETITIONS
            ),
        )
    )

    dataset_mean_valid = bool(
        abs(
            observed_mean
            -
            expected_batch_size
        )
        <=
        dataset_mean_tolerance
    )

    dataset_std_valid = bool(
        0.90
        <=
        std_ratio
        <=
        1.10
    )

    dataset_range_valid = bool(
        0
        <=
        result["min_batch_size"]
        <=
        result["max_batch_size"]
        <=
        n_train
    )

    dataset_status = bool(
        all(
            [
                dataset_mean_valid,
                dataset_std_valid,
                dataset_range_valid,
            ]
        )
    )

    if not dataset_status:
        raise RuntimeError(
            f"{dataset_id}: dataset-specific Poisson "
            "sampling validation failed."
        )

    SECTION_10_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "n_train": int(
                n_train
            ),
            "nominal_batch_size": int(
                DP_BATCH_SIZE
            ),
            "poisson_sample_rate": float(
                sample_rate
            ),
            "expected_batch_size": float(
                expected_batch_size
            ),
            "observed_mean_batch_size": float(
                observed_mean
            ),
            "expected_batch_std": float(
                expected_std
            ),
            "observed_batch_std": float(
                observed_std
            ),
            "std_ratio": float(
                std_ratio
            ),
            "min_batch_size": int(
                result["min_batch_size"]
            ),
            "max_batch_size": int(
                result["max_batch_size"]
            ),
            "repetitions": int(
                DATASET_REPETITIONS
            ),
            "mean_validation": bool(
                dataset_mean_valid
            ),
            "std_validation": bool(
                dataset_std_valid
            ),
            "range_validation": bool(
                dataset_range_valid
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"N={n_train} | "
        f"q={sample_rate:.8f} | "
        f"E[B]={expected_batch_size:.3f} | "
        f"mean={observed_mean:.3f}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate production Opacus DPDataLoader
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRODUCTION OPACUS DPDataLoader VALIDATION")
print("-" * 100)


OPACUS_LOADER_RESULTS = []

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    dataset_id = str(
        dataset_id
    )

    n_train = int(
        TRAIN_ROWS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # 7.1 Resolve critic input dimension from the frozen architecture registry
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in CRITIC_INPUT_DIMENSIONS:
        raise RuntimeError(
            f"{dataset_id}: critic input dimension is missing "
            "from CRITIC_INPUT_DIMENSIONS."
        )

    input_dimension = int(
        CRITIC_INPUT_DIMENSIONS[
            dataset_id
        ]
    )

    if input_dimension <= 0:
        raise RuntimeError(
            f"{dataset_id}: invalid critic input dimension "
            f"{input_dimension}."
        )

    torch.manual_seed(
        int(MASTER_SEED)
        +
        11000
        +
        dataset_index
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            int(MASTER_SEED)
            +
            11000
            +
            dataset_index
        )

    # ----------------------------------------------------------------------------------------------
    # 7.2 RAM-safe validation dataset
    # ----------------------------------------------------------------------------------------------

    loader_validation_rows = min(
        256,
        max(
            32,
            int(DP_BATCH_SIZE) * 2,
        ),
    )

    validation_tensor = torch.randn(
        loader_validation_rows,
        input_dimension,
        dtype=torch.float32,
    )

    validation_dataset = (
        torch.utils.data.TensorDataset(
            validation_tensor
        )
    )

    validation_loader = (
        torch.utils.data.DataLoader(
            validation_dataset,
            batch_size=int(
                DP_BATCH_SIZE
            ),
            shuffle=True,
            drop_last=False,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 7.3 Fresh critic and optimizer
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        input_dim=input_dimension
    ).to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
        weight_decay=1e-6,
    )

    # ----------------------------------------------------------------------------------------------
    # 7.4 Notebook 10 reference sigma
    #
    # IMPORTANT:
    # This is mechanism validation only.
    #
    # Final dataset-specific noise calibration and achieved epsilon
    # are determined in Notebook 11.
    # ----------------------------------------------------------------------------------------------

    noise_multiplier = float(
        REFERENCE_SIGMA
    )

    if not np.isfinite(
        noise_multiplier
    ) or noise_multiplier <= 0.0:

        raise RuntimeError(
            f"{dataset_id}: invalid reference noise multiplier."
        )

    # ----------------------------------------------------------------------------------------------
    # 7.5 Attach Opacus
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=validation_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=float(
            MAX_GRAD_NORM
        ),
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
    )

    # ----------------------------------------------------------------------------------------------
    # 7.6 Validate private loader identity
    # ----------------------------------------------------------------------------------------------

    loader_class = (
        private_loader.__class__.__name__
    )

    loader_module = (
        private_loader.__class__.__module__
    )

    dp_loader_valid = bool(
        loader_class == "DPDataLoader"
        and
        loader_module.startswith(
            "opacus."
        )
    )

    if not dp_loader_valid:
        raise RuntimeError(
            f"{dataset_id}: expected Opacus DPDataLoader. "
            f"Class={loader_class}, "
            f"Module={loader_module}"
        )

    # ----------------------------------------------------------------------------------------------
    # 7.7 Inspect sampler
    # ----------------------------------------------------------------------------------------------

    sampler = getattr(
        private_loader,
        "sampler",
        None,
    )

    batch_sampler = getattr(
        private_loader,
        "batch_sampler",
        None,
    )

    sampler_class = (
        sampler.__class__.__name__
        if sampler is not None
        else ""
    )

    sampler_module = (
        sampler.__class__.__module__
        if sampler is not None
        else ""
    )

    batch_sampler_class = (
        batch_sampler.__class__.__name__
        if batch_sampler is not None
        else ""
    )

    batch_sampler_module = (
        batch_sampler.__class__.__module__
        if batch_sampler is not None
        else ""
    )

    # ----------------------------------------------------------------------------------------------
    # 7.8 Validate Opacus sampling infrastructure
    # ----------------------------------------------------------------------------------------------

    sampler_text = (
        f"{sampler_class} "
        f"{sampler_module} "
        f"{batch_sampler_class} "
        f"{batch_sampler_module}"
    ).lower()

    sampler_infrastructure_valid = bool(
        (
            "uniformwithreplacementsampler"
            in sampler_text
        )
        or
        (
            "dpdataloader"
            in loader_class.lower()
        )
    )

    if not sampler_infrastructure_valid:
        raise RuntimeError(
            f"{dataset_id}: Opacus sampling infrastructure "
            "could not be validated."
        )

    # ----------------------------------------------------------------------------------------------
    # 7.9 Validate loader length and batch generation
    # ----------------------------------------------------------------------------------------------

    loader_length = int(
        len(
            private_loader
        )
    )

    loader_length_valid = bool(
        loader_length > 0
    )

    if not loader_length_valid:
        raise RuntimeError(
            f"{dataset_id}: DPDataLoader has zero batches."
        )

    private_iterator = iter(
        private_loader
    )

    first_batch = next(
        private_iterator
    )

    if not isinstance(
        first_batch,
        (tuple, list),
    ):
        raise RuntimeError(
            f"{dataset_id}: DPDataLoader output is not tuple/list."
        )

    first_batch_tensor = first_batch[0]

    observed_private_batch_size = int(
        first_batch_tensor.shape[0]
    )

    observed_private_batch_dimension = int(
        first_batch_tensor.shape[1]
    )

    batch_dimension_valid = bool(
        observed_private_batch_dimension
        ==
        input_dimension
    )

    batch_size_valid = bool(
        observed_private_batch_size > 0
        and
        observed_private_batch_size
        <=
        loader_validation_rows
    )

    # ----------------------------------------------------------------------------------------------
    # 7.10 Validate private batch-size variability
    # ----------------------------------------------------------------------------------------------

    observed_batch_sizes = [
        observed_private_batch_size
    ]

    for _ in range(
        min(
            19,
            max(
                0,
                loader_length - 1,
            ),
        )
    ):

        try:

            next_batch = next(
                private_iterator
            )[0]

            observed_batch_sizes.append(
                int(
                    next_batch.shape[0]
                )
            )

        except StopIteration:

            break

    observed_batch_sizes = np.asarray(
        observed_batch_sizes,
        dtype=np.int64,
    )

    private_batch_sizes_valid = bool(
        (
            observed_batch_sizes > 0
        ).all()
        and
        (
            observed_batch_sizes
            <=
            loader_validation_rows
        ).all()
    )

    # ----------------------------------------------------------------------------------------------
    # 7.11 Verify private optimizer configuration
    # ----------------------------------------------------------------------------------------------

    optimizer_noise_multiplier = float(
        getattr(
            private_optimizer,
            "noise_multiplier",
        )
    )

    optimizer_max_grad_norm = float(
        getattr(
            private_optimizer,
            "max_grad_norm",
        )
    )

    optimizer_noise_match = bool(
        np.isclose(
            optimizer_noise_multiplier,
            noise_multiplier,
            rtol=0.0,
            atol=1e-12,
        )
    )

    optimizer_clip_match = bool(
        np.isclose(
            optimizer_max_grad_norm,
            float(MAX_GRAD_NORM),
            rtol=0.0,
            atol=1e-12,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 7.12 Verify Poisson sampling configuration explicitly
    # ----------------------------------------------------------------------------------------------

    poisson_sampling_attribute = getattr(
        private_loader,
        "poisson_sampling",
        None,
    )

    poisson_sampling_valid = bool(
        poisson_sampling_attribute is True
        or
        (
            "uniformwithreplacementsampler"
            in sampler_text
        )
    )

    if not poisson_sampling_valid:
        raise RuntimeError(
            f"{dataset_id}: Poisson sampling could not be "
            "verified from the Opacus loader."
        )

    # ----------------------------------------------------------------------------------------------
    # 7.13 Final production loader validation
    # ----------------------------------------------------------------------------------------------

    production_loader_status = bool(
        all(
            [
                dp_loader_valid,
                sampler_infrastructure_valid,
                poisson_sampling_valid,
                loader_length_valid,
                batch_dimension_valid,
                batch_size_valid,
                private_batch_sizes_valid,
                optimizer_noise_match,
                optimizer_clip_match,
            ]
        )
    )

    if not production_loader_status:
        raise RuntimeError(
            f"{dataset_id}: production Opacus DPDataLoader "
            "validation failed."
        )

    OPACUS_LOADER_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "train_rows_reference": int(
                n_train
            ),
            "validation_rows": int(
                loader_validation_rows
            ),
            "nominal_batch_size": int(
                DP_BATCH_SIZE
            ),
            "poisson_sample_rate": float(
                DP_BATCH_SIZE / n_train
            ),
            "reference_noise_multiplier": float(
                noise_multiplier
            ),
            "optimizer_noise_multiplier": float(
                optimizer_noise_multiplier
            ),
            "max_grad_norm": float(
                MAX_GRAD_NORM
            ),
            "optimizer_max_grad_norm": float(
                optimizer_max_grad_norm
            ),
            "loader_class": str(
                loader_class
            ),
            "loader_module": str(
                loader_module
            ),
            "sampler_class": str(
                sampler_class
            ),
            "sampler_module": str(
                sampler_module
            ),
            "batch_sampler_class": str(
                batch_sampler_class
            ),
            "batch_sampler_module": str(
                batch_sampler_module
            ),
            "dp_loader_valid": bool(
                dp_loader_valid
            ),
            "sampler_infrastructure_valid": bool(
                sampler_infrastructure_valid
            ),
            "poisson_sampling_valid": bool(
                poisson_sampling_valid
            ),
            "loader_length": int(
                loader_length
            ),
            "observed_first_batch_size": int(
                observed_private_batch_size
            ),
            "observed_batch_dimension": int(
                observed_private_batch_dimension
            ),
            "batch_dimension_valid": bool(
                batch_dimension_valid
            ),
            "batch_size_valid": bool(
                batch_size_valid
            ),
            "private_batch_sizes_valid": bool(
                private_batch_sizes_valid
            ),
            "optimizer_noise_match": bool(
                optimizer_noise_match
            ),
            "optimizer_clip_match": bool(
                optimizer_clip_match
            ),
            "status": "PASS",
        }
    )

    print(
        f"✓ {dataset_id:<20} : PASS | "
        f"DPDataLoader=YES | "
        f"q={DP_BATCH_SIZE / n_train:.8f} | "
        f"first_batch={observed_private_batch_size}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Build Section 10 validation DataFrames
# --------------------------------------------------------------------------------------------------

SECTION_10_DIAGNOSTIC_VALIDATION_DF = pd.DataFrame(
    [
        {
            "validation": "expected_batch_size",
            "status": (
                "PASS"
                if MEAN_VALID
                else "FAIL"
            ),
            "expected": float(
                EXPECTED_BATCH_SIZE
            ),
            "observed": float(
                OBSERVED_MEAN_BATCH_SIZE
            ),
        },
        {
            "validation": "batch_size_variability",
            "status": (
                "PASS"
                if STD_VALID
                else "FAIL"
            ),
            "expected": float(
                EXPECTED_STD_BATCH_SIZE
            ),
            "observed": float(
                OBSERVED_STD_BATCH_SIZE
            ),
        },
        {
            "validation": "batch_size_bounds",
            "status": (
                "PASS"
                if BATCH_RANGE_VALID
                else "FAIL"
            ),
            "expected": "0 <= B <= N",
            "observed": (
                f"{MIN_BATCH_SIZE} <= B <= "
                f"{MAX_BATCH_SIZE}"
            ),
        },
        {
            "validation": "independent_sampling",
            "status": (
                "PASS"
                if INDEPENDENT_SAMPLING_VALID
                else "FAIL"
            ),
            "expected": True,
            "observed": bool(
                INDEPENDENT_SAMPLING_VALID
            ),
        },
        {
            "validation": "zero_inclusion_behavior",
            "status": (
                "PASS"
                if ZERO_INCLUSION_BEHAVIOR_VALID
                else "FAIL"
            ),
            "expected": True,
            "observed": bool(
                ZERO_INCLUSION_BEHAVIOR_VALID
            ),
        },
        {
            "validation": "input_validation",
            "status": (
                "PASS"
                if INPUT_VALIDATION_PASS
                else "FAIL"
            ),
            "expected": True,
            "observed": bool(
                INPUT_VALIDATION_PASS
            ),
        },
    ]
)

SECTION_10_DATASET_VALIDATION_DF = pd.DataFrame(
    SECTION_10_RESULTS
)

SECTION_10_OPACUS_VALIDATION_DF = pd.DataFrame(
    OPACUS_LOADER_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 9. Prepare canonical directories
# --------------------------------------------------------------------------------------------------

NB10_ROOT = (
    Path(
        "/content/drive/MyDrive/SPP_GAN_Research"
    )
    /
    "results"
    /
    "notebooks"
    /
    "notebook_10"
)

VALIDATION_ROOT = (
    NB10_ROOT
    /
    "validation"
)

METADATA_ROOT = (
    NB10_ROOT
    /
    "metadata"
)

VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 10. Persist validation artifacts
# --------------------------------------------------------------------------------------------------

SECTION_10_DIAGNOSTIC_PATH = (
    VALIDATION_ROOT
    /
    "poisson_sampling_diagnostic_validation.csv"
)

SECTION_10_DATASET_PATH = (
    VALIDATION_ROOT
    /
    "poisson_sampling_dataset_validation.csv"
)

SECTION_10_OPACUS_PATH = (
    VALIDATION_ROOT
    /
    "opacus_dp_dataloader_validation.csv"
)

SECTION_10_DIAGNOSTIC_VALIDATION_DF.to_csv(
    SECTION_10_DIAGNOSTIC_PATH,
    index=False,
)

SECTION_10_DATASET_VALIDATION_DF.to_csv(
    SECTION_10_DATASET_PATH,
    index=False,
)

SECTION_10_OPACUS_VALIDATION_DF.to_csv(
    SECTION_10_OPACUS_PATH,
    index=False,
)

print(
    "\n✓ Sampling validation artifacts persisted."
)

print(
    f"  Diagnostic : {SECTION_10_DIAGNOSTIC_PATH}"
)

print(
    f"  Dataset    : {SECTION_10_DATASET_PATH}"
)

print(
    f"  Opacus     : {SECTION_10_OPACUS_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 11. Build Section 10 manifest
# --------------------------------------------------------------------------------------------------

SECTION_10_MANIFEST = {
    "notebook": "10",
    "section": "10",
    "title": "Define Sampling Mechanism",
    "status": "PASS",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "sampling": {
        "production_mechanism": (
            "Opacus DPDataLoader"
        ),
        "privacy_sampling_model": (
            "Independent Bernoulli inclusion / "
            "Poisson subsampling"
        ),
        "diagnostic_sampler": (
            "poisson_sample_indices"
        ),
        "diagnostic_sampler_is_production": False,
        "nominal_batch_size": int(
            DP_BATCH_SIZE
        ),
        "reference_noise_multiplier": float(
            REFERENCE_SIGMA
        ),
        "final_noise_calibration_notebook": (
            "Notebook 11"
        ),
    },

    "mathematical_validation": {
        "distribution_of_realized_batch_size": (
            "Binomial(N,q)"
        ),
        "expected_batch_size": (
            "N*q"
        ),
        "variance": (
            "N*q*(1-q)"
        ),
        "expected_batch_size_validation": bool(
            MEAN_VALID
        ),
        "batch_size_variability_validation": bool(
            STD_VALID
        ),
        "batch_size_bounds_validation": bool(
            BATCH_RANGE_VALID
        ),
        "independent_sampling_validation": bool(
            INDEPENDENT_SAMPLING_VALID
        ),
        "zero_inclusion_validation": bool(
            ZERO_INCLUSION_BEHAVIOR_VALID
        ),
        "input_validation": bool(
            INPUT_VALIDATION_PASS
        ),
    },

    "dataset_validation": {
        "dataset_count": int(
            len(
                SECTION_10_RESULTS
            )
        ),
        "all_datasets_passed": bool(
            all(
                row["status"] == "PASS"
                for row
                in SECTION_10_RESULTS
            )
        ),
    },

    "opacus_validation": {
        "dataset_count": int(
            len(
                OPACUS_LOADER_RESULTS
            )
        ),
        "all_datasets_passed": bool(
            all(
                row["status"] == "PASS"
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "dp_dataloader_verified": bool(
            all(
                row["dp_loader_valid"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "sampling_infrastructure_verified": bool(
            all(
                row["sampler_infrastructure_valid"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "poisson_sampling_verified": bool(
            all(
                row["poisson_sampling_valid"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "optimizer_noise_verified": bool(
            all(
                row["optimizer_noise_match"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
        "optimizer_clipping_verified": bool(
            all(
                row["optimizer_clip_match"]
                for row
                in OPACUS_LOADER_RESULTS
            )
        ),
    },

    "privacy_boundary": {
        "diagnostic_sampler": (
            "mathematical validation only"
        ),
        "production_sampling": (
            "Opacus DPDataLoader"
        ),
        "dp_sgd_update": (
            "Notebook 10 Section 9"
        ),
        "reference_noise_multiplier": (
            "Notebook 10"
        ),
        "final_noise_calibration": (
            "Notebook 11"
        ),
        "privacy_accounting": (
            "Notebook 11"
        ),
        "achieved_epsilon": (
            "Notebook 11"
        ),
        "end_to_end_privacy_claim": False,
    },

    "artifacts": {
        "diagnostic_validation": str(
            SECTION_10_DIAGNOSTIC_PATH
        ),
        "dataset_validation": str(
            SECTION_10_DATASET_PATH
        ),
        "opacus_validation": str(
            SECTION_10_OPACUS_PATH
        ),
    },
}


# --------------------------------------------------------------------------------------------------
# 12. Validate manifest serializability
# --------------------------------------------------------------------------------------------------

try:

    json.dumps(
        SECTION_10_MANIFEST,
        indent=2,
        sort_keys=True,
    )

except (
    TypeError,
    ValueError,
) as exc:

    raise RuntimeError(
        "Section 10 manifest failed JSON serializability validation."
    ) from exc

print(
    "✓ Section 10 manifest JSON serializability : PASS"
)


# --------------------------------------------------------------------------------------------------
# 13. Persist manifest
# --------------------------------------------------------------------------------------------------

SECTION_10_MANIFEST_PATH = (
    METADATA_ROOT
    /
    "section_10_sampling_mechanism_manifest.json"
)

with open(
    SECTION_10_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SECTION_10_MANIFEST,
        f,
        indent=2,
        sort_keys=True,
    )

print(
    f"✓ Section 10 manifest persisted: "
    f"{SECTION_10_MANIFEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 14. Final verification
# --------------------------------------------------------------------------------------------------

DIAGNOSTIC_ALL_PASS = bool(
    (
        SECTION_10_DIAGNOSTIC_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

DATASET_ALL_PASS = bool(
    (
        SECTION_10_DATASET_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

OPACUS_ALL_PASS = bool(
    (
        SECTION_10_OPACUS_VALIDATION_DF[
            "status"
        ]
        ==
        "PASS"
    ).all()
)

ARTIFACTS_EXIST = bool(
    SECTION_10_DIAGNOSTIC_PATH.exists()
    and
    SECTION_10_DATASET_PATH.exists()
    and
    SECTION_10_OPACUS_PATH.exists()
    and
    SECTION_10_MANIFEST_PATH.exists()
)

SECTION_10_PASS = bool(
    DIAGNOSTIC_ALL_PASS
    and
    DATASET_ALL_PASS
    and
    OPACUS_ALL_PASS
    and
    ARTIFACTS_EXIST
)

if not SECTION_10_PASS:

    raise RuntimeError(
        "SECTION 10 FINAL VERIFICATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 15. Final output
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 10 FINAL VERIFICATION")
print("=" * 100)

print(
    "✓ Diagnostic Bernoulli/Poisson sampler          : PASS"
)

print(
    "✓ Expected batch-size validation                : PASS"
)

print(
    "✓ Batch-size variability validation             : PASS"
)

print(
    "✓ Batch-size bounds validation                  : PASS"
)

print(
    "✓ Independent sampling validation               : PASS"
)

print(
    "✓ Zero-inclusion behavior validation            : PASS"
)

print(
    "✓ Sampler input validation                      : PASS"
)

print(
    "✓ Dataset-specific sampling validation          : PASS"
)

print(
    "✓ Opacus DPDataLoader                           : PASS"
)

print(
    "✓ Opacus sampling infrastructure                : PASS"
)

print(
    "✓ Opacus Poisson sampling configuration         : PASS"
)

print(
    "✓ Opacus optimizer noise configuration          : PASS"
)

print(
    "✓ Opacus clipping configuration                 : PASS"
)

print(
    "✓ Validation artifacts persisted                : PASS"
)

print(
    "✓ Section 10 manifest JSON serializability      : PASS"
)

print(
    "✓ Section 10 manifest persisted                 : PASS"
)


# --------------------------------------------------------------------------------------------------
# 16. Privacy boundary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print(
    "Diagnostic sampler   : mathematical validation only"
)

print(
    "Production sampling  : Opacus DPDataLoader"
)

print(
    "Sampling model       : independent Bernoulli inclusion"
)

print(
    "Realized batch model : Binomial(N,q)"
)

print(
    "DP-SGD update        : Notebook 10 Section 9"
)

print(
    "Reference sigma      : Notebook 10"
)

print(
    "Final noise          : Notebook 11"
)

print(
    "Privacy accounting   : Notebook 11"
)

print(
    "Achieved epsilon     : Notebook 11"
)

print(
    "End-to-end DP claim  : DISABLED"
)

print("\n" + "=" * 100)
print("SECTION 10 STATUS: PASS")
print("=" * 100)


10. DEFINE SAMPLING MECHANISM
✓ Sampling configuration validated.
✓ Configured sampling mechanism : Poisson
✓ Nominal DP batch size        : 128
✓ Reference noise multiplier   : 1.000000
✓ Maximum gradient norm        : 1.000000
✓ Critic input dimensions      : validated
✓ Diagnostic Bernoulli-inclusion sampler defined.
✓ Diagnostic sampler is mathematical validation only.
✓ Diagnostic Poisson validator defined.

----------------------------------------------------------------------------------------------------
POISSON SAMPLING MATHEMATICAL VALIDATION
----------------------------------------------------------------------------------------------------
✓ Expected batch size E[B]=Nq : PASS (expected=1000.000, observed=1001.046)
✓ Poisson batch-size variability : PASS (expected SD=30.000, observed SD=28.751)
✓ Batch-size bounds [0,N] : PASS (min=908, max=1082)
✓ Independent sampling draws : PASS
✓ Zero-inclusion behavior possible : PASS
✓ Sampler input validation : PASS

----------------

/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [66]:
# ==================================================================================================
# 11. TEST GRADIENT PRIVATIZATION
# ==================================================================================================

print("\n" + "=" * 100)
print("11. TEST GRADIENT PRIVATIZATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_11_OBJECTS = [
    "ARCHITECTURE_SUMMARY_DF",
    "MASTER_SEED",
    "DEVICE",
    "SPPGANCritic",
    "wrap_critic_for_per_sample_gradients",
    "dp_discriminator_loss",
    "get_per_example_gradients",
    "MAX_GRAD_NORM",
    "DIRS",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_11_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 11 is missing required objects: "
        f"{missing_objects}"
    )


if int(MAX_GRAD_NORM) <= 0:
    raise ValueError(
        "MAX_GRAD_NORM must be strictly positive."
    )


# --------------------------------------------------------------------------------------------------
# 2. Define publication-consistent global per-example clipping
# --------------------------------------------------------------------------------------------------

def clip_per_example_gradients_global(
    gradients,
    max_grad_norm,
):
    """
    Clip per-example gradients using a global L2 norm across
    all discriminator parameters.

    Expected input:

        gradients = {
            parameter_name: tensor[B, ...],
            ...
        }

    where B is the number of examples.

    For each example i:

        ||g_i||_2 =
            sqrt(
                sum_p ||g_{i,p}||_2^2
            )

    Clipping factor:

        alpha_i =
            min(1, C / (||g_i||_2 + eps))

    Clipped gradient:

        g'_{i,p} = alpha_i * g_{i,p}

    This is the flat/global L2 clipping mechanism used by
    the DP-SGD configuration.
    """

    if not isinstance(
        gradients,
        dict,
    ):
        raise TypeError(
            "Expected per-example gradients as a dictionary."
        )

    if len(gradients) == 0:
        raise ValueError(
            "Per-example gradient dictionary is empty."
        )

    max_grad_norm = float(
        max_grad_norm
    )

    if not np.isfinite(
        max_grad_norm
    ) or max_grad_norm <= 0.0:
        raise ValueError(
            "max_grad_norm must be finite and strictly positive."
        )

    gradient_items = []

    batch_size = None

    for parameter_name, gradient in gradients.items():

        if not torch.is_tensor(
            gradient
        ):
            raise TypeError(
                "Gradient for parameter "
                f"'{parameter_name}' is not a tensor."
            )

        if gradient.ndim < 1:
            raise ValueError(
                "Per-example gradient tensor for "
                f"'{parameter_name}' has invalid shape "
                f"{tuple(gradient.shape)}."
            )

        if batch_size is None:

            batch_size = int(
                gradient.shape[0]
            )

        elif int(
            gradient.shape[0]
        ) != batch_size:

            raise RuntimeError(
                "Per-example gradient batch dimensions "
                "are inconsistent."
            )

        gradient_items.append(
            (
                parameter_name,
                gradient,
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Compute global per-example squared L2 norm
    # ----------------------------------------------------------------------------------------------

    squared_norms = torch.zeros(
        batch_size,
        dtype=torch.float32,
        device=gradient_items[0][1].device,
    )

    for _, gradient in gradient_items:

        flattened = gradient.reshape(
            batch_size,
            -1,
        )

        squared_norms = (
            squared_norms
            +
            flattened.float().pow(2).sum(
                dim=1
            )
        )

    norms = torch.sqrt(
        squared_norms
    )

    # ----------------------------------------------------------------------------------------------
    # Compute clipping factors
    # ----------------------------------------------------------------------------------------------

    epsilon = torch.finfo(
        norms.dtype
    ).eps

    clipping_factors = torch.clamp(
        max_grad_norm
        /
        (
            norms
            +
            epsilon
        ),
        max=1.0,
    )

    # ----------------------------------------------------------------------------------------------
    # Apply the same per-example factor to every parameter
    # ----------------------------------------------------------------------------------------------

    clipped_gradients = {}

    for parameter_name, gradient in gradient_items:

        reshape_dimensions = (
            [batch_size]
            +
            [1] * (
                gradient.ndim - 1
            )
        )

        clipped_gradients[
            parameter_name
        ] = (
            gradient
            *
            clipping_factors.reshape(
                reshape_dimensions
            )
        )

    return (
        clipped_gradients,
        norms,
        clipping_factors,
    )


print(
    "✓ Global per-example gradient clipping validator defined."
)


# --------------------------------------------------------------------------------------------------
# 3. Run gradient privatization tests
# --------------------------------------------------------------------------------------------------

GRADIENT_TEST_ROWS = []

TEST_BATCH_SIZE = 8

for dataset_index, row in enumerate(
    ARCHITECTURE_SUMMARY_DF.itertuples()
):

    dataset_id = str(
        row.dataset
    )

    transformed_dimension = int(
        row.transformed_dimension
    )

    if transformed_dimension <= 0:
        raise RuntimeError(
            f"{dataset_id}: invalid transformed dimension "
            f"{transformed_dimension}."
        )

    # ----------------------------------------------------------------------------------------------
    # 3.1 Deterministic test seed
    # ----------------------------------------------------------------------------------------------

    test_seed = (
        int(MASTER_SEED)
        +
        12000
        +
        dataset_index
    )

    torch.manual_seed(
        test_seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            test_seed
        )

    # ----------------------------------------------------------------------------------------------
    # 3.2 Fresh frozen SPP-GAN critic
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        input_dim=transformed_dimension
    ).to(
        DEVICE
    )

    dp_critic = (
        wrap_critic_for_per_sample_gradients(
            critic
        )
    )

    dp_critic.train()

    # ----------------------------------------------------------------------------------------------
    # 3.3 Diagnostic real/fake batches
    # ----------------------------------------------------------------------------------------------

    real_batch = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
        dtype=torch.float32,
    )

    fake_batch = torch.randn(
        TEST_BATCH_SIZE,
        transformed_dimension,
        device=DEVICE,
        dtype=torch.float32,
    )

    dp_critic.zero_grad(
        set_to_none=True
    )

    # ----------------------------------------------------------------------------------------------
    # 3.4 Compute discriminator loss
    # ----------------------------------------------------------------------------------------------

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic=dp_critic,
            real_batch=real_batch,
            fake_batch=fake_batch,
        )
    )

    if not torch.is_tensor(
        loss
    ):
        raise RuntimeError(
            f"{dataset_id}: discriminator loss is not a tensor."
        )

    if per_example_loss.ndim != 1:
        raise RuntimeError(
            f"{dataset_id}: per-example loss must be 1-dimensional. "
            f"Observed shape={tuple(per_example_loss.shape)}"
        )

    if int(
        per_example_loss.shape[0]
    ) != TEST_BATCH_SIZE:

        raise RuntimeError(
            f"{dataset_id}: invalid per-example loss size. "
            f"Expected={TEST_BATCH_SIZE}, "
            f"Observed={per_example_loss.shape[0]}"
        )

    # ----------------------------------------------------------------------------------------------
    # 3.5 Backpropagate
    # ----------------------------------------------------------------------------------------------

    loss.backward()

    # ----------------------------------------------------------------------------------------------
    # 3.6 Retrieve per-example gradients
    # ----------------------------------------------------------------------------------------------

    gradients = (
        get_per_example_gradients(
            dp_critic
        )
    )

    if not isinstance(
        gradients,
        dict
    ):
        raise TypeError(
            f"{dataset_id}: get_per_example_gradients() "
            "must return a dictionary."
        )

    if len(gradients) == 0:
        raise RuntimeError(
            f"{dataset_id}: no per-example gradients were returned."
        )

    # ----------------------------------------------------------------------------------------------
    # 3.7 Validate gradient shapes
    # ----------------------------------------------------------------------------------------------

    gradient_batch_sizes = [
        int(
            gradient.shape[0]
        )
        for gradient
        in gradients.values()
    ]

    shape_pass = bool(
        all(
            batch_size == TEST_BATCH_SIZE
            for batch_size
            in gradient_batch_sizes
        )
    )

    if not shape_pass:
        raise RuntimeError(
            f"{dataset_id}: inconsistent per-example gradient "
            "batch dimensions."
        )

    # ----------------------------------------------------------------------------------------------
    # 3.8 Validate finite raw gradients
    # ----------------------------------------------------------------------------------------------

    finite_pass = bool(
        all(
            torch.isfinite(
                gradient
            ).all().item()
            for gradient
            in gradients.values()
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 3.9 Compute global raw per-example gradient norms
    # ----------------------------------------------------------------------------------------------

    raw_squared_norms = torch.zeros(
        TEST_BATCH_SIZE,
        dtype=torch.float32,
        device=DEVICE,
    )

    for gradient in gradients.values():

        flattened = gradient.reshape(
            TEST_BATCH_SIZE,
            -1,
        )

        raw_squared_norms = (
            raw_squared_norms
            +
            flattened.float().pow(2).sum(
                dim=1
            )
        )

    norms = torch.sqrt(
        raw_squared_norms
    )

    # ----------------------------------------------------------------------------------------------
    # 3.10 Apply global flat L2 clipping
    # ----------------------------------------------------------------------------------------------

    (
        clipped_gradients,
        returned_norms,
        clipping_factors,
    ) = clip_per_example_gradients_global(
        gradients=gradients,
        max_grad_norm=MAX_GRAD_NORM,
    )

    # ----------------------------------------------------------------------------------------------
    # 3.11 Validate norm computation consistency
    # ----------------------------------------------------------------------------------------------

    norm_consistency_pass = bool(
        torch.allclose(
            norms,
            returned_norms,
            rtol=1e-5,
            atol=1e-6,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 3.12 Compute clipped global norms
    # ----------------------------------------------------------------------------------------------

    clipped_squared_norms = torch.zeros(
        TEST_BATCH_SIZE,
        dtype=torch.float32,
        device=DEVICE,
    )

    for gradient in clipped_gradients.values():

        flattened = gradient.reshape(
            TEST_BATCH_SIZE,
            -1,
        )

        clipped_squared_norms = (
            clipped_squared_norms
            +
            flattened.float().pow(2).sum(
                dim=1
            )
        )

    clipped_norms = torch.sqrt(
        clipped_squared_norms
    )

    # ----------------------------------------------------------------------------------------------
    # 3.13 Validate clipping
    # ----------------------------------------------------------------------------------------------

    clipping_pass = bool(
        torch.isfinite(
            clipped_norms
        ).all().item()
        and
        float(
            clipped_norms.max()
            .detach()
            .cpu()
        )
        <= (
            float(MAX_GRAD_NORM)
            +
            1e-5
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 3.14 Validate clipping factors
    # ----------------------------------------------------------------------------------------------

    clipping_factor_pass = bool(
        torch.isfinite(
            clipping_factors
        ).all().item()
        and
        float(
            clipping_factors.min()
            .detach()
            .cpu()
        )
        >= 0.0
        and
        float(
            clipping_factors.max()
            .detach()
            .cpu()
        )
        <= 1.0
        +
        1e-6
    )

    # ----------------------------------------------------------------------------------------------
    # 3.15 Validate nonzero gradients
    # ----------------------------------------------------------------------------------------------

    nonzero_pass = bool(
        float(
            norms.max()
            .detach()
            .cpu()
        )
        > 0.0
    )

    # ----------------------------------------------------------------------------------------------
    # 3.16 Validate clipped gradients are finite
    # ----------------------------------------------------------------------------------------------

    clipped_finite_pass = bool(
        all(
            torch.isfinite(
                gradient
            ).all().item()
            for gradient
            in clipped_gradients.values()
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 3.17 Validate actual clipping behavior
    # ----------------------------------------------------------------------------------------------

    raw_max_norm = float(
        norms.max()
        .detach()
        .cpu()
    )

    clipped_max_norm = float(
        clipped_norms.max()
        .detach()
        .cpu()
    )

    if raw_max_norm > (
        float(MAX_GRAD_NORM)
        +
        1e-6
    ):

        actual_clipping_pass = bool(
            clipped_max_norm
            <=
            float(MAX_GRAD_NORM)
            +
            1e-5
        )

    else:

        actual_clipping_pass = bool(
            clipped_max_norm
            <=
            float(MAX_GRAD_NORM)
            +
            1e-5
        )

    # ----------------------------------------------------------------------------------------------
    # 3.18 Final row status
    # ----------------------------------------------------------------------------------------------

    status = (
        "PASS"
        if all(
            [
                shape_pass,
                finite_pass,
                norm_consistency_pass,
                clipping_pass,
                clipping_factor_pass,
                nonzero_pass,
                clipped_finite_pass,
                actual_clipping_pass,
            ]
        )
        else "FAIL"
    )

    GRADIENT_TEST_ROWS.append(
        {
            "dataset": dataset_id,
            "transformed_dimension": transformed_dimension,
            "batch_size": TEST_BATCH_SIZE,
            "gradient_parameter_count": len(
                gradients
            ),
            "max_raw_gradient_norm": raw_max_norm,
            "max_clipped_gradient_norm": clipped_max_norm,
            "max_clipping_factor": float(
                clipping_factors.max()
                .detach()
                .cpu()
            ),
            "min_clipping_factor": float(
                clipping_factors.min()
                .detach()
                .cpu()
            ),
            "shape_validation": (
                "PASS"
                if shape_pass
                else "FAIL"
            ),
            "finite_validation": (
                "PASS"
                if finite_pass
                else "FAIL"
            ),
            "norm_consistency_validation": (
                "PASS"
                if norm_consistency_pass
                else "FAIL"
            ),
            "clipping_validation": (
                "PASS"
                if clipping_pass
                else "FAIL"
            ),
            "clipping_factor_validation": (
                "PASS"
                if clipping_factor_pass
                else "FAIL"
            ),
            "nonzero_validation": (
                "PASS"
                if nonzero_pass
                else "FAIL"
            ),
            "clipped_finite_validation": (
                "PASS"
                if clipped_finite_pass
                else "FAIL"
            ),
            "actual_clipping_validation": (
                "PASS"
                if actual_clipping_pass
                else "FAIL"
            ),
            "status": status,
        }
    )

    # ----------------------------------------------------------------------------------------------
    # 3.19 Release diagnostic objects
    # ----------------------------------------------------------------------------------------------

    del (
        dp_critic,
        critic,
        real_batch,
        fake_batch,
        gradients,
        clipped_gradients,
        norms,
        returned_norms,
        clipping_factors,
        clipped_norms,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# --------------------------------------------------------------------------------------------------
# 4. Build validation DataFrame
# --------------------------------------------------------------------------------------------------

GRADIENT_TEST_DF = pd.DataFrame(
    GRADIENT_TEST_ROWS
)


# --------------------------------------------------------------------------------------------------
# 5. Validate complete test result
# --------------------------------------------------------------------------------------------------

if GRADIENT_TEST_DF.empty:
    raise RuntimeError(
        "Gradient privatization validation produced no results."
    )

if not (
    GRADIENT_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Gradient privatization validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 6. Persist validation artifact
# --------------------------------------------------------------------------------------------------

GRADIENT_TEST_PATH = (
    DIRS["validation"]
    /
    "dp_gradient_privatization_tests.csv"
)

GRADIENT_TEST_DF.to_csv(
    GRADIENT_TEST_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 7. Display validation results
# --------------------------------------------------------------------------------------------------

display(
    GRADIENT_TEST_DF
)

print(
    f"✓ Gradient privatization tests saved:\n"
    f"  {GRADIENT_TEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 8. Final verification
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 11 FINAL VERIFICATION")
print("-" * 100)

print(
    "✓ Per-example gradient generation       : PASS"
)

print(
    "✓ Gradient shape validation              : PASS"
)

print(
    "✓ Gradient finite-value validation      : PASS"
)

print(
    "✓ Global L2 norm validation              : PASS"
)

print(
    "✓ Global flat clipping validation       : PASS"
)

print(
    "✓ Clipping-factor validation             : PASS"
)

print(
    "✓ Clipped-gradient finite validation    : PASS"
)

print(
    "✓ Nonzero gradient validation             : PASS"
)

print(
    "✓ Validation artifact persisted           : PASS"
)

print(
    "\nSECTION 11 STATUS: PASS"
)


11. TEST GRADIENT PRIVATIZATION
✓ Global per-example gradient clipping validator defined.


<sys>:0: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


,dataset,transformed_dimension,batch_size,gradient_parameter_count,max_raw_gradient_norm,max_clipped_gradient_norm,max_clipping_factor,min_clipping_factor,shape_validation,finite_validation,norm_consistency_validation,clipping_validation,clipping_factor_validation,nonzero_validation,clipped_finite_validation,actual_clipping_validation,status
0,adult_income,105,8,6,6.309125,1.0,0.200568,0.158501,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
1,bank_marketing,51,8,6,6.258184,1.0,0.236558,0.159791,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
2,diabetes_130us,2329,8,6,13.205875,1.0,0.085346,0.075724,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS


✓ Gradient privatization tests saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_gradient_privatization_tests.csv

----------------------------------------------------------------------------------------------------
SECTION 11 FINAL VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Per-example gradient generation       : PASS
✓ Gradient shape validation              : PASS
✓ Gradient finite-value validation      : PASS
✓ Global L2 norm validation              : PASS
✓ Global flat clipping validation       : PASS
✓ Clipping-factor validation             : PASS
✓ Clipped-gradient finite validation    : PASS
✓ Nonzero gradient validation             : PASS
✓ Validation artifact persisted           : PASS

SECTION 11 STATUS: PASS


In [67]:
# ==================================================================================================
# 12. TEST NOISE INJECTION
# ==================================================================================================

print("\n" + "=" * 100)
print("12. TEST NOISE INJECTION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_12_OBJECTS = [
    "PRIVACY_PARAMETER_DF",
    "REFERENCE_SIGMA",
    "MAX_GRAD_NORM",
    "MASTER_SEED",
    "DEVICE",
    "gaussian_noise",
    "DIRS",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_12_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 12 is missing required objects: "
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate privacy parameter table
# --------------------------------------------------------------------------------------------------

if not isinstance(
    PRIVACY_PARAMETER_DF,
    pd.DataFrame,
):
    raise TypeError(
        "PRIVACY_PARAMETER_DF must be a pandas DataFrame."
    )

if PRIVACY_PARAMETER_DF.empty:
    raise RuntimeError(
        "PRIVACY_PARAMETER_DF is empty."
    )

if "dataset" not in PRIVACY_PARAMETER_DF.columns:
    raise RuntimeError(
        "PRIVACY_PARAMETER_DF must contain a 'dataset' column."
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate reference Gaussian noise parameters
# --------------------------------------------------------------------------------------------------

REFERENCE_SIGMA = float(
    REFERENCE_SIGMA
)

MAX_GRAD_NORM = float(
    MAX_GRAD_NORM
)

if not np.isfinite(
    REFERENCE_SIGMA
) or REFERENCE_SIGMA <= 0.0:
    raise ValueError(
        "REFERENCE_SIGMA must be finite and strictly positive."
    )

if not np.isfinite(
    MAX_GRAD_NORM
) or MAX_GRAD_NORM <= 0.0:
    raise ValueError(
        "MAX_GRAD_NORM must be finite and strictly positive."
    )

print(
    f"✓ Reference noise multiplier : "
    f"{REFERENCE_SIGMA:.6f}"
)

print(
    f"✓ Maximum gradient norm      : "
    f"{MAX_GRAD_NORM:.6f}"
)


# --------------------------------------------------------------------------------------------------
# 4. Deterministic validation seed
# --------------------------------------------------------------------------------------------------

torch.manual_seed(
    int(MASTER_SEED)
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        int(MASTER_SEED)
    )


# --------------------------------------------------------------------------------------------------
# 5. Test configuration
# --------------------------------------------------------------------------------------------------

NOISE_TEST_SIZE = 8192

NOISE_TEST_ROWS = []


# --------------------------------------------------------------------------------------------------
# 6. Gaussian noise validation
# --------------------------------------------------------------------------------------------------

for row in PRIVACY_PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        getattr(
            row,
            "dataset",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 6.1 Generate Gaussian noise
    # ----------------------------------------------------------------------------------------------

    noise = gaussian_noise(
        shape=NOISE_TEST_SIZE,
        noise_multiplier=REFERENCE_SIGMA,
        max_grad_norm=MAX_GRAD_NORM,
        device=DEVICE,
        dtype=torch.float32,
    )

    # ----------------------------------------------------------------------------------------------
    # 6.2 Expected standard deviation
    # ----------------------------------------------------------------------------------------------

    expected_std = (
        REFERENCE_SIGMA
        *
        MAX_GRAD_NORM
    )

    # ----------------------------------------------------------------------------------------------
    # 6.3 Observed standard deviation
    # ----------------------------------------------------------------------------------------------

    observed_std = float(
        noise.std(
            unbiased=True
        )
        .detach()
        .cpu()
    )

    # ----------------------------------------------------------------------------------------------
    # 6.4 Observed mean
    # ----------------------------------------------------------------------------------------------

    observed_mean = float(
        noise.mean()
        .detach()
        .cpu()
    )

    # ----------------------------------------------------------------------------------------------
    # 6.5 Finite-value validation
    # ----------------------------------------------------------------------------------------------

    finite_pass = bool(
        torch.isfinite(
            noise
        ).all().item()
    )

    # ----------------------------------------------------------------------------------------------
    # 6.6 Nonzero validation
    # ----------------------------------------------------------------------------------------------

    nonzero_fraction = float(
        (
            noise != 0
        )
        .float()
        .mean()
        .detach()
        .cpu()
    )

    nonzero_pass = bool(
        nonzero_fraction > 0.0
    )

    # ----------------------------------------------------------------------------------------------
    # 6.7 Standard-deviation validation
    # ----------------------------------------------------------------------------------------------

    relative_error = (
        abs(
            observed_std
            -
            expected_std
        )
        /
        expected_std
    )

    scale_pass = bool(
        relative_error < 0.15
    )

    # ----------------------------------------------------------------------------------------------
    # 6.8 Zero-mean validation
    # ----------------------------------------------------------------------------------------------

    expected_mean_standard_error = (
        expected_std
        /
        np.sqrt(
            NOISE_TEST_SIZE
        )
    )

    mean_tolerance = (
        5.0
        *
        expected_mean_standard_error
    )

    mean_pass = bool(
        abs(
            observed_mean
        )
        <=
        mean_tolerance
    )

    # ----------------------------------------------------------------------------------------------
    # 6.9 Final status
    # ----------------------------------------------------------------------------------------------

    status = (
        "PASS"
        if all(
            [
                finite_pass,
                nonzero_pass,
                scale_pass,
                mean_pass,
            ]
        )
        else "FAIL"
    )

    NOISE_TEST_ROWS.append(
        {
            "dataset": dataset_id,
            "noise_multiplier": float(
                REFERENCE_SIGMA
            ),
            "max_grad_norm": float(
                MAX_GRAD_NORM
            ),
            "expected_std": float(
                expected_std
            ),
            "observed_std": float(
                observed_std
            ),
            "relative_std_error": float(
                relative_error
            ),
            "observed_mean": float(
                observed_mean
            ),
            "mean_tolerance": float(
                mean_tolerance
            ),
            "nonzero_fraction": float(
                nonzero_fraction
            ),
            "finite": bool(
                finite_pass
            ),
            "nonzero": bool(
                nonzero_pass
            ),
            "mean_validation": bool(
                mean_pass
            ),
            "scale_validation": bool(
                scale_pass
            ),
            "status": status,
        }
    )

    print(
        f"✓ {dataset_id:<20} : "
        f"expected SD={expected_std:.6f} | "
        f"observed SD={observed_std:.6f} | "
        f"mean={observed_mean:.6f} | "
        f"status={status}"
    )

    del noise

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# --------------------------------------------------------------------------------------------------
# 7. Build validation DataFrame
# --------------------------------------------------------------------------------------------------

NOISE_TEST_DF = pd.DataFrame(
    NOISE_TEST_ROWS
)


# --------------------------------------------------------------------------------------------------
# 8. Validate complete test
# --------------------------------------------------------------------------------------------------

if NOISE_TEST_DF.empty:
    raise RuntimeError(
        "Gaussian noise validation produced no results."
    )

if not (
    NOISE_TEST_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Gaussian noise validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 9. Persist validation artifact
# --------------------------------------------------------------------------------------------------

NOISE_TEST_PATH = (
    DIRS["validation"]
    /
    "dp_gaussian_noise_tests.csv"
)

NOISE_TEST_DF.to_csv(
    NOISE_TEST_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 10. Display results
# --------------------------------------------------------------------------------------------------

display(
    NOISE_TEST_DF
)

print(
    f"✓ Noise tests saved:\n"
    f"  {NOISE_TEST_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 11. Final verification
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 12 FINAL VERIFICATION")
print("-" * 100)

print(
    "✓ Gaussian noise generation             : PASS"
)

print(
    "✓ Finite noise validation               : PASS"
)

print(
    "✓ Nonzero noise validation              : PASS"
)

print(
    "✓ Zero-mean validation                  : PASS"
)

print(
    "✓ Noise standard-deviation validation   : PASS"
)

print(
    "✓ Noise scale validation                : PASS"
)

print(
    "✓ Validation artifact persisted         : PASS"
)

print(
    "\nSECTION 12 STATUS: PASS"
)


12. TEST NOISE INJECTION
✓ Reference noise multiplier : 1.000000
✓ Maximum gradient norm      : 1.000000
✓ adult_income         : expected SD=1.000000 | observed SD=0.993355 | mean=0.003513 | status=PASS
✓ bank_marketing       : expected SD=1.000000 | observed SD=0.999142 | mean=-0.015460 | status=PASS
✓ diabetes_130us       : expected SD=1.000000 | observed SD=0.990080 | mean=0.012219 | status=PASS


,dataset,noise_multiplier,max_grad_norm,expected_std,observed_std,relative_std_error,observed_mean,mean_tolerance,nonzero_fraction,finite,nonzero,mean_validation,scale_validation,status
0,adult_income,1.0,1.0,1.0,0.993355,0.006645,0.003513,0.055243,1.0,True,True,True,True,PASS
1,bank_marketing,1.0,1.0,1.0,0.999142,0.000858,-0.015460,0.055243,1.0,True,True,True,True,PASS
2,diabetes_130us,1.0,1.0,1.0,0.990080,0.009920,0.012219,0.055243,1.0,True,True,True,True,PASS


✓ Noise tests saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_gaussian_noise_tests.csv

----------------------------------------------------------------------------------------------------
SECTION 12 FINAL VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Gaussian noise generation             : PASS
✓ Finite noise validation               : PASS
✓ Nonzero noise validation              : PASS
✓ Zero-mean validation                  : PASS
✓ Noise standard-deviation validation   : PASS
✓ Noise scale validation                : PASS
✓ Validation artifact persisted         : PASS

SECTION 12 STATUS: PASS


In [68]:
# ==================================================================================================
# 13. VALIDATE PRIVACY MECHANISM
# ==================================================================================================

print("\n" + "=" * 100)
print("13. VALIDATE PRIVACY MECHANISM")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate required dependencies
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_13_OBJECTS = [
    "ARCHITECTURE_SUMMARY_DF",
    "PRIVACY_PARAMETER_DF",
    "REFERENCE_SIGMA",
    "MAX_GRAD_NORM",
    "DP_BATCH_SIZE",
    "TRAIN_ROWS",
    "DEVICE",
    "SPPGANCritic",
    "PrivacyEngine",
    "ModuleValidator",
    "validate_poisson_sampling",
    "DIRS",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_13_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 13 is missing required objects: "
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate configuration
# --------------------------------------------------------------------------------------------------

REFERENCE_SIGMA = float(
    REFERENCE_SIGMA
)

MAX_GRAD_NORM = float(
    MAX_GRAD_NORM
)

DP_BATCH_SIZE = int(
    DP_BATCH_SIZE
)

if REFERENCE_SIGMA <= 0:
    raise ValueError(
        "REFERENCE_SIGMA must be > 0."
    )

if MAX_GRAD_NORM <= 0:
    raise ValueError(
        "MAX_GRAD_NORM must be > 0."
    )

if DP_BATCH_SIZE <= 0:
    raise ValueError(
        "DP_BATCH_SIZE must be > 0."
    )

if "dataset" not in PRIVACY_PARAMETER_DF.columns:
    raise RuntimeError(
        "PRIVACY_PARAMETER_DF must contain a 'dataset' column."
    )


print(
    f"✓ Reference noise multiplier : "
    f"{REFERENCE_SIGMA:.6f}"
)

print(
    f"✓ Maximum gradient norm      : "
    f"{MAX_GRAD_NORM:.6f}"
)

print(
    f"✓ DP batch size              : "
    f"{DP_BATCH_SIZE}"
)


# --------------------------------------------------------------------------------------------------
# 3. Validate required datasets
# --------------------------------------------------------------------------------------------------

architecture_datasets = set(
    ARCHITECTURE_SUMMARY_DF[
        "dataset"
    ].astype(str)
)

privacy_datasets = set(
    PRIVACY_PARAMETER_DF[
        "dataset"
    ].astype(str)
)

train_row_datasets = set(
    TRAIN_ROWS.keys()
)

if not (
    architecture_datasets
    ==
    privacy_datasets
    ==
    train_row_datasets
):
    raise RuntimeError(
        "Dataset registries are inconsistent across "
        "architecture, privacy, and training-row metadata."
    )


# --------------------------------------------------------------------------------------------------
# 4. Privacy mechanism validation
# --------------------------------------------------------------------------------------------------

PRIVACY_MECHANISM_ROWS = []


for row in ARCHITECTURE_SUMMARY_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    transformed_dimension = int(
        row.transformed_dimension
    )

    train_rows = int(
        TRAIN_ROWS[
            dataset_id
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # 4.1 Critic construction
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        transformed_dimension
    ).to(
        DEVICE
    )

    critic.train()

    # ----------------------------------------------------------------------------------------------
    # 4.2 Opacus module compatibility
    # ----------------------------------------------------------------------------------------------

    validation_errors = ModuleValidator.validate(
        critic,
        strict=False,
    )

    critic_compatible = bool(
        len(validation_errors) == 0
    )

    # ----------------------------------------------------------------------------------------------
    # 4.3 Dummy training data
    # ----------------------------------------------------------------------------------------------

    dummy_rows = 32

    dummy_x = torch.randn(
        dummy_rows,
        transformed_dimension,
        device=DEVICE,
        dtype=torch.float32,
    )

    dummy_dataset = torch.utils.data.TensorDataset(
        dummy_x
    )

    dummy_loader = torch.utils.data.DataLoader(
        dummy_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )

    optimizer = torch.optim.Adam(
        critic.parameters(),
        lr=2e-4,
    )

    # ----------------------------------------------------------------------------------------------
    # 4.4 PrivacyEngine compatibility
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    try:

        engine_compatible = bool(
            privacy_engine.is_compatible(
                module=critic,
                optimizer=optimizer,
                data_loader=dummy_loader,
            )
        )

    except Exception as exc:

        engine_compatible = False

        print(
            f"⚠ PrivacyEngine compatibility failed for "
            f"{dataset_id}: {exc}"
        )

    # ----------------------------------------------------------------------------------------------
    # 4.5 Poisson sampling diagnostic
    # ----------------------------------------------------------------------------------------------

    sample_rate = (
        DP_BATCH_SIZE
        /
        train_rows
    )

    poisson_test = validate_poisson_sampling(
        n_samples=train_rows,
        sample_rate=sample_rate,
        repetitions=50,
    )

    poisson_pass = bool(
        poisson_test[
            "mean_batch_size"
        ] > 0
        and
        poisson_test[
            "mean_batch_size"
        ] < train_rows
        and
        poisson_test[
            "max_batch_size"
        ] > 0
    )

    # ----------------------------------------------------------------------------------------------
    # 4.6 Actual Opacus wrapping
    #
    # Section 13 uses REFERENCE_SIGMA for mechanism compatibility
    # validation only.
    #
    # Final calibrated noise and achieved epsilon remain deferred
    # to the subsequent privacy-accounting/training stage.
    # ----------------------------------------------------------------------------------------------

    wrapping_pass = False

    private_critic = None
    private_optimizer = None
    private_loader = None

    try:

        (
            private_critic,
            private_optimizer,
            private_loader,
        ) = privacy_engine.make_private(
            module=critic,
            optimizer=optimizer,
            data_loader=dummy_loader,
            noise_multiplier=float(
                REFERENCE_SIGMA
            ),
            max_grad_norm=float(
                MAX_GRAD_NORM
            ),
            batch_first=True,
            loss_reduction="mean",
            poisson_sampling=True,
            clipping="flat",
            grad_sample_mode="hooks",
            wrap_model=True,
        )

        wrapping_pass = bool(
            private_critic is not None
            and
            private_optimizer is not None
            and
            private_loader is not None
        )

    except Exception as exc:

        wrapping_pass = False

        print(
            f"⚠ DP wrapping failed for "
            f"{dataset_id}: {exc}"
        )

    # ----------------------------------------------------------------------------------------------
    # 4.7 Verify production loader type
    # ----------------------------------------------------------------------------------------------

    dp_loader_pass = False

    if wrapping_pass:

        loader_class_name = (
            private_loader.__class__.__name__
        )

        dp_loader_pass = bool(
            "DPDataLoader"
            in loader_class_name
        )

    # ----------------------------------------------------------------------------------------------
    # 4.8 Verify wrapping configuration
    # ----------------------------------------------------------------------------------------------

    wrapped_noise_multiplier = None
    wrapped_max_grad_norm = None

    if wrapping_pass:

        wrapped_noise_multiplier = float(
            private_optimizer.noise_multiplier
        )

        wrapped_max_grad_norm = float(
            private_optimizer.max_grad_norm
        )

    noise_configuration_pass = bool(
        wrapping_pass
        and
        wrapped_noise_multiplier is not None
        and
        np.isclose(
            wrapped_noise_multiplier,
            REFERENCE_SIGMA,
        )
    )

    clipping_configuration_pass = bool(
        wrapping_pass
        and
        wrapped_max_grad_norm is not None
        and
        np.isclose(
            wrapped_max_grad_norm,
            MAX_GRAD_NORM,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 4.9 Final dataset status
    # ----------------------------------------------------------------------------------------------

    status = (
        "PASS"
        if all(
            [
                critic_compatible,
                engine_compatible,
                poisson_pass,
                wrapping_pass,
                dp_loader_pass,
                noise_configuration_pass,
                clipping_configuration_pass,
            ]
        )
        else "FAIL"
    )

    PRIVACY_MECHANISM_ROWS.append(
        {
            "dataset": dataset_id,
            "transformed_dimension": transformed_dimension,
            "train_rows": train_rows,
            "sample_rate_q": sample_rate,
            "reference_noise_multiplier": REFERENCE_SIGMA,
            "max_grad_norm": MAX_GRAD_NORM,
            "critic_compatible": critic_compatible,
            "privacy_engine_compatible": engine_compatible,
            "poisson_sampling": poisson_pass,
            "dp_wrapping": wrapping_pass,
            "dp_dataloader": dp_loader_pass,
            "wrapped_noise_multiplier": wrapped_noise_multiplier,
            "noise_configuration": noise_configuration_pass,
            "wrapped_max_grad_norm": wrapped_max_grad_norm,
            "clipping_configuration": clipping_configuration_pass,
            "status": status,
        }
    )

    print(
        f"✓ {dataset_id:<20} : "
        f"critic={critic_compatible} | "
        f"engine={engine_compatible} | "
        f"poisson={poisson_pass} | "
        f"DPDataLoader={dp_loader_pass} | "
        f"status={status}"
    )

    # ----------------------------------------------------------------------------------------------
    # 4.10 Cleanup
    # ----------------------------------------------------------------------------------------------

    del (
        critic,
        optimizer,
        dummy_loader,
        dummy_dataset,
        dummy_x,
        privacy_engine,
        private_critic,
        private_optimizer,
        private_loader,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# --------------------------------------------------------------------------------------------------
# 5. Build result DataFrame
# --------------------------------------------------------------------------------------------------

PRIVACY_MECHANISM_DF = pd.DataFrame(
    PRIVACY_MECHANISM_ROWS
)


# --------------------------------------------------------------------------------------------------
# 6. Validate complete mechanism
# --------------------------------------------------------------------------------------------------

if PRIVACY_MECHANISM_DF.empty:

    raise RuntimeError(
        "Privacy mechanism validation produced no results."
    )

if not (
    PRIVACY_MECHANISM_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):

    raise RuntimeError(
        "Privacy mechanism validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 7. Persist validation artifact
# --------------------------------------------------------------------------------------------------

PRIVACY_MECHANISM_PATH = (
    DIRS["validation"]
    /
    "dp_privacy_mechanism_validation.csv"
)

PRIVACY_MECHANISM_DF.to_csv(
    PRIVACY_MECHANISM_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 8. Display results
# --------------------------------------------------------------------------------------------------

display(
    PRIVACY_MECHANISM_DF
)

print(
    f"✓ Privacy mechanism validation saved:\n"
    f"  {PRIVACY_MECHANISM_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 9. Final verification
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 13 FINAL VERIFICATION")
print("-" * 100)

print(
    "✓ Critic compatibility                   : PASS"
)

print(
    "✓ PrivacyEngine compatibility            : PASS"
)

print(
    "✓ Poisson sampling validation            : PASS"
)

print(
    "✓ Opacus DP wrapping                     : PASS"
)

print(
    "✓ DPDataLoader validation                : PASS"
)

print(
    "✓ Reference noise configuration          : PASS"
)

print(
    "✓ Flat clipping configuration            : PASS"
)

print(
    "✓ Validation artifact persisted          : PASS"
)

print(
    "\nSECTION 13 STATUS: PASS"
)


13. VALIDATE PRIVACY MECHANISM
✓ Reference noise multiplier : 1.000000
✓ Maximum gradient norm      : 1.000000
✓ DP batch size              : 128
✓ adult_income         : critic=True | engine=True | poisson=True | DPDataLoader=True | status=PASS
✓ bank_marketing       : critic=True | engine=True | poisson=True | DPDataLoader=True | status=PASS
✓ diabetes_130us       : critic=True | engine=True | poisson=True | DPDataLoader=True | status=PASS


/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


,dataset,transformed_dimension,train_rows,sample_rate_q,reference_noise_multiplier,max_grad_norm,critic_compatible,privacy_engine_compatible,poisson_sampling,dp_wrapping,dp_dataloader,wrapped_noise_multiplier,noise_configuration,wrapped_max_grad_norm,clipping_configuration,status
0,adult_income,105,34189,0.003744,1.0,1.0,True,True,True,True,True,1.0,True,1.0,True,PASS
1,bank_marketing,51,31647,0.004045,1.0,1.0,True,True,True,True,True,1.0,True,1.0,True,PASS
2,diabetes_130us,2329,71236,0.001797,1.0,1.0,True,True,True,True,True,1.0,True,1.0,True,PASS


✓ Privacy mechanism validation saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/dp_privacy_mechanism_validation.csv

----------------------------------------------------------------------------------------------------
SECTION 13 FINAL VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Critic compatibility                   : PASS
✓ PrivacyEngine compatibility            : PASS
✓ Poisson sampling validation            : PASS
✓ Opacus DP wrapping                     : PASS
✓ DPDataLoader validation                : PASS
✓ Reference noise configuration          : PASS
✓ Flat clipping configuration            : PASS
✓ Validation artifact persisted          : PASS

SECTION 13 STATUS: PASS


In [69]:
# ==================================================================================================
# 14. RECORD PRIVACY METADATA
# ==================================================================================================

print("\n" + "=" * 100)
print("14. RECORD PRIVACY METADATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate canonical privacy parameter source
# --------------------------------------------------------------------------------------------------

REQUIRED_PRIVACY_COLUMNS = {
    "dataset",
    "train_rows",
    "batch_size",
    "sampling_rate_q",
    "epochs",
    "planned_steps_per_epoch",
    "planned_total_steps",
    "max_grad_norm",
    "target_epsilon",
    "delta",
    "accountant",
    "sampling",
    "clipping",
    "noise_mechanism",
    "secure_mode",
    "rng_mode",
}

missing_privacy_columns = (
    REQUIRED_PRIVACY_COLUMNS
    -
    set(
        PRIVACY_PARAMETER_DF.columns
    )
)

if missing_privacy_columns:

    raise RuntimeError(
        "PRIVACY_PARAMETER_DF is missing required canonical columns: "
        f"{sorted(missing_privacy_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate canonical dataset coverage
# --------------------------------------------------------------------------------------------------

if not set(DATASET_IDS).issubset(
    set(
        PRIVACY_PARAMETER_DF[
            "dataset"
        ]
    )
):

    raise RuntimeError(
        "PRIVACY_PARAMETER_DF does not contain all canonical datasets."
    )


if len(
    PRIVACY_PARAMETER_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Expected exactly one privacy-parameter row per dataset."
    )


if (
    PRIVACY_PARAMETER_DF[
        "dataset"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "PRIVACY_PARAMETER_DF contains duplicate dataset rows."
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate reference noise parameter
# --------------------------------------------------------------------------------------------------

REFERENCE_SIGMA = float(
    REFERENCE_SIGMA
)

if not np.isfinite(
    REFERENCE_SIGMA
) or REFERENCE_SIGMA <= 0.0:

    raise RuntimeError(
        "REFERENCE_SIGMA must be finite and strictly positive."
    )


print(
    f"✓ Canonical privacy parameter source validated."
)

print(
    f"✓ Reference noise multiplier : "
    f"{REFERENCE_SIGMA:.6f}"
)


# --------------------------------------------------------------------------------------------------
# 4. Build privacy metadata
# --------------------------------------------------------------------------------------------------

PRIVACY_METADATA_ROWS = []


for row in PRIVACY_PARAMETER_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    if dataset_id not in DATASET_IDS:

        raise RuntimeError(
            f"Unexpected dataset in privacy metadata: "
            f"{dataset_id}"
        )

    # ----------------------------------------------------------------------------------------------
    # 4.1 Dataset and privacy parameters
    # ----------------------------------------------------------------------------------------------

    PRIVACY_METADATA_ROWS.append({

        "dataset":
            dataset_id,

        "n_train":
            int(
                row.train_rows
            ),

        "target_epsilon":
            float(
                row.target_epsilon
            ),

        "delta":
            float(
                row.delta
            ),

        "batch_size":
            int(
                row.batch_size
            ),

        "sample_rate":
            float(
                row.sampling_rate_q
            ),

        "epochs":
            int(
                row.epochs
            ),

        "nominal_steps_per_epoch":
            float(
                row.planned_steps_per_epoch
            ),

        "nominal_total_steps":
            float(
                row.planned_total_steps
            ),

        "max_grad_norm":
            float(
                row.max_grad_norm
            ),

        "noise_multiplier":
            float(
                REFERENCE_SIGMA
            ),

        # ------------------------------------------------------------------------------------------
        # 4.2 Mechanism configuration
        # ------------------------------------------------------------------------------------------

        "accountant":
            str(
                row.accountant
            ),

        "sampling":
            str(
                row.sampling
            ),

        "clipping":
            str(
                row.clipping
            ),

        "loss_reduction":
            "mean",

        "noise_mechanism":
            str(
                row.noise_mechanism
            ),

        "secure_mode":
            bool(
                row.secure_mode
            ),

        "rng_mode":
            str(
                row.rng_mode
            ),

        "protected_component":
            "SPP-GAN discriminator",

        "per_example_gradients":
            True,

        # ------------------------------------------------------------------------------------------
        # 4.3 Privacy boundary
        # ------------------------------------------------------------------------------------------

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "conditioning_private":
            False,

        # ------------------------------------------------------------------------------------------
        # 4.4 Achieved privacy accounting
        # ------------------------------------------------------------------------------------------

        "achieved_epsilon":
            None,

        "achieved_epsilon_status":
            "DEFERRED_TO_NOTEBOOK_11",

        # ------------------------------------------------------------------------------------------
        # 4.5 Downstream execution status
        # ------------------------------------------------------------------------------------------

        "training_status":
            "DEFERRED_TO_NOTEBOOK_12",

        "synthetic_generation_status":
            "DEFERRED_TO_NOTEBOOK_13",

        # ------------------------------------------------------------------------------------------
        # 4.6 Formal end-to-end privacy claim
        # ------------------------------------------------------------------------------------------

        "end_to_end_privacy_claim":
            False,

        "status":
            "PASS",
    })


# --------------------------------------------------------------------------------------------------
# 5. Create metadata DataFrame
# --------------------------------------------------------------------------------------------------

PRIVACY_METADATA_DF = pd.DataFrame(
    PRIVACY_METADATA_ROWS
)


# --------------------------------------------------------------------------------------------------
# 6. Validate metadata coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_METADATA_DATASETS = set(
    DATASET_IDS
)

if set(
    PRIVACY_METADATA_DF[
        "dataset"
    ]
) != EXPECTED_METADATA_DATASETS:

    raise RuntimeError(
        "Privacy metadata dataset coverage does not match DATASET_IDS."
    )


if len(
    PRIVACY_METADATA_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Privacy metadata must contain exactly one row per dataset."
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate numeric metadata
# --------------------------------------------------------------------------------------------------

numeric_metadata_columns = [
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
]

for column in numeric_metadata_columns:

    values = pd.to_numeric(
        PRIVACY_METADATA_DF[
            column
        ],
        errors="coerce",
    )

    if not np.isfinite(
        values.to_numpy(
            dtype=float
        )
    ).all():

        raise RuntimeError(
            "Non-finite privacy metadata detected "
            f"in column: {column}"
        )


# --------------------------------------------------------------------------------------------------
# 8. Validate privacy boundary
# --------------------------------------------------------------------------------------------------

if not PRIVACY_METADATA_DF[
    "generator_private"
].eq(False).all():

    raise RuntimeError(
        "Generator privacy boundary is inconsistent."
    )


if not PRIVACY_METADATA_DF[
    "statistical_guidance_private"
].eq(False).all():

    raise RuntimeError(
        "Statistical-guidance privacy boundary is inconsistent."
    )


if not PRIVACY_METADATA_DF[
    "preprocessing_private"
].eq(False).all():

    raise RuntimeError(
        "Preprocessing privacy boundary is inconsistent."
    )


if not PRIVACY_METADATA_DF[
    "conditioning_private"
].eq(False).all():

    raise RuntimeError(
        "Conditioning privacy boundary is inconsistent."
    )


if not PRIVACY_METADATA_DF[
    "end_to_end_privacy_claim"
].eq(False).all():

    raise RuntimeError(
        "End-to-end privacy claim must remain disabled "
        "in Notebook 10."
    )


# --------------------------------------------------------------------------------------------------
# 9. Validate achieved-epsilon status
# --------------------------------------------------------------------------------------------------

if not PRIVACY_METADATA_DF[
    "achieved_epsilon_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_11"
).all():

    raise RuntimeError(
        "Achieved epsilon must remain deferred to Notebook 11."
    )


if not PRIVACY_METADATA_DF[
    "achieved_epsilon"
].isna().all():

    raise RuntimeError(
        "Achieved epsilon must remain unset in Notebook 10."
    )


# --------------------------------------------------------------------------------------------------
# 10. Validate downstream status
# --------------------------------------------------------------------------------------------------

if not PRIVACY_METADATA_DF[
    "training_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_12"
).all():

    raise RuntimeError(
        "Training status must remain deferred to Notebook 12."
    )


if not PRIVACY_METADATA_DF[
    "synthetic_generation_status"
].eq(
    "DEFERRED_TO_NOTEBOOK_13"
).all():

    raise RuntimeError(
        "Synthetic generation status must remain deferred "
        "to Notebook 13."
    )


# --------------------------------------------------------------------------------------------------
# 11. Validate mechanism consistency
# --------------------------------------------------------------------------------------------------

if not PRIVACY_METADATA_DF[
    "noise_multiplier"
].eq(
    REFERENCE_SIGMA
).all():

    raise RuntimeError(
        "Recorded noise multiplier does not match "
        "REFERENCE_SIGMA."
    )


if not PRIVACY_METADATA_DF[
    "max_grad_norm"
].gt(
    0
).all():

    raise RuntimeError(
        "Maximum gradient norm must be positive."
    )


# --------------------------------------------------------------------------------------------------
# 12. Persist metadata
# --------------------------------------------------------------------------------------------------

PRIVACY_METADATA_PATH = (
    DIRS["metadata"]
    /
    "sppgan_privacy_metadata.csv"
)

PRIVACY_METADATA_DF.to_csv(
    PRIVACY_METADATA_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 13. Display metadata
# --------------------------------------------------------------------------------------------------

display(
    PRIVACY_METADATA_DF
)


# --------------------------------------------------------------------------------------------------
# 14. Final verification
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Privacy metadata saved:\n"
    f"  {PRIVACY_METADATA_PATH}"
)

print(
    f"✓ Dataset coverage validated: "
    f"{len(PRIVACY_METADATA_DF)} datasets"
)

print(
    "✓ Canonical privacy parameters preserved."
)

print(
    "✓ Reference noise multiplier recorded from "
    "REFERENCE_SIGMA."
)

print(
    "✓ Achieved epsilon correctly deferred to Notebook 11."
)

print(
    "✓ Training correctly deferred to Notebook 12."
)

print(
    "✓ Synthetic generation correctly deferred to Notebook 13."
)

print(
    "✓ Generator privacy boundary remains disabled."
)

print(
    "✓ Statistical-guidance privacy boundary remains disabled."
)

print(
    "✓ Preprocessing privacy boundary remains disabled."
)

print(
    "✓ End-to-end privacy claim remains disabled."
)

print(
    "\nSECTION 14 STATUS: PASS"
)


14. RECORD PRIVACY METADATA
✓ Canonical privacy parameter source validated.
✓ Reference noise multiplier : 1.000000


,dataset,n_train,target_epsilon,delta,batch_size,sample_rate,epochs,nominal_steps_per_epoch,nominal_total_steps,max_grad_norm,...,generator_private,statistical_guidance_private,preprocessing_private,conditioning_private,achieved_epsilon,achieved_epsilon_status,training_status,synthetic_generation_status,end_to_end_privacy_claim,status
0,adult_income,34189,5.0,0.00001,128,0.003744,300,268.0,80400.0,1.0,...,False,False,False,False,None,DEFERRED_TO_NOTEBOOK_11,DEFERRED_TO_NOTEBOOK_12,DEFERRED_TO_NOTEBOOK_13,False,PASS
1,bank_marketing,31647,5.0,0.00001,128,0.004045,300,248.0,74400.0,1.0,...,False,False,False,False,None,DEFERRED_TO_NOTEBOOK_11,DEFERRED_TO_NOTEBOOK_12,DEFERRED_TO_NOTEBOOK_13,False,PASS
2,diabetes_130us,71236,5.0,0.00001,128,0.001797,300,557.0,167100.0,1.0,...,False,False,False,False,None,DEFERRED_TO_NOTEBOOK_11,DEFERRED_TO_NOTEBOOK_12,DEFERRED_TO_NOTEBOOK_13,False,PASS


✓ Privacy metadata saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/sppgan_privacy_metadata.csv
✓ Dataset coverage validated: 3 datasets
✓ Canonical privacy parameters preserved.
✓ Reference noise multiplier recorded from REFERENCE_SIGMA.
✓ Achieved epsilon correctly deferred to Notebook 11.
✓ Training correctly deferred to Notebook 12.
✓ Synthetic generation correctly deferred to Notebook 13.
✓ Generator privacy boundary remains disabled.
✓ Statistical-guidance privacy boundary remains disabled.
✓ Preprocessing privacy boundary remains disabled.
✓ End-to-end privacy claim remains disabled.

SECTION 14 STATUS: PASS


In [73]:
# ==================================================================================================
# 15. SAVE DP MODULE
# ==================================================================================================

print("\n" + "=" * 100)
print("15. SAVE DP MODULE")
print("=" * 100)

import hashlib
import importlib.util
import inspect
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.grad_sample import GradSampleModule


# --------------------------------------------------------------------------------------------------
# 1. Validate required Notebook 10 objects
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_15_OBJECTS = [
    "SPPGANCritic",
    "ARCHITECTURE_SUMMARY_DF",
    "CRITIC_INPUT_DIMS",
    "REFERENCE_SIGMA",
    "MAX_GRAD_NORM",
    "DIRS",
]

missing_objects = [
    name
    for name in REQUIRED_SECTION_15_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 15 is missing required objects: "
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Validate DP configuration
# --------------------------------------------------------------------------------------------------

REFERENCE_SIGMA = float(
    REFERENCE_SIGMA
)

MAX_GRAD_NORM = float(
    MAX_GRAD_NORM
)

if not np.isfinite(
    REFERENCE_SIGMA
) or REFERENCE_SIGMA <= 0.0:

    raise ValueError(
        "REFERENCE_SIGMA must be finite and strictly positive."
    )


if not np.isfinite(
    MAX_GRAD_NORM
) or MAX_GRAD_NORM <= 0.0:

    raise ValueError(
        "MAX_GRAD_NORM must be finite and strictly positive."
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate canonical Notebook 08 critic interface
# --------------------------------------------------------------------------------------------------

CANONICAL_CRITIC_CLASS = SPPGANCritic

if not inspect.isclass(
    CANONICAL_CRITIC_CLASS
):

    raise TypeError(
        "SPPGANCritic must be a class."
    )


if not issubclass(
    CANONICAL_CRITIC_CLASS,
    nn.Module,
):

    raise TypeError(
        "SPPGANCritic must inherit from torch.nn.Module."
    )


CRITIC_SIGNATURE = inspect.signature(
    CANONICAL_CRITIC_CLASS.__init__
)

CRITIC_PARAMETER_NAMES = [
    parameter.name
    for parameter in CRITIC_SIGNATURE.parameters.values()
    if parameter.name != "self"
]


if CRITIC_PARAMETER_NAMES != [
    "input_dim"
]:

    raise RuntimeError(
        "Frozen SPPGANCritic constructor mismatch.\n"
        f"Expected : ['input_dim']\n"
        f"Observed : {CRITIC_PARAMETER_NAMES}\n"
        f"Signature: {CRITIC_SIGNATURE}"
    )


if "hidden_dims" in CRITIC_PARAMETER_NAMES:

    raise RuntimeError(
        "The frozen SPPGANCritic must not expose "
        "'hidden_dims'."
    )


print(
    "✓ Canonical SPPGANCritic detected."
)

print(
    f"✓ Frozen critic constructor : {CRITIC_SIGNATURE}"
)


# --------------------------------------------------------------------------------------------------
# 4. Validate Notebook 08 architecture schema
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = [
    "dataset",
    "transformed_dimension",
    "critic_hidden_1",
    "critic_hidden_2",
    "critic_output",
]

missing_architecture_columns = [
    column
    for column in REQUIRED_ARCHITECTURE_COLUMNS
    if column not in ARCHITECTURE_SUMMARY_DF.columns
]

if missing_architecture_columns:

    raise RuntimeError(
        "Notebook 08 architecture summary is missing "
        f"required columns: {missing_architecture_columns}"
    )


EXPECTED_CRITIC_HIDDEN_1 = 256
EXPECTED_CRITIC_HIDDEN_2 = 256
EXPECTED_CRITIC_OUTPUT = "scalar"


architecture_datasets = set(
    ARCHITECTURE_SUMMARY_DF[
        "dataset"
    ].astype(str)
)

critic_dimension_datasets = set(
    CRITIC_INPUT_DIMS.keys()
)

if architecture_datasets != critic_dimension_datasets:

    raise RuntimeError(
        "Architecture dataset coverage does not match "
        "CRITIC_INPUT_DIMS."
    )


for row in ARCHITECTURE_SUMMARY_DF.itertuples(
    index=False
):

    dataset_id = str(
        row.dataset
    )

    if dataset_id not in CRITIC_INPUT_DIMS:
        continue

    if int(
        row.critic_hidden_1
    ) != EXPECTED_CRITIC_HIDDEN_1:

        raise RuntimeError(
            f"{dataset_id}: critic hidden layer 1 mismatch."
        )

    if int(
        row.critic_hidden_2
    ) != EXPECTED_CRITIC_HIDDEN_2:

        raise RuntimeError(
            f"{dataset_id}: critic hidden layer 2 mismatch."
        )

    critic_output = str(
        row.critic_output
    ).strip().lower()

    if critic_output != EXPECTED_CRITIC_OUTPUT:

        raise RuntimeError(
            f"{dataset_id}: critic output mismatch. "
            f"Expected={EXPECTED_CRITIC_OUTPUT}, "
            f"Observed={critic_output}"
        )


print(
    "✓ Notebook 08 critic architecture contract validated."
)

print(
    "✓ Critic hidden layers : 256 / 256"
)

print(
    "✓ Critic output        : scalar"
)


# --------------------------------------------------------------------------------------------------
# 5. Define the frozen architecture snapshot
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# The canonical SPPGANCritic was defined dynamically in the Colab/Jupyter
# namespace. Therefore inspect.getsource() is unavailable.
#
# We DO NOT fabricate a source fingerprint.
#
# Instead, the frozen architecture contract from Notebook 08 is used:
#
#     Linear(input_dim, 256)
#     LeakyReLU(0.2)
#     Linear(256, 256)
#     LeakyReLU(0.2)
#     Linear(256, 1)
#
# The persisted critic is subsequently compared against the canonical
# in-memory critic by parameter names, shapes, counts, and forward output.
# --------------------------------------------------------------------------------------------------

FROZEN_CRITIC_ARCHITECTURE = {
    "constructor": "(self, input_dim)",
    "input_argument": "input_dim",
    "hidden_layer_1": 256,
    "activation_1": "LeakyReLU(0.2)",
    "hidden_layer_2": 256,
    "activation_2": "LeakyReLU(0.2)",
    "output_dimension": 1,
    "output_type": "scalar",
    "sigmoid": False,
    "dropout": False,
    "batch_normalization": False,
}


# --------------------------------------------------------------------------------------------------
# 6. Build canonical architecture fingerprint
# --------------------------------------------------------------------------------------------------

def architecture_fingerprint(
    model
):

    fingerprint_rows = []

    for name, parameter in model.named_parameters():

        fingerprint_rows.append(
            {
                "name":
                    str(name),

                "shape":
                    list(
                        parameter.shape
                    ),

                "dtype":
                    str(
                        parameter.dtype
                    ),

                "requires_grad":
                    bool(
                        parameter.requires_grad
                    ),
            }
        )

    payload = json.dumps(
        fingerprint_rows,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )

    return hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()


CANONICAL_CRITIC_FINGERPRINTS = {}
CANONICAL_CRITIC_PARAMETER_COUNTS = {}


for dataset_id, input_dim in (
    CRITIC_INPUT_DIMS.items()
):

    input_dim = int(
        input_dim
    )

    canonical_critic = (
        CANONICAL_CRITIC_CLASS(
            input_dim=input_dim
        )
    )

    CANONICAL_CRITIC_FINGERPRINTS[
        dataset_id
    ] = architecture_fingerprint(
        canonical_critic
    )

    CANONICAL_CRITIC_PARAMETER_COUNTS[
        dataset_id
    ] = int(
        sum(
            parameter.numel()
            for parameter
            in canonical_critic.parameters()
            if parameter.requires_grad
        )
    )

    del canonical_critic


print(
    "✓ Canonical critic structural fingerprints generated."
)


# --------------------------------------------------------------------------------------------------
# 7. Validate expected parameter counts
# --------------------------------------------------------------------------------------------------

for dataset_id, input_dim in (
    CRITIC_INPUT_DIMS.items()
):

    input_dim = int(
        input_dim
    )

    expected_parameter_count = (
        input_dim * 256
        + 256
        + 256 * 256
        + 256
        + 256
        + 1
    )

    observed_parameter_count = (
        CANONICAL_CRITIC_PARAMETER_COUNTS[
            dataset_id
        ]
    )

    if observed_parameter_count != (
        expected_parameter_count
    ):

        raise RuntimeError(
            f"{dataset_id}: canonical critic parameter "
            "count mismatch.\n"
            f"Expected : {expected_parameter_count}\n"
            f"Observed : {observed_parameter_count}"
        )


print(
    "✓ Canonical critic parameter counts validated."
)


# --------------------------------------------------------------------------------------------------
# 8. DP module output path
# --------------------------------------------------------------------------------------------------

DP_MODULE_PATH = (
    DIRS["models"]
    /
    "sppgan_differential_privacy.py"
)

DP_MODULE_PATH = Path(
    DP_MODULE_PATH
)

DP_MODULE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 9. Generate persisted DP module
# --------------------------------------------------------------------------------------------------

DP_MODULE_SOURCE = r'''
# ==================================================================================================
# SPP-GAN DIFFERENTIAL PRIVACY MODULE
# ==================================================================================================
#
# Canonical architecture snapshot
# --------------------------------
# The SPPGANCritic implementation in this module follows the frozen
# Notebook 08 architecture contract validated by Notebook 10.
#
# Frozen constructor:
#     SPPGANCritic(input_dim)
#
# Frozen critic:
#     Linear(input_dim, 256)
#     LeakyReLU(0.2)
#     Linear(256, 256)
#     LeakyReLU(0.2)
#     Linear(256, 1)
#
# The original Notebook 08 class was dynamically defined in the
# Colab/Jupyter namespace, so Python source extraction using
# inspect.getsource() is not available. Architecture equivalence
# is therefore verified structurally by Notebook 10 Section 15.
# ==================================================================================================

import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.grad_sample import GradSampleModule


# --------------------------------------------------------------------------------------------------
# FROZEN SPP-GAN CRITIC
# --------------------------------------------------------------------------------------------------

class SPPGANCritic(
    nn.Module
):

    def __init__(
        self,
        input_dim,
    ):

        super().__init__()

        self.input_dim = int(
            input_dim
        )

        if self.input_dim <= 0:

            raise ValueError(
                "Critic input dimension must be positive."
            )

        self.network = nn.Sequential(

            nn.Linear(
                self.input_dim,
                256,
            ),

            nn.LeakyReLU(
                negative_slope=0.2
            ),

            nn.Linear(
                256,
                256,
            ),

            nn.LeakyReLU(
                negative_slope=0.2
            ),

            nn.Linear(
                256,
                1,
            ),
        )

    def forward(
        self,
        x,
    ):

        if x.ndim != 2:

            raise ValueError(
                "Critic input must be a two-dimensional tensor."
            )

        return self.network(
            x
        )


# --------------------------------------------------------------------------------------------------
# PER-EXAMPLE GRADIENT WRAPPER
# --------------------------------------------------------------------------------------------------

def wrap_critic_for_per_sample_gradients(
    critic,
):

    return GradSampleModule(
        critic,
        batch_first=True,
        loss_reduction="mean",
        strict=True,
        force_functorch=False,
    )


# --------------------------------------------------------------------------------------------------
# DP DISCRIMINATOR LOSS
# --------------------------------------------------------------------------------------------------

def dp_discriminator_loss(
    private_critic,
    real_batch,
    fake_batch,
):

    if real_batch.shape != fake_batch.shape:

        raise ValueError(
            "Real and fake batches must have identical shapes."
        )

    real_scores = (
        private_critic(
            real_batch
        )
        .reshape(-1)
    )

    fake_scores = (
        private_critic(
            fake_batch.detach()
        )
        .reshape(-1)
    )

    per_example_loss = (
        fake_scores
        -
        real_scores
    )

    return (
        per_example_loss.mean(),
        per_example_loss,
    )


# --------------------------------------------------------------------------------------------------
# MAKE DISCRIMINATOR PRIVATE
# --------------------------------------------------------------------------------------------------

def make_private_discriminator(
    critic,
    optimizer,
    data_loader,
    noise_multiplier,
    max_grad_norm,
):

    privacy_engine = PrivacyEngine(
        accountant="rdp"
    )

    (
        private_critic,
        private_optimizer,
        private_loader,
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=optimizer,
        data_loader=data_loader,
        noise_multiplier=float(
            noise_multiplier
        ),
        max_grad_norm=float(
            max_grad_norm
        ),
        batch_first=True,
        loss_reduction="mean",
        poisson_sampling=True,
        clipping="flat",
        grad_sample_mode="hooks",
        wrap_model=True,
    )

    return (
        private_critic,
        private_optimizer,
        private_loader,
        privacy_engine,
    )


# --------------------------------------------------------------------------------------------------
# DP DISCRIMINATOR STEP
# --------------------------------------------------------------------------------------------------

def dp_discriminator_step(
    private_critic,
    private_optimizer,
    real_batch,
    fake_batch,
):

    private_optimizer.zero_grad(
        set_to_none=True
    )

    loss, per_example_loss = (
        dp_discriminator_loss(
            private_critic,
            real_batch,
            fake_batch,
        )
    )

    loss.backward()

    private_optimizer.step()

    return {
        "loss":
            loss.detach(),

        "per_example_loss":
            per_example_loss.detach(),
    }
'''


# --------------------------------------------------------------------------------------------------
# 10. Persist DP module
# --------------------------------------------------------------------------------------------------

with open(
    DP_MODULE_PATH,
    "w",
    encoding="utf-8",
) as file:

    file.write(
        DP_MODULE_SOURCE
    )


if not DP_MODULE_PATH.exists():

    raise RuntimeError(
        "DP module was not created."
    )


if DP_MODULE_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "DP module was created but is empty."
    )


print(
    "✓ DP module saved:"
)

print(
    f"  {DP_MODULE_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 11. Syntax validation
# --------------------------------------------------------------------------------------------------

with open(
    DP_MODULE_PATH,
    "r",
    encoding="utf-8",
) as file:

    persisted_source = file.read()


compile(
    persisted_source,
    str(
        DP_MODULE_PATH
    ),
    "exec",
)

print(
    "✓ Python syntax validation: PASS"
)


# --------------------------------------------------------------------------------------------------
# 12. Import persisted DP module
# --------------------------------------------------------------------------------------------------

MODULE_NAME = (
    "sppgan_differential_privacy"
)

SPEC = importlib.util.spec_from_file_location(
    MODULE_NAME,
    DP_MODULE_PATH,
)

if SPEC is None:

    raise RuntimeError(
        "Unable to create DP module import specification."
    )


if SPEC.loader is None:

    raise RuntimeError(
        "DP module import loader is unavailable."
    )


DP_MODULE = (
    importlib.util.module_from_spec(
        SPEC
    )
)

SPEC.loader.exec_module(
    DP_MODULE
)

print(
    "✓ Persisted DP module imported successfully."
)


# --------------------------------------------------------------------------------------------------
# 13. Validate persisted module interface
# --------------------------------------------------------------------------------------------------

REQUIRED_DP_MODULE_OBJECTS = [

    "SPPGANCritic",

    "wrap_critic_for_per_sample_gradients",

    "dp_discriminator_loss",

    "make_private_discriminator",

    "dp_discriminator_step",
]


missing_dp_module_objects = [

    name

    for name in REQUIRED_DP_MODULE_OBJECTS

    if not hasattr(
        DP_MODULE,
        name,
    )
]


if missing_dp_module_objects:

    raise RuntimeError(
        "Persisted DP module is missing required objects: "
        f"{missing_dp_module_objects}"
    )


for function_name in (

    "wrap_critic_for_per_sample_gradients",

    "dp_discriminator_loss",

    "make_private_discriminator",

    "dp_discriminator_step",

):

    if not callable(
        getattr(
            DP_MODULE,
            function_name,
        )
    ):

        raise RuntimeError(
            f"Persisted DP function is not callable: "
            f"{function_name}"
        )


print(
    "✓ Persisted DP module interface validated."
)


# --------------------------------------------------------------------------------------------------
# 14. Validate persisted critic constructor
# --------------------------------------------------------------------------------------------------

PERSISTED_CRITIC_CLASS = (
    DP_MODULE.SPPGANCritic
)

PERSISTED_CRITIC_SIGNATURE = inspect.signature(
    PERSISTED_CRITIC_CLASS.__init__
)

PERSISTED_CRITIC_PARAMETER_NAMES = [

    parameter.name

    for parameter
    in PERSISTED_CRITIC_SIGNATURE.parameters.values()

    if parameter.name != "self"
]


if PERSISTED_CRITIC_PARAMETER_NAMES != [
    "input_dim"
]:

    raise RuntimeError(
        "Persisted SPPGANCritic constructor mismatch.\n"
        f"Expected : ['input_dim']\n"
        f"Observed : {PERSISTED_CRITIC_PARAMETER_NAMES}\n"
        f"Signature: {PERSISTED_CRITIC_SIGNATURE}"
    )


print(
    "✓ Persisted critic constructor validated."
)


# --------------------------------------------------------------------------------------------------
# 15. Structural equivalence validation
# --------------------------------------------------------------------------------------------------

PERSISTED_CRITIC_FINGERPRINTS = {}
PERSISTED_CRITIC_PARAMETER_COUNTS = {}


for dataset_id, input_dim in (
    CRITIC_INPUT_DIMS.items()
):

    input_dim = int(
        input_dim
    )

    canonical_critic = (
        CANONICAL_CRITIC_CLASS(
            input_dim=input_dim
        )
    )

    persisted_critic = (
        PERSISTED_CRITIC_CLASS(
            input_dim=input_dim
        )
    )

    canonical_state = (
        canonical_critic.state_dict()
    )

    persisted_state = (
        persisted_critic.state_dict()
    )


    if list(
        canonical_state.keys()
    ) != list(
        persisted_state.keys()
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted critic parameter "
            "names differ from canonical critic."
        )


    for parameter_name in canonical_state:

        canonical_tensor = (
            canonical_state[
                parameter_name
            ]
        )

        persisted_tensor = (
            persisted_state[
                parameter_name
            ]
        )


        if (
            canonical_tensor.shape
            !=
            persisted_tensor.shape
        ):

            raise RuntimeError(
                f"{dataset_id}: parameter shape mismatch "
                f"for {parameter_name}."
            )


        if (
            canonical_tensor.dtype
            !=
            persisted_tensor.dtype
        ):

            raise RuntimeError(
                f"{dataset_id}: parameter dtype mismatch "
                f"for {parameter_name}."
            )


    canonical_parameter_count = sum(
        parameter.numel()
        for parameter
        in canonical_critic.parameters()
        if parameter.requires_grad
    )

    persisted_parameter_count = sum(
        parameter.numel()
        for parameter
        in persisted_critic.parameters()
        if parameter.requires_grad
    )


    if (
        canonical_parameter_count
        !=
        persisted_parameter_count
    ):

        raise RuntimeError(
            f"{dataset_id}: parameter count mismatch.\n"
            f"Canonical : {canonical_parameter_count}\n"
            f"Persisted : {persisted_parameter_count}"
        )


    canonical_fingerprint = (
        architecture_fingerprint(
            canonical_critic
        )
    )

    persisted_fingerprint = (
        architecture_fingerprint(
            persisted_critic
        )
    )


    CANONICAL_CRITIC_FINGERPRINTS[
        dataset_id
    ] = canonical_fingerprint

    PERSISTED_CRITIC_FINGERPRINTS[
        dataset_id
    ] = persisted_fingerprint


    CANONICAL_CRITIC_PARAMETER_COUNTS[
        dataset_id
    ] = int(
        canonical_parameter_count
    )

    PERSISTED_CRITIC_PARAMETER_COUNTS[
        dataset_id
    ] = int(
        persisted_parameter_count
    )


    if (
        canonical_fingerprint
        !=
        persisted_fingerprint
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted critic architecture "
            "fingerprint differs from canonical critic."
        )


    del (
        canonical_critic,
        persisted_critic,
    )


print(
    "✓ Persisted critic structurally matches canonical critic."
)


# --------------------------------------------------------------------------------------------------
# 16. Validate forward-output contract
# --------------------------------------------------------------------------------------------------

for dataset_id, input_dim in (
    CRITIC_INPUT_DIMS.items()
):

    input_dim = int(
        input_dim
    )

    canonical_critic = (
        CANONICAL_CRITIC_CLASS(
            input_dim=input_dim
        )
    )

    persisted_critic = (
        PERSISTED_CRITIC_CLASS(
            input_dim=input_dim
        )
    )

    canonical_critic.eval()
    persisted_critic.eval()


    test_batch = torch.zeros(
        4,
        input_dim,
        dtype=torch.float32,
    )


    with torch.no_grad():

        canonical_output = (
            canonical_critic(
                test_batch
            )
        )

        persisted_output = (
            persisted_critic(
                test_batch
            )
        )


    if tuple(
        canonical_output.shape
    ) != (
        4,
        1,
    ):

        raise RuntimeError(
            f"{dataset_id}: canonical critic output "
            "shape is not (4, 1)."
        )


    if tuple(
        persisted_output.shape
    ) != (
        4,
        1,
    ):

        raise RuntimeError(
            f"{dataset_id}: persisted critic output "
            "shape is not (4, 1)."
        )


    if not torch.isfinite(
        canonical_output
    ).all().item():

        raise RuntimeError(
            f"{dataset_id}: canonical critic produced "
            "non-finite output."
        )


    if not torch.isfinite(
        persisted_output
    ).all().item():

        raise RuntimeError(
            f"{dataset_id}: persisted critic produced "
            "non-finite output."
        )


    del (
        canonical_critic,
        persisted_critic,
        test_batch,
        canonical_output,
        persisted_output,
    )


print(
    "✓ Canonical and persisted critic forward contracts validated."
)


# --------------------------------------------------------------------------------------------------
# 17. Persist module provenance metadata
# --------------------------------------------------------------------------------------------------

DP_MODULE_METADATA_PATH = (
    DIRS["models"]
    /
    "sppgan_differential_privacy_metadata.json"
)


DP_MODULE_METADATA = {

    "module":
        "sppgan_differential_privacy",

    "module_path":
        str(
            DP_MODULE_PATH
        ),

    "architecture_source":
        "FROZEN_NOTEBOOK_08_ARCHITECTURE_CONTRACT",

    "source_code_fingerprint":
        "NOT_AVAILABLE_DYNAMIC_NOTEBOOK_CLASS",

    "critic_class":
        "SPPGANCritic",

    "critic_constructor":
        str(
            CRITIC_SIGNATURE
        ),

    "critic_architecture":
        FROZEN_CRITIC_ARCHITECTURE,

    "canonical_critic_fingerprints":
        CANONICAL_CRITIC_FINGERPRINTS,

    "persisted_critic_fingerprints":
        PERSISTED_CRITIC_FINGERPRINTS,

    "canonical_parameter_counts":
        CANONICAL_CRITIC_PARAMETER_COUNTS,

    "persisted_parameter_counts":
        PERSISTED_CRITIC_PARAMETER_COUNTS,

    "critic_input_dimensions":
        {
            dataset_id:
                int(
                    input_dim
                )
            for dataset_id, input_dim
            in CRITIC_INPUT_DIMS.items()
        },

    "accountant":
        "rdp",

    "sampling":
        "poisson",

    "clipping":
        "flat",

    "loss_reduction":
        "mean",

    "reference_noise_multiplier":
        float(
            REFERENCE_SIGMA
        ),

    "max_grad_norm":
        float(
            MAX_GRAD_NORM
        ),

    "secure_mode":
        False,

    "rng_mode":
        "STANDARD_PYTORCH_RNG",

    "cryptographically_secure_rng":
        False,

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "conditioning_private":
        False,

    "end_to_end_privacy_claim":
        False,
}


with open(
    DP_MODULE_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        DP_MODULE_METADATA,
        file,
        indent=2,
        sort_keys=True,
    )


print(
    "✓ DP module provenance metadata saved:"
)

print(
    f"  {DP_MODULE_METADATA_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 18. Final verification
# --------------------------------------------------------------------------------------------------

FINAL_SECTION_15_CHECKS = {

    "dp_module_exists":
        DP_MODULE_PATH.exists(),

    "dp_module_nonempty":
        DP_MODULE_PATH.stat().st_size > 0,

    "python_syntax_valid":
        True,

    "dp_module_imported":
        DP_MODULE is not None,

    "sppgan_critic_available":
        hasattr(
            DP_MODULE,
            "SPPGANCritic",
        ),

    "critic_constructor_frozen":
        PERSISTED_CRITIC_PARAMETER_NAMES
        ==
        [
            "input_dim"
        ],

    "critic_architecture_contract_validated":
        True,

    "critic_parameter_counts_match":
        (
            CANONICAL_CRITIC_PARAMETER_COUNTS
            ==
            PERSISTED_CRITIC_PARAMETER_COUNTS
        ),

    "critic_structural_fingerprints_match":
        (
            CANONICAL_CRITIC_FINGERPRINTS
            ==
            PERSISTED_CRITIC_FINGERPRINTS
        ),

    "forward_contract_validated":
        True,

    "dp_functions_available":
        all(
            hasattr(
                DP_MODULE,
                function_name,
            )
            for function_name in (

                "wrap_critic_for_per_sample_gradients",

                "dp_discriminator_loss",

                "make_private_discriminator",

                "dp_discriminator_step",
            )
        ),

    "rdp_accountant":
        True,

    "poisson_sampling":
        True,

    "flat_clipping":
        True,

    "mean_loss_reduction":
        True,

    "secure_mode_false":
        True,

    # IMPORTANT:
    # The project intentionally does NOT make an end-to-end
    # record-level DP claim in Notebook 10.
    #
    # Therefore the validation condition is:
    # "end-to-end privacy claim is disabled"
    # rather than:
    # "end-to-end privacy claim is True".
    "end_to_end_privacy_claim_disabled":
        (
            "end_to_end_privacy_claim"
            in DP_MODULE_METADATA
            and
            DP_MODULE_METADATA[
                "end_to_end_privacy_claim"
            ]
            is False
        ),
}


FAILED_SECTION_15_CHECKS = [
    name
    for name, passed
    in FINAL_SECTION_15_CHECKS.items()
    if not passed
]


print("\n" + "-" * 100)
print("SECTION 15 FINAL VERIFICATION")
print("-" * 100)


for check_name, passed in (
    FINAL_SECTION_15_CHECKS.items()
):

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name.replace('_', ' ')}"
    )


if FAILED_SECTION_15_CHECKS:

    raise RuntimeError(
        "Section 15 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_15_CHECKS
        )
    )


print()
print(
    f"✓ DP module saved:\n"
    f"  {DP_MODULE_PATH}"
)

print(
    f"✓ DP module metadata saved:\n"
    f"  {DP_MODULE_METADATA_PATH}"
)

print(
    "✓ Frozen Notebook 08 critic contract preserved."
)

print(
    "✓ No hidden_dims argument introduced."
)

print(
    "✓ Canonical and persisted critic structures match."
)

print(
    "✓ Canonical and persisted parameter counts match."
)

print(
    "✓ Canonical and persisted architecture fingerprints match."
)

print(
    "✓ RDP / Poisson / flat clipping / mean-loss contract preserved."
)

print(
    "✓ Standard PyTorch RNG explicitly preserved."
)

print(
    "✓ End-to-end privacy claim correctly remains disabled."
)

print(
    "\nSECTION 15 STATUS: PASS"
)


15. SAVE DP MODULE
✓ Canonical SPPGANCritic detected.
✓ Frozen critic constructor : (self, input_dim)
✓ Notebook 08 critic architecture contract validated.
✓ Critic hidden layers : 256 / 256
✓ Critic output        : scalar
✓ Canonical critic structural fingerprints generated.
✓ Canonical critic parameter counts validated.
✓ DP module saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_privacy.py
✓ Python syntax validation: PASS
✓ Persisted DP module imported successfully.
✓ Persisted DP module interface validated.
✓ Persisted critic constructor validated.
✓ Persisted critic structurally matches canonical critic.
✓ Canonical and persisted critic forward contracts validated.
✓ DP module provenance metadata saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_privacy_metadata.json

----------------------------------------------------------------------------------------------------
S

In [74]:
# ==================================================================================================
# 16. SAVE PRIVACY CONFIGURATION
# ==================================================================================================

print("\n" + "=" * 100)
print("16. SAVE PRIVACY CONFIGURATION")
print("=" * 100)

PERSISTED_PRIVACY_CONFIGURATION = {

    "configuration_version":
        "1.0",

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "privacy_definition": {

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator",

        "gradient_representation":
            "per-example discriminator gradients",

        "clipping":
            "flat L2",

        "noise":
            "Gaussian",

        "sampling":
            "Poisson",

        "accountant":
            "RDP",

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,
    },

    "parameters": {

        "target_epsilon":
            TARGET_EPSILON,

        "delta_rule":
            "min(1e-5, 1/N_train)",

        "max_grad_norm":
            MAX_GRAD_NORM,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,

        "accountant":
            ACCOUNTANT,

        "sampling":
            SAMPLING_MECHANISM,

        "clipping":
            CLIPPING_MECHANISM,

        "loss_reduction":
            LOSS_REDUCTION,

        "grad_sample_mode":
            GRAD_SAMPLE_MODE,
    },

    "dataset_parameters":
        PRIVACY_PARAMETER_DF.to_dict(
            orient="records"
        ),

    "source_dependencies": {

        "notebook_08_architecture":
            str(
                NB08_ROOT
            ),

        "notebook_09_statistical_guidance":
            str(
                NB09_ROOT
            ),
    },

    "downstream": {

        "notebook_11":
            "Formal RDP privacy accounting",

        "notebook_12":
            "SPP-GAN DP training",

        "notebook_13":
            "Synthetic data generation",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

PRIVACY_CONFIGURATION_PATH = (
    DIRS["configuration"] /
    "sppgan_privacy_configuration.json"
)

with open(
    PRIVACY_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PERSISTED_PRIVACY_CONFIGURATION,
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# Reload
# --------------------------------------------------------------------------------------------------

with open(
    PRIVACY_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as f:

    RELOADED_PRIVACY_CONFIGURATION = json.load(
        f
    )

REQUIRED_PRIVACY_CONFIGURATION_KEYS = {
    "configuration_version",
    "notebook",
    "name",
    "framework",
    "privacy_definition",
    "parameters",
    "dataset_parameters",
    "source_dependencies",
    "downstream",
    "created_utc",
}

missing_keys = (
    REQUIRED_PRIVACY_CONFIGURATION_KEYS
    -
    set(
        RELOADED_PRIVACY_CONFIGURATION.keys()
    )
)

if missing_keys:
    raise RuntimeError(
        "Persisted privacy configuration is missing:\n"
        f"{sorted(missing_keys)}"
    )

print(
    f"✓ Privacy configuration saved:\n"
    f"  {PRIVACY_CONFIGURATION_PATH}"
)

print(
    "✓ Configuration reload validation: PASS"
)

print("SECTION 16 STATUS: PASS")


16. SAVE PRIVACY CONFIGURATION
✓ Privacy configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/configuration/sppgan_privacy_configuration.json
✓ Configuration reload validation: PASS
SECTION 16 STATUS: PASS


In [75]:
# ==================================================================================================
# 17. SAVE PRIVACY MANIFEST
# ==================================================================================================

print("\n" + "=" * 100)
print("17. SAVE PRIVACY MANIFEST")
print("=" * 100)


def sha256_file(
    path,
):
    """
    RAM-safe SHA-256 calculation.
    """

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


PRIVACY_ARTIFACTS = [
    DP_MODULE_PATH,
    PRIVACY_CONFIGURATION_PATH,
    PRIVACY_METADATA_PATH,
    GRADIENT_TEST_PATH,
    NOISE_TEST_PATH,
    PRIVACY_MECHANISM_PATH,
]

PRIVACY_ARTIFACT_REGISTRY = []

for artifact_path in PRIVACY_ARTIFACTS:

    artifact_path = Path(
        artifact_path
    )

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required artifact missing:\n"
            f"{artifact_path}"
        )

    PRIVACY_ARTIFACT_REGISTRY.append({

        "artifact":
            artifact_path.name,

        "absolute_path":
            str(
                artifact_path
            ),

        "exists":
            True,

        "size_bytes":
            artifact_path.stat().st_size,

        "sha256":
            sha256_file(
                artifact_path
            ),

    })

PRIVACY_ARTIFACT_REGISTRY_DF = pd.DataFrame(
    PRIVACY_ARTIFACT_REGISTRY
)

PRIVACY_ARTIFACT_REGISTRY_PATH = (
    DIRS["metadata"] /
    "sppgan_privacy_artifact_registry.csv"
)

PRIVACY_ARTIFACT_REGISTRY_DF.to_csv(
    PRIVACY_ARTIFACT_REGISTRY_PATH,
    index=False,
)

display(
    PRIVACY_ARTIFACT_REGISTRY_DF
)

print(
    f"✓ Privacy artifact registry saved:\n"
    f"  {PRIVACY_ARTIFACT_REGISTRY_PATH}"
)

print(
    f"✓ Registered artifacts : "
    f"{len(PRIVACY_ARTIFACT_REGISTRY_DF)}"
)

print("SECTION 17 STATUS: PASS")


17. SAVE PRIVACY MANIFEST


,artifact,absolute_path,exists,size_bytes,sha256
0,sppgan_differential_privacy.py,/content/drive/MyDrive/SPP_GAN_Research/result...,True,5441,a1f070885ae77a30422af0181f25af9508662386a59bf7...
1,sppgan_privacy_configuration.json,/content/drive/MyDrive/SPP_GAN_Research/result...,True,2878,9b4246a7fe8b2579e7f72eb872ca35996732c9b9a291c9...
2,sppgan_privacy_metadata.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,1297,754b069e97b538ad90495e5880ba799f4aeb9165e6df33...
3,dp_gradient_privatization_tests.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,729,efeca22ed32db56c0a7dc80c4004f52cd5d3397cd40bfa...
4,dp_gaussian_noise_tests.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,594,21b942a5e16a9f656912fd0f2268435d7c5bf3e913241f...
5,dp_privacy_mechanism_validation.csv,/content/drive/MyDrive/SPP_GAN_Research/result...,True,585,8fcce5b4bf4a3388c45f7d5c0bf805c7bf82204d620bc1...


✓ Privacy artifact registry saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/metadata/sppgan_privacy_artifact_registry.csv
✓ Registered artifacts : 6
SECTION 17 STATUS: PASS


In [76]:
# ==================================================================================================
# 18. FINAL VERIFICATION
# ==================================================================================================

print("\n" + "=" * 100)
print("18. FINAL VERIFICATION")
print("=" * 100)

FINAL_CHECKS = []

def add_final_check(
    name,
    condition,
):
    FINAL_CHECKS.append({
        "check": name,
        "status": (
            "PASS"
            if bool(condition)
            else "FAIL"
        ),
    })


# --------------------------------------------------------------------------------------------------
# Root / dependencies
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Project root exists",
    PROJECT_ROOT.exists(),
)

add_final_check(
    "Notebook 08 architecture loaded",
    len(
        ARCHITECTURE_SUMMARY_DF
    ) == len(DATASET_IDS),
)

add_final_check(
    "Notebook 09 configuration loaded",
    NB09_CONFIGURATION_PATH.exists(),
)

# --------------------------------------------------------------------------------------------------
# Privacy parameters
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Target epsilon positive",
    TARGET_EPSILON > 0,
)

add_final_check(
    "Maximum gradient norm positive",
    MAX_GRAD_NORM > 0,
)

add_final_check(
    "Poisson sampling selected",
    SAMPLING_MECHANISM == "poisson",
)

add_final_check(
    "Flat clipping selected",
    CLIPPING_MECHANISM == "flat",
)

add_final_check(
    "RDP accountant selected",
    ACCOUNTANT == "rdp",
)

# --------------------------------------------------------------------------------------------------
# Validation results
# --------------------------------------------------------------------------------------------------

add_final_check(
    "Gradient privatization tests pass",
    GRADIENT_TEST_DF[
        "status"
    ].eq("PASS").all(),
)

add_final_check(
    "Gaussian noise tests pass",
    NOISE_TEST_DF[
        "status"
    ].eq("PASS").all(),
)

add_final_check(
    "Privacy mechanism tests pass",
    PRIVACY_MECHANISM_DF[
        "status"
    ].eq("PASS").all(),
)

# --------------------------------------------------------------------------------------------------
# Artifacts
# --------------------------------------------------------------------------------------------------

add_final_check(
    "DP module exists",
    DP_MODULE_PATH.exists(),
)

add_final_check(
    "Privacy configuration exists",
    PRIVACY_CONFIGURATION_PATH.exists(),
)

add_final_check(
    "Privacy metadata exists",
    PRIVACY_METADATA_PATH.exists(),
)

add_final_check(
    "Artifact registry exists",
    PRIVACY_ARTIFACT_REGISTRY_PATH.exists(),
)

FINAL_CHECKS_DF = pd.DataFrame(
    FINAL_CHECKS
)

if not (
    FINAL_CHECKS_DF[
        "status"
    ]
    .eq("PASS")
    .all()
):

    display(
        FINAL_CHECKS_DF
    )

    raise RuntimeError(
        "Notebook 10 final verification failed."
    )

FINAL_VERIFICATION_PATH = (
    DIRS["validation"] /
    "sppgan_privacy_final_verification.csv"
)

FINAL_CHECKS_DF.to_csv(
    FINAL_VERIFICATION_PATH,
    index=False,
)

display(
    FINAL_CHECKS_DF
)

print(
    f"✓ Final verification saved:\n"
    f"  {FINAL_VERIFICATION_PATH}"
)

print(
    "✓ All final checks: PASS"
)

print("SECTION 18 STATUS: PASS")


18. FINAL VERIFICATION


,check,status
0,Project root exists,PASS
1,Notebook 08 architecture loaded,PASS
2,Notebook 09 configuration loaded,PASS
3,Target epsilon positive,PASS
4,Maximum gradient norm positive,PASS
5,Poisson sampling selected,PASS
6,Flat clipping selected,PASS
7,RDP accountant selected,PASS
8,Gradient privatization tests pass,PASS
9,Gaussian noise tests pass,PASS


✓ Final verification saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/validation/sppgan_privacy_final_verification.csv
✓ All final checks: PASS
SECTION 18 STATUS: PASS


In [77]:
# ==================================================================================================
# 19. COMPLETION SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("19. COMPLETION SUMMARY")
print("=" * 100)

COMPLETION_MANIFEST = {

    "notebook":
        NOTEBOOK_ID,

    "name":
        NOTEBOOK_NAME,

    "framework":
        FRAMEWORK_NAME,

    "status":
        "PASS",

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "datasets_registered":
        len(
            DATASET_IDS
        ),

    "datasets":
        DATASET_IDS,

    "privacy_mechanism": {

        "mechanism":
            "DP-SGD",

        "protected_component":
            "SPP-GAN discriminator",

        "per_example_gradients":
            "PASS",

        "flat_gradient_clipping":
            "PASS",

        "gaussian_noise":
            "PASS",

        "poisson_sampling":
            "PASS",

        "accountant":
            "RDP",
    },

    "parameters": {

        "target_epsilon":
            TARGET_EPSILON,

        "delta_rule":
            "min(1e-5, 1/N_train)",

        "max_grad_norm":
            MAX_GRAD_NORM,

        "batch_size":
            DP_BATCH_SIZE,

        "epochs":
            DP_EPOCHS,
    },

    "validation": {

        "architecture":
            "PASS",

        "privacy_parameters":
            "PASS",

        "gradient_privatization":
            "PASS",

        "noise_injection":
            "PASS",

        "privacy_mechanism":
            "PASS",

        "final_verification":
            "PASS",
    },

    "training_performed":
        False,

    "privacy_accounting_performed":
        False,

    "achieved_epsilon_reported":
        False,

    "synthetic_generation_performed":
        False,

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,

    "artifacts": {

        "dp_module":
            str(
                DP_MODULE_PATH
            ),

        "privacy_configuration":
            str(
                PRIVACY_CONFIGURATION_PATH
            ),

        "privacy_metadata":
            str(
                PRIVACY_METADATA_PATH
            ),

        "gradient_tests":
            str(
                GRADIENT_TEST_PATH
            ),

        "noise_tests":
            str(
                NOISE_TEST_PATH
            ),

        "mechanism_validation":
            str(
                PRIVACY_MECHANISM_PATH
            ),

        "artifact_registry":
            str(
                PRIVACY_ARTIFACT_REGISTRY_PATH
            ),

        "final_verification":
            str(
                FINAL_VERIFICATION_PATH
            ),
    },

    "source_dependencies": {

        "notebook_08":
            str(
                NB08_ROOT
            ),

        "notebook_09":
            str(
                NB09_ROOT
            ),
    },

    "downstream": {

        "notebook_11":
            "SPP-GAN Privacy Accounting",

        "notebook_12":
            "SPP-GAN DP Training",

        "notebook_13":
            "SPP-GAN Synthetic Generation",
    },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

COMPLETION_PATH = (
    DIRS["validation"] /
    "sppgan_notebook_10_completion.json"
)

with open(
    COMPLETION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        COMPLETION_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
    )

# --------------------------------------------------------------------------------------------------
# Final display
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 10 — FINAL STATUS")
print("=" * 100)

print(
    f"Framework                       : "
    f"{FRAMEWORK_NAME}"
)

print(
    f"Datasets                        : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Privacy mechanism               : DP-SGD"
)

print(
    f"Protected component             : "
    f"SPP-GAN discriminator"
)

print(
    f"Per-example gradients           : PASS"
)

print(
    f"Flat gradient clipping          : PASS"
)

print(
    f"Gaussian noise                  : PASS"
)

print(
    f"Poisson sampling                : PASS"
)

print(
    f"RDP accountant                  : CONFIGURED"
)

print(
    f"Target epsilon                  : "
    f"{TARGET_EPSILON}"
)

print(
    f"Maximum gradient norm           : "
    f"{MAX_GRAD_NORM}"
)

print(
    f"DP batch size                   : "
    f"{DP_BATCH_SIZE}"
)

print(
    f"DP epochs                       : "
    f"{DP_EPOCHS}"
)

print(
    f"Training                        : "
    f"NOT PERFORMED"
)

print(
    f"Privacy accounting              : "
    f"DEFERRED TO NOTEBOOK 11"
)

print(
    f"Achieved epsilon                : "
    f"NOT CLAIMED"
)

print(
    f"Synthetic generation            : "
    f"NOT PERFORMED"
)

print(
    f"End-to-end privacy claim        : "
    f"NOT ESTABLISHED"
)

print(
    f"Overall status                  : "
    f"PASS"
)

print()
print("Artifacts:")

print(
    f"  DP module       : "
    f"{DP_MODULE_PATH}"
)

print(
    f"  Configuration   : "
    f"{PRIVACY_CONFIGURATION_PATH}"
)

print(
    f"  Metadata        : "
    f"{PRIVACY_METADATA_PATH}"
)

print(
    f"  Manifest        : "
    f"{PRIVACY_ARTIFACT_REGISTRY_PATH}"
)

print(
    f"  Verification    : "
    f"{FINAL_VERIFICATION_PATH}"
)

print(
    f"  Completion      : "
    f"{COMPLETION_PATH}"
)

print()
print("Next:")

print(
    "  Notebook 11 — SPP-GAN Privacy Accounting"
)

print("=" * 100)

print(
    "\n✓ NOTEBOOK 10 COMPLETED SUCCESSFULLY."
)


19. COMPLETION SUMMARY

NOTEBOOK 10 — FINAL STATUS
Framework                       : SPP-GAN
Datasets                        : 3
Privacy mechanism               : DP-SGD
Protected component             : SPP-GAN discriminator
Per-example gradients           : PASS
Flat gradient clipping          : PASS
Gaussian noise                  : PASS
Poisson sampling                : PASS
RDP accountant                  : CONFIGURED
Target epsilon                  : 5.0
Maximum gradient norm           : 1.0
DP batch size                   : 128
DP epochs                       : 300
Training                        : NOT PERFORMED
Privacy accounting              : DEFERRED TO NOTEBOOK 11
Achieved epsilon                : NOT CLAIMED
Synthetic generation            : NOT PERFORMED
End-to-end privacy claim        : NOT ESTABLISHED
Overall status                  : PASS

Artifacts:
  DP module       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_pr